# ApexID CVPR Stage 02
## Final Three-Model BHCA Benchmark with Frozen Logistic Rejection

This is the evaluation-only final stage. It performs no model training, inference, embedding extraction, split generation, logistic fitting, feature-scaler fitting, or official threshold selection.

For `margin_gallery_logistic`, Stage 02 reconstructs top-1 cosine, top-2 cosine, their margin, and gallery size from frozen test embeddings; validates the Stage 01 calibrator JSON and SHA-256; manually computes frozen `P(known)`; and applies the frozen validation probability threshold. Nearest-centroid cosine still supplies the identity assignment. Rejected queries enter the existing frozen clustering method.

The `raw_max_cosine` Stage 01 ablation remains dispatchable. Cache provenance includes rejection method, calibration protocol, feature order, gallery identity count, embedding provenance, and calibrator hash, so raw or stale open-set, discovery, bootstrap, paired, and oracle caches cannot be silently mixed with logistic results.

Final-test labels are used only after predictions are frozen for metrics and clearly separated oracle threshold diagnostics. The oracle rejection sweep operates on the already-frozen knownness score and never refits a scaler or logistic model.


## Frozen inputs

- Stage 00: Seif prepared bundle (`apexid_cvpr_prepared`)
- Stage 01: `apexid_cvpr_stage01_bhca_logistic` from notebook 01
- Stage 02 output: `apexid_cvpr_final_benchmark_bhca_logistic`

This run evaluates the three BHCA RGB ConvNeXt-Small backbones only. Legacy HA / ApexID_v2 / MiewIDv3 / MegaDescriptor artifacts are ignored.


In [1]:
# ============================================================
# 1. Minimal local CPU environment and explicit paths
# ============================================================

import ast
import json
import math
import os
import platform
import shutil
import sys
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()


def _is_usable_path(path: Path | None) -> bool:
    if path is None:
        return False
    text = str(path).strip()
    if not text:
        return False
    # Never treat a Windows drive path as valid on Linux/Kaggle.
    if ON_KAGGLE and (":\\" in text or text.startswith(("D:\\", "C:\\", "d:\\", "c:\\"))):
        return False
    return True


def _first_existing_file(candidates: list[Path]) -> Path | None:
    for path in candidates:
        if _is_usable_path(path) and path.is_file():
            return path.resolve()
    return None


def _first_existing_root(candidates: list[Path], marker: str) -> Path | None:
    for path in candidates:
        if not _is_usable_path(path):
            continue
        if (path / marker).is_file():
            return path.resolve()
    return None


def _search_kaggle_input(marker: str, preferred_dirname: str | None = None) -> Path | None:
    """Find a Stage root under /kaggle/input regardless of owner/slug nesting."""
    root = Path("/kaggle/input")
    if not root.is_dir():
        return None
    preferred_hits: list[Path] = []
    other_hits: list[Path] = []
    for candidate in root.rglob(marker):
        if not candidate.is_file():
            continue
        parent = candidate.parent.resolve()
        if preferred_dirname and parent.name == preferred_dirname:
            preferred_hits.append(parent)
        else:
            other_hits.append(parent)
    hits = preferred_hits or other_hits
    return hits[0] if hits else None


STAGE02_NOTEBOOK_PATH = _first_existing_file([
    Path(os.environ.get("APEXID_STAGE02_NOTEBOOK", "")),
    Path("/kaggle/working/02_run_cvpr_benchmark.ipynb"),
    Path.cwd() / "02_run_cvpr_benchmark.ipynb",
    *([] if ON_KAGGLE else [
        Path(r"D:\CVPR_BM\new_bm\02_run_cvpr_benchmark.ipynb"),
        Path(r"c:\Users\Legion 5\Desktop\ApexID\notebooks\02_run_cvpr_benchmark.ipynb"),
    ]),
])
if STAGE02_NOTEBOOK_PATH is None:
    STAGE02_NOTEBOOK_PATH = Path("02_run_cvpr_benchmark.ipynb")

STAGE00_CANDIDATES = [
    Path(os.environ.get("APEXID_PREPARED_INPUT") or os.environ.get("APEXID_PREPARED_ROOT") or ""),
    Path("/kaggle/input/datasets/seifelkerdany/apexid-cvpr-prepared-bundle/apexid_cvpr_prepared"),
    Path("/kaggle/input/apexid-cvpr-prepared-bundle/apexid_cvpr_prepared"),
    Path("/kaggle/input/datasets/seifelkerdany/apexid-cvpr-prepared-bundle"),
    Path("/kaggle/working/apexid_cvpr_prepared"),
]
if not ON_KAGGLE:
    STAGE00_CANDIDATES.append(Path(r"D:\CVPR_BM\apexid_cvpr_prepared"))
STAGE00_ROOT = (
    _first_existing_root(STAGE00_CANDIDATES, "preparation_manifest.json")
    or _search_kaggle_input("preparation_manifest.json", preferred_dirname="apexid_cvpr_prepared")
)
if STAGE00_ROOT is None:
    raise FileNotFoundError(
        "Missing frozen Stage 00 root (preparation_manifest.json). "
        "On Kaggle: Add Input → attach the Stage 00 prepared bundle "
        "(apexid-cvpr-prepared-bundle / apexid_cvpr_prepared). "
        "Or set APEXID_PREPARED_INPUT to that folder."
    )

STAGE01_CANDIDATES = [
    Path(os.environ.get("APEXID_STAGE01_INPUT", "")),
    Path("/kaggle/working/apexid_cvpr_stage01_bhca_logistic"),
    Path("/kaggle/input/apexid-cvpr-stage01-bhca-logistic"),
    Path("/kaggle/input/apexid-cvpr-stage01-bhca-logistic/apexid_cvpr_stage01_bhca_logistic"),
    Path("/kaggle/input/datasets/mohamedbassat/apexid-cvpr-stage01-bhca-logistic"),
    Path("/kaggle/input/datasets/mohamedbassat/apexid-cvpr-stage01-bhca-logistic/apexid_cvpr_stage01_bhca_logistic"),
]
if not ON_KAGGLE:
    STAGE01_CANDIDATES.append(Path(r"D:\CVPR_BM\apexid_cvpr_stage01_bhca_logistic"))
STAGE01_ROOT = (
    _first_existing_root(STAGE01_CANDIDATES, "stage01_manifest.json")
    or _search_kaggle_input("stage01_manifest.json", preferred_dirname="apexid_cvpr_stage01_bhca_logistic")
)
if STAGE01_ROOT is None:
    raise FileNotFoundError(
        "Missing frozen Stage 01 root (stage01_manifest.json). "
        "On Kaggle: Add Input → apexid-cvpr-stage01-bhca-logistic. "
        "Or set APEXID_STAGE01_INPUT to the folder that contains stage01_manifest.json."
    )

OUTPUT_ROOT = Path(os.environ.get(
    "APEXID_STAGE02_OUTPUT",
    "/kaggle/working/apexid_cvpr_final_benchmark_bhca_logistic" if ON_KAGGLE else r"D:\CVPR_BM\apexid_cvpr_final_benchmark_bhca_logistic",
)).resolve()
WORKSPACE_ROOT = STAGE02_NOTEBOOK_PATH.parent.resolve()
CLOSED_SET_PRE_LOGISTIC_FUNCTION_SHA256 = "59d5877444690a393eb70f385e9981520c8e4cc95d0678e4ca2196880813eea1"

# No outer process pool is used. BLAS owns this single configured level of
# parallelism, avoiding nested joblib/OpenMP oversubscription.
N_JOBS = int(os.environ.get("APEXID_N_JOBS", "-1"))
CPU_COUNT = os.cpu_count() or 1
NUMERIC_THREADS = CPU_COUNT if N_JOBS == -1 else max(1, min(N_JOBS, CPU_COUNT))
for _variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_variable, str(NUMERIC_THREADS))

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

BENCHMARK_WALL_START = time.perf_counter()

print("Stage 02 notebook:", STAGE02_NOTEBOOK_PATH)
print("Stage 00:", STAGE00_ROOT)
print("Stage 01:", STAGE01_ROOT)
print("Final output:", OUTPUT_ROOT)
print("CPU count:", CPU_COUNT, "configured N_JOBS:", N_JOBS)
print("Stage 02 requires no model inference and no GPU.")


Stage 02 notebook: 02_run_cvpr_benchmark.ipynb
Stage 00: /kaggle/input/datasets/seifelkerdany/apexid-cvpr-prepared-bundle/apexid_cvpr_prepared
Stage 01: /kaggle/input/datasets/mohamedbassat/apexid-cvpr-stage01-bhca-logistic/apexid_cvpr_stage01_bhca_logistic
Final output: /kaggle/working/apexid_cvpr_final_benchmark_bhca_logistic
CPU count: 4 configured N_JOBS: -1
Stage 02 requires no model inference and no GPU.


## 2. Embedded benchmark metric/statistics runtime

This is the same reviewed benchmark runtime used by the consolidated CVPR notebook. Stage 02 uses only its metric, clustering, bootstrap, plotting, reporting, and cache utilities; model-loading and dataset-preparation functions remain unused.

In [2]:
"""Evaluation-only utilities for the frozen ApexID CVPR Stage 02 benchmark.

This dependency-closed runtime intentionally excludes data preparation, model loading,
image transforms, checkpoint access, embedding extraction, training, and CUDA helpers.
"""

from __future__ import annotations

import contextlib

import dataclasses

import hashlib

import json

import math

import os

import platform

import random

import sys

import tempfile

import time

from dataclasses import dataclass, field

from pathlib import Path

from typing import Any, Callable, Mapping, Sequence

import numpy as np

import pandas as pd

from scipy.cluster.hierarchy import fcluster, linkage

from scipy.spatial.distance import pdist

from sklearn.metrics import adjusted_rand_score, roc_auc_score, v_measure_score

BENCHMARK_VERSION = "apexid-cvpr-benchmark-margin-gallery-logistic-1.0.0"
LOGISTIC_FEATURE_NAMES = ("s1", "margin", "log_gallery_size")
SUPPORTED_REJECTION_METHODS = {"raw_max_cosine", "margin_gallery_logistic"}

MODEL_NAMES = (
    "ApexID_BHCA_A_v2",
    "ApexID_BHCA_SCA_v2",
    "ApexID_BHCA_SCSF2_v2",
)
EXPECTED_MODELS = set(MODEL_NAMES)
if len(MODEL_NAMES) != 3:
    raise RuntimeError("This notebook is sealed to the three BHCA RGB backbones")

REQUIRED_RESULT_DIRECTORIES = (
    "manifests",
    "splits",
    "embeddings",
    "scores",
    "calibration",
    "closed_set",
    "open_set",
    "oracle_novel_clustering",
    "end_to_end_discovery",
    "statistics",
    "plots",
    "tables",
    "reports",
    "oracle_diagnostics",
)

def _json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if dataclasses.is_dataclass(value):
        return dataclasses.asdict(value)
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"Cannot serialize {type(value).__name__}")

def stable_json(value: Any) -> str:
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        default=_json_default,
    )

def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def atomic_write_text(path: Path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", dir=path.parent)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        temporary.write_text(text, encoding="utf-8")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

def write_json(path: Path, value: Any) -> None:
    atomic_write_text(Path(path), json.dumps(value, indent=2, sort_keys=True, default=_json_default) + "\n")

def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", dir=path.parent)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        frame.to_csv(temporary, index=False, lineterminator="\n")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

def atomic_save_npz(path: Path, **arrays: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.stem}.", suffix=".npz", dir=path.parent)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        np.savez_compressed(temporary, **arrays)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

def ensure_result_tree(root: Path) -> dict[str, Path]:
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    paths = {name: root / name for name in REQUIRED_RESULT_DIRECTORIES}
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths

@dataclass
class RuntimeRecorder:
    rows: list[dict[str, Any]] = field(default_factory=list)

    @contextlib.contextmanager
    def measure(self, stage: str, **metadata: Any):
        start = time.perf_counter()
        try:
            yield
        finally:
            row = {"stage": stage, "seconds": time.perf_counter() - start, **metadata}
            self.rows.append(row)

    def record(self, stage: str, seconds: float, **metadata: Any) -> None:
        self.rows.append({"stage": stage, "seconds": float(seconds), **metadata})

    def frame(self) -> pd.DataFrame:
        base = [
            "stage",
            "model",
            "dataset",
            "split_seed",
            "seconds",
            "cache_hit",
            "number_images",
            "number_queries",
            "number_gallery_identities",
        ]
        frame = pd.DataFrame(self.rows)
        for column in base:
            if column not in frame:
                frame[column] = np.nan
        return frame[base + [c for c in frame.columns if c not in base]]

    def save(self, path: Path) -> None:
        atomic_write_csv(path, self.frame())

def stable_seed(*parts: Any) -> int:
    digest = hashlib.sha256("::".join(map(str, parts)).encode("utf-8")).digest()
    return int.from_bytes(digest[:8], byteorder="big", signed=False) % (2**32 - 1)

def seed_everything(seed: int) -> None:
    """Seed only the CPU libraries used by this evaluation-only notebook."""
    random.seed(int(seed))
    np.random.seed(int(seed))

def _package_version(name: str) -> str | None:
    try:
        from importlib.metadata import version

        return version(name)
    except Exception:
        return None

def environment_provenance() -> dict[str, Any]:
    """Record CPU evaluation dependencies without importing accelerator/model libraries."""
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "os": os.name,
        "execution_device": "CPU",
        "accelerator_runtime_queried": False,
        "packages": {
            name: _package_version(name)
            for name in ("numpy", "pandas", "scipy", "scikit-learn")
        },
    }

def l2_normalize(embeddings: np.ndarray) -> np.ndarray:
    values = np.asarray(embeddings, dtype=np.float32)
    if values.ndim != 2 or not np.isfinite(values).all():
        raise ValueError("Embeddings must be a finite two-dimensional array")
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    if np.any(norms <= 1e-12):
        raise ValueError("Embedding cache contains zero-norm rows")
    return values / norms

def load_embedding_cache(
    path: Path,
    *,
    expected_metadata: Mapping[str, Any] | None = None,
    expected_image_ids: Sequence[str] | None = None,
) -> tuple[np.ndarray, dict[str, Any], np.ndarray, np.ndarray, np.ndarray]:
    with np.load(Path(path), allow_pickle=False) as cached:
        metadata = json.loads(str(cached["metadata_json"].item()))
        embeddings = l2_normalize(cached["embeddings"])
        image_ids = cached["image_ids"].astype(str)
        paths = cached["paths"].astype(str)
        identities = cached["identities"].astype(str)
    if expected_metadata is not None and stable_json(metadata) != stable_json(dict(expected_metadata)):
        raise ValueError("Embedding cache provenance mismatch")
    if expected_image_ids is not None and not np.array_equal(image_ids, np.asarray(expected_image_ids).astype(str)):
        raise ValueError("Embedding cache image order mismatch")
    if not (len(embeddings) == len(image_ids) == len(paths) == len(identities)):
        raise ValueError("Embedding cache array lengths differ")
    expected_dimension = int(metadata["embedding_dimension"])
    if embeddings.shape[1] != expected_dimension:
        raise ValueError(
            f"Embedding cache dimension {embeddings.shape[1]} != provenance {expected_dimension}"
        )
    return embeddings, metadata, image_ids, paths, identities

@dataclass
class IdentityGallery:
    identities: np.ndarray
    centroids: np.ndarray
    image_count_by_identity: np.ndarray

def build_identity_gallery(embeddings: np.ndarray, identities: Sequence[str]) -> IdentityGallery:
    embeddings = l2_normalize(embeddings)
    labels = np.asarray(identities).astype(str)
    if len(labels) != len(embeddings):
        raise ValueError("Gallery labels and embeddings have different lengths")
    unique = np.asarray(sorted(set(labels)), dtype=str)
    if not len(unique):
        raise ValueError("Cannot build an empty identity gallery")
    centroids = []
    counts = []
    for identity in unique:
        values = embeddings[labels == identity]
        centroids.append(values.mean(axis=0))
        counts.append(len(values))
    return IdentityGallery(unique, l2_normalize(np.vstack(centroids)), np.asarray(counts, dtype=int))

def score_identity_gallery(
    query_embeddings: np.ndarray,
    gallery: IdentityGallery,
    *,
    chunk_size: int = 2048,
    top_k: int = 50,
) -> dict[str, np.ndarray]:
    query = l2_normalize(query_embeddings)
    top_count = min(max(int(top_k), 1), len(gallery.identities))
    top_indices = np.empty((len(query), top_count), dtype=np.int64)
    top_scores = np.empty((len(query), top_count), dtype=np.float32)
    for start in range(0, len(query), max(int(chunk_size), 1)):
        stop = min(start + max(int(chunk_size), 1), len(query))
        scores = query[start:stop] @ gallery.centroids.T
        if top_count == scores.shape[1]:
            indices = np.argsort(-scores, axis=1, kind="stable")
        else:
            indices = np.argpartition(-scores, kth=top_count - 1, axis=1)[:, :top_count]
            selected = np.take_along_axis(scores, indices, axis=1)
            order = np.argsort(-selected, axis=1, kind="stable")
            indices = np.take_along_axis(indices, order, axis=1)
        top_indices[start:stop] = indices
        top_scores[start:stop] = np.take_along_axis(scores, indices, axis=1)
    return {
        "top_identity_indices": top_indices,
        "top_identity_labels": gallery.identities[top_indices],
        "top_scores": top_scores,
        "knownness_scores": top_scores[:, 0],
        "predicted_identities": gallery.identities[top_indices[:, 0]],
    }


def stable_sigmoid(values: Sequence[float]) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    output = np.empty_like(values)
    positive = values >= 0
    output[positive] = 1.0 / (1.0 + np.exp(-values[positive]))
    exp_values = np.exp(values[~positive])
    output[~positive] = exp_values / (1.0 + exp_values)
    return output


def validate_rejection_calibrator(calibrator: Mapping[str, Any], *, expected_model: str) -> None:
    if calibrator.get("method") != "margin_gallery_logistic":
        raise ValueError(f"{expected_model}: calibrator method is not margin_gallery_logistic")
    if calibrator.get("version") != "margin_gallery_logistic_v1":
        raise ValueError(f"{expected_model}: unsupported calibrator version")
    if calibrator.get("model") != expected_model:
        raise ValueError(f"{expected_model}: calibrator model provenance mismatch")
    if tuple(calibrator.get("feature_names", ())) != LOGISTIC_FEATURE_NAMES:
        raise ValueError(f"{expected_model}: calibrator feature order mismatch")
    if calibrator.get("contains_final_test_labels") is not False:
        raise ValueError(f"{expected_model}: calibrator does not deny final-test-label use")
    mean = np.asarray(calibrator["standardizer"]["mean"], dtype=np.float64)
    scale = np.asarray(calibrator["standardizer"]["scale"], dtype=np.float64)
    coef = np.asarray(calibrator["logistic_regression"]["coef"], dtype=np.float64).reshape(-1)
    intercept = float(calibrator["logistic_regression"]["intercept"])
    threshold = float(calibrator["rejection_probability_threshold"])
    if not (len(mean) == len(scale) == len(coef) == len(LOGISTIC_FEATURE_NAMES)):
        raise ValueError(f"{expected_model}: calibrator parameter dimensions are invalid")
    if not np.isfinite(mean).all() or not np.isfinite(scale).all() or np.any(scale <= 0):
        raise ValueError(f"{expected_model}: calibrator standardizer is invalid")
    if not np.isfinite(coef).all() or not np.isfinite(intercept):
        raise ValueError(f"{expected_model}: calibrator coefficients are invalid")
    if not 0.0 <= threshold <= 1.0:
        raise ValueError(f"{expected_model}: probability threshold is outside [0, 1]")


def apply_margin_gallery_logistic(s1, s2, gallery_size: int, calibrator: Mapping[str, Any]) -> np.ndarray:
    if int(gallery_size) < 2:
        raise ValueError("Logistic rejection requires at least two gallery identities")
    s1 = np.asarray(s1, dtype=np.float64)
    s2 = np.asarray(s2, dtype=np.float64)
    if s1.shape != s2.shape or s1.ndim != 1:
        raise ValueError("s1 and s2 must be aligned one-dimensional arrays")
    if np.any(s1 + 1e-6 < s2):
        raise ValueError("Top-2 score ordering is invalid")
    margin = s1 - s2
    log_gallery_size = math.log(float(gallery_size))
    features = np.column_stack([s1, margin, np.full(len(s1), log_gallery_size)])
    if tuple(calibrator["feature_names"]) != LOGISTIC_FEATURE_NAMES:
        raise ValueError("Frozen calibrator feature order mismatch")
    mean = np.asarray(calibrator["standardizer"]["mean"], dtype=np.float64)
    scale = np.asarray(calibrator["standardizer"]["scale"], dtype=np.float64)
    coef = np.asarray(calibrator["logistic_regression"]["coef"], dtype=np.float64).reshape(-1)
    intercept = float(calibrator["logistic_regression"]["intercept"])
    probabilities = stable_sigmoid(((features - mean) / scale) @ coef + intercept)
    if not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise RuntimeError("Frozen logistic calibrator produced invalid P(known)")
    return probabilities

def score_cache_metadata(
    *,
    model: str,
    dataset: str,
    split_seed: int,
    embedding_metadata: Mapping[str, Any],
    split_manifest_sha256: str,
    rejection_method: str,
    calibration_protocol_version: str,
    calibrator_sha256: str | None,
    feature_names: Sequence[str],
    gallery_identity_count: int,
    gallery_scoring: str = "centroid",
) -> dict[str, Any]:
    return {
        "schema_version": 3,
        "benchmark_version": BENCHMARK_VERSION,
        "model": model,
        "dataset": dataset,
        "split_seed": int(split_seed),
        "embedding_provenance_sha256": sha256_bytes(stable_json(dict(embedding_metadata)).encode("utf-8")),
        "split_manifest_sha256": split_manifest_sha256,
        "gallery_scoring": gallery_scoring,
        "gallery_identity_count": int(gallery_identity_count),
        "rejection_method": rejection_method,
        "calibration_protocol_version": calibration_protocol_version,
        "calibrator_sha256": calibrator_sha256,
        "feature_names": list(feature_names),
    }
def save_open_score_cache(
    path: Path,
    *,
    metadata: Mapping[str, Any],
    image_ids: Sequence[str],
    true_identities: Sequence[str],
    is_novel: Sequence[bool],
    predicted_identities: Sequence[str],
    s1: Sequence[float],
    s2: Sequence[float],
    margin: Sequence[float],
    gallery_size: Sequence[int],
    log_gallery_size: Sequence[float],
    p_known: Sequence[float],
    knownness_scores: Sequence[float],
) -> None:
    atomic_save_npz(
        path,
        metadata_json=np.asarray(stable_json(dict(metadata))),
        image_ids=np.asarray(image_ids, dtype="U"),
        true_identities=np.asarray(true_identities, dtype="U"),
        is_novel=np.asarray(is_novel, dtype=bool),
        predicted_identities=np.asarray(predicted_identities, dtype="U"),
        s1=np.asarray(s1, dtype=np.float32),
        s2=np.asarray(s2, dtype=np.float32),
        margin=np.asarray(margin, dtype=np.float32),
        gallery_size=np.asarray(gallery_size, dtype=np.int64),
        log_gallery_size=np.asarray(log_gallery_size, dtype=np.float64),
        p_known=np.asarray(p_known, dtype=np.float64),
        knownness_scores=np.asarray(knownness_scores, dtype=np.float64),
    )
def load_open_score_cache(
    path: Path,
    *,
    expected_metadata: Mapping[str, Any] | None = None,
) -> dict[str, Any]:
    with np.load(path, allow_pickle=False) as cached:
        required = {
            "metadata_json", "image_ids", "true_identities", "is_novel",
            "predicted_identities", "s1", "s2", "margin", "gallery_size",
            "log_gallery_size", "p_known", "knownness_scores",
        }
        missing = required - set(cached.files)
        if missing:
            raise ValueError(f"Open score cache uses an obsolete schema; missing {sorted(missing)}")
        metadata = json.loads(str(cached["metadata_json"].item()))
        result = {
            "metadata": metadata,
            "image_ids": cached["image_ids"].astype(str),
            "true_identities": cached["true_identities"].astype(str),
            "is_novel": cached["is_novel"].astype(bool),
            "predicted_identities": cached["predicted_identities"].astype(str),
            "s1": cached["s1"].astype(np.float32),
            "s2": cached["s2"].astype(np.float32),
            "margin": cached["margin"].astype(np.float32),
            "gallery_size": cached["gallery_size"].astype(np.int64),
            "log_gallery_size": cached["log_gallery_size"].astype(np.float64),
            "p_known": cached["p_known"].astype(np.float64),
            "knownness_scores": cached["knownness_scores"].astype(np.float64),
        }
    if expected_metadata is not None and stable_json(metadata) != stable_json(dict(expected_metadata)):
        raise ValueError("Open score cache provenance mismatch")
    lengths = {len(value) for key, value in result.items() if key != "metadata"}
    if len(lengths) != 1:
        raise ValueError("Open score cache arrays have different lengths")
    if not np.isfinite(result["knownness_scores"]).all():
        raise ValueError("Open score cache contains non-finite official knownness scores")
    if metadata.get("rejection_method") == "margin_gallery_logistic":
        if not np.isfinite(result["p_known"]).all() or np.any((result["p_known"] < 0) | (result["p_known"] > 1)):
            raise ValueError("Open score cache contains invalid P(known)")
        if not np.array_equal(result["knownness_scores"], result["p_known"]):
            raise ValueError("Logistic cache knownness_scores is not exactly p_known")
    return result
def save_closed_result_cache(
    path: Path,
    *,
    metadata: Mapping[str, Any],
    metrics: Mapping[str, Any],
    query_details: pd.DataFrame,
) -> None:
    required = {"image_id", "identity", "valid", "rank1", "average_precision"}
    if not required <= set(query_details):
        raise ValueError(f"Closed result details missing {sorted(required - set(query_details))}")
    atomic_save_npz(
        path,
        metadata_json=np.asarray(stable_json(dict(metadata))),
        metrics_json=np.asarray(stable_json(dict(metrics))),
        image_ids=query_details["image_id"].astype(str).to_numpy(dtype="U"),
        identities=query_details["identity"].astype(str).to_numpy(dtype="U"),
        valid=query_details["valid"].astype(bool).to_numpy(),
        rank1=query_details["rank1"].astype(float).to_numpy(),
        average_precision=query_details["average_precision"].astype(float).to_numpy(),
    )

def load_closed_result_cache(
    path: Path,
    *,
    expected_metadata: Mapping[str, Any] | None = None,
) -> tuple[dict[str, Any], pd.DataFrame]:
    with np.load(path, allow_pickle=False) as cached:
        metadata = json.loads(str(cached["metadata_json"].item()))
        metrics = json.loads(str(cached["metrics_json"].item()))
        details = pd.DataFrame(
            {
                "image_id": cached["image_ids"].astype(str),
                "identity": cached["identities"].astype(str),
                "valid": cached["valid"].astype(bool),
                "rank1": cached["rank1"].astype(float),
                "average_precision": cached["average_precision"].astype(float),
            }
        )
    if expected_metadata is not None and stable_json(metadata) != stable_json(dict(expected_metadata)):
        raise ValueError("Closed result cache provenance mismatch")
    return metrics, details

def _balanced_mean(labels: Sequence[str], values: Sequence[float]) -> float:
    labels_array = np.asarray(labels).astype(str)
    values_array = np.asarray(values, dtype=float)
    if not len(values_array):
        return float("nan")
    return float(np.mean([values_array[labels_array == identity].mean() for identity in np.unique(labels_array)]))

def open_set_metrics_from_scores(
    *,
    knownness_scores: Sequence[float],
    predicted_identities: Sequence[str],
    true_identities: Sequence[str],
    is_novel: Sequence[bool],
    rejection_threshold: float,
) -> dict[str, float]:
    """Apply a frozen rejection threshold and compute the repository's exact metrics.

    The formulas are embedded here so evaluating frozen NumPy embeddings does not
    import ``apexid.__init__`` (and therefore does not require PyTorch).
    """
    from sklearn.metrics import auc

    def threshold_at_target_tpr(known_scores: np.ndarray, target_tpr: float = 0.95):
        finite = np.asarray(known_scores, dtype=np.float64)
        finite = finite[np.isfinite(finite)]
        if finite.size == 0:
            return float("nan"), float("nan")
        if not 0.0 < float(target_tpr) <= 1.0:
            raise ValueError("target_tpr must be in the interval (0, 1].")
        ordered = np.sort(finite)[::-1]
        index = min(max(int(math.ceil(float(target_tpr) * ordered.size)) - 1, 0), ordered.size - 1)
        threshold = float(ordered[index])
        return threshold, float(np.mean(finite >= threshold))

    def acceptance_curve(known_scores: np.ndarray, unknown_scores: np.ndarray, known_correct: np.ndarray):
        known_scores = np.asarray(known_scores, dtype=np.float64)
        unknown_scores = np.asarray(unknown_scores, dtype=np.float64)
        known_correct = np.asarray(known_correct, dtype=bool)
        if known_scores.size == 0 or unknown_scores.size == 0:
            return None, None
        if known_correct.shape[0] != known_scores.shape[0]:
            raise ValueError("known_correct length must match known_scores.")
        combined = np.concatenate([known_scores, unknown_scores])
        is_known = np.concatenate([
            np.ones(known_scores.shape[0], dtype=bool),
            np.zeros(unknown_scores.shape[0], dtype=bool),
        ])
        correct = np.concatenate([known_correct, np.zeros(unknown_scores.shape[0], dtype=bool)])
        finite_indices = np.flatnonzero(np.isfinite(combined))
        if finite_indices.size == 0:
            return None, None
        fpr_values = [0.0]
        ccr_values = [0.0]
        unknown_accepted = 0
        known_correct_accepted = 0
        order = finite_indices[np.argsort(-combined[finite_indices])]
        cursor = 0
        while cursor < order.size:
            threshold = combined[order[cursor]]
            next_cursor = cursor + 1
            while next_cursor < order.size and combined[order[next_cursor]] == threshold:
                next_cursor += 1
            indices = order[cursor:next_cursor]
            unknown_accepted += int(np.sum(~is_known[indices]))
            known_correct_accepted += int(np.sum(is_known[indices] & correct[indices]))
            fpr_values.append(float(unknown_accepted / unknown_scores.shape[0]))
            ccr_values.append(float(known_correct_accepted / known_scores.shape[0]))
            cursor = next_cursor
        return np.asarray(fpr_values, dtype=np.float64), np.asarray(ccr_values, dtype=np.float64)

    def oscr_auc(known_scores: np.ndarray, unknown_scores: np.ndarray, known_correct: np.ndarray) -> float:
        fpr_values, ccr_values = acceptance_curve(known_scores, unknown_scores, known_correct)
        if fpr_values is None:
            return float("nan")
        order = np.argsort(fpr_values)
        fpr_values = fpr_values[order]
        ccr_values = ccr_values[order]
        unique_fpr = np.unique(fpr_values)
        max_ccr = np.asarray([np.max(ccr_values[fpr_values == fpr]) for fpr in unique_fpr])
        return float(auc(unique_fpr, max_ccr))

    def dir_at_far(
        known_scores: np.ndarray,
        unknown_scores: np.ndarray,
        known_correct: np.ndarray,
        far_level: float,
    ) -> float:
        fpr_values, ccr_values = acceptance_curve(known_scores, unknown_scores, known_correct)
        if fpr_values is None:
            return float("nan")
        valid = fpr_values <= float(far_level)
        return float(np.max(ccr_values[valid])) if np.any(valid) else float("nan")

    scores = np.asarray(knownness_scores, dtype=float)
    predicted = np.asarray(predicted_identities).astype(str)
    truth = np.asarray(true_identities).astype(str)
    novel = np.asarray(is_novel, dtype=bool)
    if not (len(scores) == len(predicted) == len(truth) == len(novel)):
        raise ValueError("Open-set score arrays have different lengths")
    known = ~novel
    if not known.any() or not novel.any():
        raise ValueError("Open-set metrics require both known and novel queries")
    accepted = scores >= float(rejection_threshold)
    known_correct = predicted[known] == truth[known]
    baks = _balanced_mean(truth[known], (accepted[known] & known_correct).astype(float))
    baus = _balanced_mean(truth[novel], (~accepted[novel]).astype(float))
    normacc = float(math.sqrt(max(baks, 0.0) * max(baus, 0.0)))
    auroc = float(roc_auc_score(known.astype(int), scores))
    threshold_95, achieved_tpr = threshold_at_target_tpr(scores[known], target_tpr=0.95)
    fpr95 = float(np.mean(scores[novel] >= threshold_95))
    accepted_known = accepted[known]
    return {
        "BAKS": baks,
        "BAUS": baus,
        "NormAcc": normacc,
        "Known-vs-Novel AUROC": auroc,
        "TNR@95TPR": 1.0 - fpr95,
        "FPR@95TPR": fpr95,
        "threshold_at_95_tpr": float(threshold_95),
        "achieved_known_tpr_at_95_threshold": float(achieved_tpr),
        "FAR_at_frozen_threshold": float(np.mean(accepted[novel])),
        "known_TPR_at_frozen_threshold": float(np.mean(accepted_known)),
        "novel_rejection_rate": float(np.mean(~accepted[novel])),
        "OSCR-AUC": oscr_auc(scores[known], scores[novel], known_correct),
        "DIR@1%FAR": dir_at_far(scores[known], scores[novel], known_correct, far_level=0.01),
        "DIR@5%FAR": dir_at_far(scores[known], scores[novel], known_correct, far_level=0.05),
        "known_identification_Rank-1_among_accepted": float(
            known_correct[accepted_known].mean() if accepted_known.any() else float("nan")
        ),
        "n_known_query_samples": int(known.sum()),
        "n_novel_query_samples": int(novel.sum()),
        "n_known_identities": int(len(np.unique(truth[known]))),
        "n_novel_identities": int(len(np.unique(truth[novel]))),
        "empirical_FAR_resolution": float(1.0 / novel.sum()),
        "rejection_threshold": float(rejection_threshold),
    }


def oscr_curve(
    *,
    known_scores: Sequence[float],
    novel_scores: Sequence[float],
    known_correct: Sequence[bool],
) -> pd.DataFrame:
    """Return the empirical OSCR false-positive/correct-classification curve."""
    known_scores = np.asarray(known_scores, dtype=float)
    novel_scores = np.asarray(novel_scores, dtype=float)
    known_correct = np.asarray(known_correct, dtype=bool)
    if len(known_scores) != len(known_correct) or not len(known_scores) or not len(novel_scores):
        raise ValueError("OSCR curve needs known scores/correctness and novel scores")
    scores = np.concatenate([known_scores, novel_scores])
    is_known = np.concatenate(
        [np.ones(len(known_scores), dtype=bool), np.zeros(len(novel_scores), dtype=bool)]
    )
    correct = np.concatenate([known_correct, np.zeros(len(novel_scores), dtype=bool)])
    order = np.argsort(-scores, kind="stable")
    false_accepts = 0
    correct_accepts = 0
    rows = [{"threshold": float("inf"), "FPR": 0.0, "CCR": 0.0}]
    cursor = 0
    while cursor < len(order):
        threshold = scores[order[cursor]]
        stop = cursor + 1
        while stop < len(order) and scores[order[stop]] == threshold:
            stop += 1
        selected = order[cursor:stop]
        false_accepts += int((~is_known[selected]).sum())
        correct_accepts += int((is_known[selected] & correct[selected]).sum())
        rows.append(
            {
                "threshold": float(threshold),
                "FPR": false_accepts / len(novel_scores),
                "CCR": correct_accepts / len(known_scores),
            }
        )
        cursor = stop
    return pd.DataFrame(rows)

def select_rejection_threshold(
    *,
    knownness_scores: Sequence[float],
    predicted_identities: Sequence[str],
    true_identities: Sequence[str],
    is_novel: Sequence[bool],
) -> tuple[float, pd.DataFrame]:
    """Test-oracle diagnostic only; never used by official predictions."""
    scores = np.asarray(knownness_scores, dtype=np.float64)
    predicted = np.asarray(predicted_identities).astype(str)
    truth = np.asarray(true_identities).astype(str)
    novel = np.asarray(is_novel, dtype=bool)
    if not (scores.shape[0] == predicted.shape[0] == truth.shape[0] == novel.shape[0]):
        raise ValueError("scores, masks, true labels, and predicted labels must have the same length.")
    finite_indices = np.flatnonzero(np.isfinite(scores))
    if finite_indices.size != scores.size:
        raise ValueError("Oracle rejection scores must all be finite")

    # Exact O(N log N) repository sweep. It selects the highest threshold among
    # all operating points that maximize identity-balanced NormAcc.
    known = ~novel
    known_positions = np.flatnonzero(known)
    novel_positions = np.flatnonzero(novel)
    if known_positions.size == 0 or novel_positions.size == 0:
        raise ValueError("Oracle rejection diagnostic requires known and novel queries")
    _, known_inverse, known_counts = np.unique(
        truth[known_positions], return_inverse=True, return_counts=True
    )
    _, novel_inverse, novel_counts = np.unique(
        truth[novel_positions], return_inverse=True, return_counts=True
    )
    known_counts = known_counts.astype(np.float64)
    novel_counts = novel_counts.astype(np.float64)
    known_inverse_by_position = np.full(scores.shape[0], -1, dtype=np.int64)
    novel_inverse_by_position = np.full(scores.shape[0], -1, dtype=np.int64)
    known_inverse_by_position[known_positions] = known_inverse
    novel_inverse_by_position[novel_positions] = novel_inverse
    known_correct_accepted = np.zeros(known_counts.shape[0], dtype=np.float64)
    novel_accepted = np.zeros(novel_counts.shape[0], dtype=np.float64)

    def score_from_counts() -> float:
        baks = float(np.mean(known_correct_accepted / known_counts))
        baus = float(np.mean((novel_counts - novel_accepted) / novel_counts))
        return float(math.sqrt(max(baks, 0.0) * max(baus, 0.0)))

    tau = float(np.nextafter(np.max(scores), np.inf))
    best_score = score_from_counts()
    order = finite_indices[np.argsort(-scores[finite_indices])]
    cursor = 0
    while cursor < order.size:
        threshold = scores[order[cursor]]
        next_cursor = cursor + 1
        while next_cursor < order.size and scores[order[next_cursor]] == threshold:
            next_cursor += 1
        indices = order[cursor:next_cursor]
        known_group = indices[known[indices]]
        if known_group.size:
            correct_group = known_group[predicted[known_group] == truth[known_group]]
            if correct_group.size:
                np.add.at(known_correct_accepted, known_inverse_by_position[correct_group], 1.0)
        novel_group = indices[novel[indices]]
        if novel_group.size:
            np.add.at(novel_accepted, novel_inverse_by_position[novel_group], 1.0)
        score = score_from_counts()
        if score > best_score or (score == best_score and float(threshold) > tau):
            best_score = score
            tau = float(threshold)
        cursor = next_cursor

    all_candidates = np.unique(np.concatenate((
        [np.nextafter(scores.min(), -np.inf)],
        scores,
        [np.nextafter(scores.max(), np.inf)],
    )))
    if len(all_candidates) > 501:
        positions = np.unique(np.rint(np.linspace(0, len(all_candidates) - 1, 501)).astype(int))
        candidates = all_candidates[positions]
    else:
        candidates = all_candidates
    candidates = np.unique(np.concatenate([candidates, [tau]]))
    rows = []
    for value in candidates:
        accepted = scores >= float(value)
        baks = _balanced_mean(truth[known], (accepted[known] & (predicted[known] == truth[known])).astype(float))
        baus = _balanced_mean(truth[novel], (~accepted[novel]).astype(float))
        normacc = float(math.sqrt(max(baks, 0.0) * max(baus, 0.0)))
        rows.append({
            "rejection_threshold": float(value),
            "BAKS": baks,
            "BAUS": baus,
            "NormAcc": normacc,
            "selected": bool(value == tau),
        })
    return tau, pd.DataFrame(rows)



def closed_set_retrieval_metrics(
    *,
    query_embeddings: np.ndarray,
    query_identities: Sequence[str],
    gallery_embeddings: np.ndarray,
    gallery_identities: Sequence[str],
    chunk_size: int = 1024,
) -> tuple[dict[str, float], pd.DataFrame]:
    """Repository-equivalent retrieval metrics without importing PyTorch."""
    query = np.asarray(query_embeddings).astype(np.float64, copy=False)
    gallery = np.asarray(gallery_embeddings).astype(np.float64, copy=False)
    if query.ndim != 2 or gallery.ndim != 2:
        raise ValueError("embeddings must have shape (num_samples, embedding_dim).")
    if query.shape[1:] != gallery.shape[1:]:
        raise ValueError("query and gallery embeddings must have the same embedding dimension.")
    query = (query / np.maximum(np.linalg.norm(query, axis=1, keepdims=True), 1e-12)).astype(np.float32, copy=False)
    gallery = (gallery / np.maximum(np.linalg.norm(gallery, axis=1, keepdims=True), 1e-12)).astype(np.float32, copy=False)
    query_labels = np.asarray(query_identities).astype(str)
    gallery_labels = np.asarray(gallery_identities).astype(str)
    if query_labels.ndim != 1 or len(query_labels) != len(query):
        raise ValueError("query identity labels do not align with query embeddings")
    if gallery_labels.ndim != 1 or len(gallery_labels) != len(gallery):
        raise ValueError("gallery identity labels do not align with gallery embeddings")
    if int(chunk_size) <= 0:
        raise ValueError("chunk_size must be > 0.")

    top_k = (1, 5, 10)
    cmc_ranks = (20, 50)
    ranks = tuple(sorted(set(top_k) | set(cmc_ranks)))
    result = {"known/mAP": float("nan"), "known/BAKS": float("nan")}
    result.update({f"known/top{k}": float("nan") for k in top_k})
    result.update({f"known/cmc_rank{k}": float("nan") for k in cmc_ranks})
    num_queries = int(len(query_labels))
    num_gallery = int(len(gallery_labels))
    query_values = {
        "valid": np.zeros(num_queries, dtype=bool),
        "rank1": np.full(num_queries, np.nan, dtype=np.float64),
        "average_precision": np.full(num_queries, np.nan, dtype=np.float64),
    }
    if num_queries and num_gallery:
        max_rank = max(ranks)
        top_hits = {k: 0 for k in ranks}
        valid_queries = 0
        ap_values = []
        top1_correct = np.zeros(num_queries, dtype=np.float64)
        positive_indices = {
            label: np.flatnonzero(gallery_labels == label) for label in np.unique(query_labels)
        }
        positive_counts = {label: int(indices.size) for label, indices in positive_indices.items()}
        for start in range(0, num_queries, int(chunk_size)):
            stop = min(start + int(chunk_size), num_queries)
            labels_chunk = query_labels[start:stop]
            similarity = query[start:stop] @ gallery.T
            has_positive = np.asarray([positive_counts.get(label, 0) > 0 for label in labels_chunk], dtype=bool)
            if not np.any(has_positive):
                continue
            valid_queries += int(np.sum(has_positive))
            query_values["valid"][start:stop] = has_positive
            rank_limit = min(max_rank, num_gallery)
            if rank_limit == num_gallery:
                top_indices = np.argsort(-similarity, axis=1)[:, :rank_limit]
            else:
                top_indices = np.argpartition(-similarity, kth=rank_limit - 1, axis=1)[:, :rank_limit]
                top_scores = np.take_along_axis(similarity, top_indices, axis=1)
                top_order = np.argsort(-top_scores, axis=1)
                top_indices = np.take_along_axis(top_indices, top_order, axis=1)
            top_relevance = gallery_labels[top_indices] == labels_chunk[:, None]
            top1_correct[start:stop] = top_relevance[:, :1].any(axis=1).astype(np.float64)
            query_values["rank1"][start:stop][has_positive] = top1_correct[start:stop][has_positive]
            for k in ranks:
                top_hits[k] += int(top_relevance[has_positive, :min(k, rank_limit)].any(axis=1).sum())

            valid_local = np.flatnonzero(has_positive)
            sorted_scores = -np.sort(-similarity[valid_local], axis=1)
            for sorted_row, row_offset in enumerate(valid_local):
                matches = positive_indices[labels_chunk[row_offset]]
                scores = similarity[row_offset, matches]
                score_order = np.argsort(-scores)
                scores = scores[score_order]
                positive_ranks = np.searchsorted(-sorted_scores[sorted_row], -scores, side="left")
                if positive_ranks.size > 1:
                    tie_offsets = np.zeros(positive_ranks.shape[0], dtype=np.int64)
                    tie_start = 0
                    while tie_start < scores.size:
                        tie_stop = tie_start + 1
                        while tie_stop < scores.size and scores[tie_stop] == scores[tie_start]:
                            tie_stop += 1
                        tie_offsets[tie_start:tie_stop] = np.arange(tie_stop - tie_start, dtype=np.int64)
                        tie_start = tie_stop
                    positive_ranks = positive_ranks + tie_offsets
                positive_ranks = positive_ranks + 1
                positive_ranks.sort()
                ap = float(np.mean(np.arange(1, positive_ranks.size + 1, dtype=np.float64) / positive_ranks))
                ap_values.append(ap)
                query_values["average_precision"][start + row_offset] = ap

        if valid_queries:
            for k in top_k:
                result[f"known/top{k}"] = float(top_hits[k] / valid_queries)
            for k in cmc_ranks:
                result[f"known/cmc_rank{k}"] = float(top_hits[k] / valid_queries)
            result["known/mAP"] = float(np.mean(ap_values)) if ap_values else float("nan")
            per_identity = [float(np.mean(top1_correct[query_labels == label])) for label in np.unique(query_labels)]
            result["known/BAKS"] = float(np.mean(per_identity)) if per_identity else float("nan")

    point = {
        "Rank-1": float(result["known/top1"]),
        "Rank-5": float(result["known/top5"]),
        "Rank-10": float(result["known/top10"]),
        "CMC@20": float(result["known/cmc_rank20"]),
        "CMC@50": float(result["known/cmc_rank50"]),
        "mAP": float(result["known/mAP"]),
        "identity-balanced Rank-1": float(result["known/BAKS"]),
        "effective_rank_5": int(min(5, num_gallery)),
        "effective_rank_10": int(min(10, num_gallery)),
        "effective_rank_20": int(min(20, num_gallery)),
        "effective_rank_50": int(min(50, num_gallery)),
    }
    details = pd.DataFrame({
        "identity": query_labels,
        "valid": query_values["valid"],
        "rank1": query_values["rank1"],
        "average_precision": query_values["average_precision"],
    })
    return point, details


@dataclass
class ClusterHierarchy:
    number_samples: int
    linkage_matrix: np.ndarray

def build_cluster_hierarchy(embeddings: np.ndarray) -> ClusterHierarchy:
    values = l2_normalize(embeddings)
    if len(values) <= 1:
        return ClusterHierarchy(len(values), np.empty((0, 4), dtype=float))
    distances = pdist(values, metric="cosine")
    return ClusterHierarchy(len(values), linkage(distances, method="average", optimal_ordering=False))

def cut_cluster_hierarchy(hierarchy: ClusterHierarchy, clustering_threshold: float) -> np.ndarray:
    tau = float(clustering_threshold)
    if not -1.0 <= tau <= 1.0:
        raise ValueError("clustering_threshold must lie in [-1, 1]")
    if hierarchy.number_samples == 0:
        return np.empty(0, dtype=np.int64)
    if hierarchy.number_samples == 1:
        return np.zeros(1, dtype=np.int64)
    labels = fcluster(hierarchy.linkage_matrix, t=1.0 - tau, criterion="distance")
    return labels.astype(np.int64) - 1

def discovery_metric_bundle(true_labels: Sequence[str], predicted_labels: Sequence[int]) -> dict[str, float]:
    """Exact repository discovery metrics without importing the model package."""
    from collections import Counter, defaultdict
    from sklearn.metrics import adjusted_mutual_info_score, normalized_mutual_info_score

    true = np.asarray(true_labels).astype(str)
    predicted = np.asarray(predicted_labels)
    if true.ndim != 1 or predicted.ndim != 1 or true.shape != predicted.shape:
        raise ValueError("true_labels and predicted_labels must be same-length 1D sequences")
    if true.size == 0:
        names = (
            "ari", "ami", "nmi", "b3_precision", "b3_recall", "b3_f1",
            "pairwise_precision", "pairwise_recall", "pairwise_f1",
            "mean_fragments_per_identity", "median_fragments_per_identity",
            "max_fragments_per_identity", "fraction_true_identities_fragmented",
            "mean_true_ids_per_predicted_cluster", "max_true_ids_per_predicted_cluster",
            "fraction_predicted_clusters_merged", "relative_cluster_count_error", "v_measure",
        )
        result = {name: float("nan") for name in names}
        result.update({"true_identity_count": 0.0, "predicted_cluster_count": 0.0, "absolute_cluster_count_error": 0.0})
        return result

    contingency = Counter(zip(true.tolist(), predicted.tolist()))
    true_sizes = Counter(true.tolist())
    predicted_sizes = Counter(predicted.tolist())
    b3_precision = sum(count * count / predicted_sizes[pred] for (_target, pred), count in contingency.items()) / true.size
    b3_recall = sum(count * count / true_sizes[target] for (target, _pred), count in contingency.items()) / true.size
    b3_f1 = 2 * b3_precision * b3_recall / (b3_precision + b3_recall) if b3_precision + b3_recall else 0.0
    choose2 = lambda count: count * (count - 1) // 2
    tp = sum(choose2(count) for count in contingency.values())
    predicted_pairs = sum(choose2(count) for count in predicted_sizes.values())
    true_pairs = sum(choose2(count) for count in true_sizes.values())
    pairwise_precision = tp / predicted_pairs if predicted_pairs else 0.0
    pairwise_recall = tp / true_pairs if true_pairs else 0.0
    pairwise_f1 = 2 * pairwise_precision * pairwise_recall / (pairwise_precision + pairwise_recall) if pairwise_precision + pairwise_recall else 0.0

    fragments_by_identity = defaultdict(set)
    identities_by_cluster = defaultdict(set)
    for target, prediction in zip(true.tolist(), predicted.tolist()):
        fragments_by_identity[target].add(prediction)
        identities_by_cluster[prediction].add(target)
    fragments = np.asarray([len(values) for values in fragments_by_identity.values()], dtype=float)
    merged = np.asarray([len(values) for values in identities_by_cluster.values()], dtype=float)
    true_count = len(true_sizes)
    predicted_count = len(predicted_sizes)
    count_error = abs(predicted_count - true_count)
    return {
        "ari": float(adjusted_rand_score(true, predicted)),
        "ami": float(adjusted_mutual_info_score(true, predicted)),
        "nmi": float(normalized_mutual_info_score(true, predicted)),
        "b3_precision": float(b3_precision),
        "b3_recall": float(b3_recall),
        "b3_f1": float(b3_f1),
        "pairwise_precision": float(pairwise_precision),
        "pairwise_recall": float(pairwise_recall),
        "pairwise_f1": float(pairwise_f1),
        "mean_fragments_per_identity": float(fragments.mean()),
        "median_fragments_per_identity": float(np.median(fragments)),
        "max_fragments_per_identity": float(fragments.max()),
        "fraction_true_identities_fragmented": float(np.mean(fragments > 1)),
        "mean_true_ids_per_predicted_cluster": float(merged.mean()),
        "max_true_ids_per_predicted_cluster": float(merged.max()),
        "fraction_predicted_clusters_merged": float(np.mean(merged > 1)),
        "true_identity_count": float(true_count),
        "predicted_cluster_count": float(predicted_count),
        "absolute_cluster_count_error": float(count_error),
        "relative_cluster_count_error": float(count_error / true_count) if true_count else float("nan"),
        "v_measure": float(v_measure_score(true, predicted)),
    }


def calibrate_clustering_threshold(
    embeddings: np.ndarray,
    true_labels: Sequence[str],
    *,
    candidates: Sequence[float],
    hierarchy: ClusterHierarchy | None = None,
) -> tuple[float, pd.DataFrame, ClusterHierarchy]:
    if hierarchy is None:
        hierarchy = build_cluster_hierarchy(embeddings)
    elif hierarchy.number_samples != len(embeddings):
        raise ValueError("Provided clustering hierarchy does not match embedding row count")
    rows = []
    for tau in sorted(set(map(float, candidates))):
        predicted = cut_cluster_hierarchy(hierarchy, tau)
        metrics = discovery_metric_bundle(true_labels, predicted)
        rows.append({"clustering_threshold": tau, **metrics})
    curve = pd.DataFrame(rows)
    if curve.empty or curve["b3_f1"].isna().all():
        raise ValueError("Cannot calibrate clustering threshold without valid B3 F1 values")
    best_value = float(curve["b3_f1"].max())
    # Deterministic conservative tie-break: highest similarity threshold, hence
    # fewer false merges when B3 F1 is exactly tied.
    tau = float(curve.loc[curve["b3_f1"] == best_value, "clustering_threshold"].max())
    curve["selected"] = curve["clustering_threshold"].eq(tau)
    return tau, curve, hierarchy

def end_to_end_discovery(
    *,
    query_embeddings: np.ndarray,
    true_labels: Sequence[str],
    is_novel: Sequence[bool],
    knownness_scores: Sequence[float],
    rejection_threshold: float,
    clustering_threshold: float,
) -> tuple[dict[str, float], np.ndarray, np.ndarray]:
    labels = np.asarray(true_labels).astype(str)
    novel = np.asarray(is_novel, dtype=bool)
    scores = np.asarray(knownness_scores, dtype=float)
    rejected = scores < float(rejection_threshold)
    rejected_embeddings = l2_normalize(query_embeddings)[rejected]
    hierarchy = build_cluster_hierarchy(rejected_embeddings)
    predicted = cut_cluster_hierarchy(hierarchy, clustering_threshold)
    result = end_to_end_metrics_from_assignments(
        true_labels=labels,
        is_novel=novel,
        rejected_mask=rejected,
        rejected_cluster_labels=predicted,
    )
    result["rejection_threshold"] = float(rejection_threshold)
    result["clustering_threshold"] = float(clustering_threshold)
    return result, rejected, predicted

def end_to_end_metrics_from_assignments(
    *,
    true_labels: Sequence[str],
    is_novel: Sequence[bool],
    rejected_mask: Sequence[bool],
    rejected_cluster_labels: Sequence[int],
) -> dict[str, float]:
    """Score discovery decisions without using truth to select the clustered pool.

    The predeclared coverage-adjusted score is
    ``conditional novel-cluster metric × identity-balanced novel discovery recall
    × (1 - known contamination rate)``.  It supplements, rather than replaces,
    the standard conditional clustering, coverage, and contamination fields.
    """
    labels = np.asarray(true_labels).astype(str)
    novel = np.asarray(is_novel, dtype=bool)
    rejected = np.asarray(rejected_mask, dtype=bool)
    predicted = np.asarray(rejected_cluster_labels)
    if not (len(labels) == len(novel) == len(rejected)):
        raise ValueError("End-to-end assignment arrays have different lengths")
    if len(predicted) != int(rejected.sum()):
        raise ValueError("Rejected cluster labels do not match rejected sample count")
    number_rejected = int(rejected.sum())
    if number_rejected:
        all_metrics = discovery_metric_bundle(labels[rejected], predicted)
        novel_within_rejected = novel[rejected]
        novel_metrics = discovery_metric_bundle(
            labels[rejected][novel_within_rejected], predicted[novel_within_rejected]
        )
    else:
        all_metrics = discovery_metric_bundle([], [])
        novel_metrics = discovery_metric_bundle([], [])
    sample_recall = float(np.mean(rejected[novel])) if novel.any() else float("nan")
    balanced_recall = (
        _balanced_mean(labels[novel], rejected[novel].astype(float)) if novel.any() else float("nan")
    )
    contamination = float((rejected & ~novel).sum() / number_rejected) if number_rejected else 0.0
    conditional_b3 = float(novel_metrics["b3_f1"])
    conditional_ari = float(novel_metrics["ari"])
    if not math.isfinite(conditional_b3):
        conditional_b3 = 0.0
    if not math.isfinite(conditional_ari):
        conditional_ari = 0.0
    result = {f"rejected_pool_{key}": value for key, value in all_metrics.items()}
    result.update({f"rejected_novel_{key}": value for key, value in novel_metrics.items()})
    # Empty discovery output is a valid poor system outcome, not a missing metric.
    if not number_rejected:
        result["rejected_pool_b3_f1"] = 0.0
        result["rejected_pool_ari"] = 0.0
    result.update(
        {
            "End-to-End Discovery B3_F1": conditional_b3 * balanced_recall * (1.0 - contamination),
            "End-to-End Discovery ARI": conditional_ari * balanced_recall * (1.0 - contamination),
            "novel_sample_discovery_recall": sample_recall,
            "novel_identity_balanced_discovery_recall": balanced_recall,
            "known_contamination_rate": contamination,
            "discovery_output_available": float(number_rejected > 0),
            "number_rejected_samples": number_rejected,
            "number_rejected_known_samples": int((rejected & ~novel).sum()),
            "number_rejected_novel_samples": int((rejected & novel).sum()),
            "number_missed_novel_samples": int((~rejected & novel).sum()),
        }
    )
    return result

def _resample_identity_indices(
    labels: Sequence[str],
    generator: np.random.Generator,
) -> np.ndarray:
    labels_array = np.asarray(labels).astype(str)
    identities = np.unique(labels_array)
    if not len(identities):
        return np.empty(0, dtype=np.int64)
    selected = generator.choice(identities, size=len(identities), replace=True)
    return np.concatenate([np.flatnonzero(labels_array == identity) for identity in selected])

def bootstrap_closed_details(
    details: pd.DataFrame,
    *,
    replicates: int,
    seed: int,
) -> dict[str, np.ndarray]:
    valid = details.loc[details["valid"].astype(bool)].reset_index(drop=True)
    generator = np.random.default_rng(seed)
    output = {"Rank-1": np.empty(replicates), "mAP": np.empty(replicates)}
    for replicate in range(replicates):
        indices = _resample_identity_indices(valid["identity"], generator)
        output["Rank-1"][replicate] = float(valid["rank1"].to_numpy()[indices].mean())
        output["mAP"][replicate] = float(valid["average_precision"].to_numpy()[indices].mean())
    return output

def _stratified_open_indices(
    labels: np.ndarray,
    novel: np.ndarray,
    generator: np.random.Generator,
) -> np.ndarray:
    groups = []
    for stratum in (False, True):
        positions = np.flatnonzero(novel == stratum)
        local = _resample_identity_indices(labels[positions], generator)
        groups.append(positions[local])
    return np.concatenate(groups)

OPEN_BOOTSTRAP_METRICS = (
    "BAKS",
    "BAUS",
    "NormAcc",
    "Known-vs-Novel AUROC",
    "OSCR-AUC",
    "DIR@1%FAR",
)

def bootstrap_open_scores(
    *,
    knownness_scores: Sequence[float],
    predicted_identities: Sequence[str],
    true_identities: Sequence[str],
    is_novel: Sequence[bool],
    rejection_threshold: float,
    replicates: int,
    seed: int,
) -> dict[str, np.ndarray]:
    scores = np.asarray(knownness_scores)
    predicted = np.asarray(predicted_identities).astype(str)
    labels = np.asarray(true_identities).astype(str)
    novel = np.asarray(is_novel, dtype=bool)
    generator = np.random.default_rng(seed)
    output = {metric: np.empty(replicates) for metric in OPEN_BOOTSTRAP_METRICS}
    for replicate in range(replicates):
        indices = _stratified_open_indices(labels, novel, generator)
        metrics = open_set_metrics_from_scores(
            knownness_scores=scores[indices],
            predicted_identities=predicted[indices],
            true_identities=labels[indices],
            is_novel=novel[indices],
            rejection_threshold=rejection_threshold,
        )
        for metric in output:
            output[metric][replicate] = metrics[metric]
    return output

def bootstrap_clustering(
    *,
    true_labels: Sequence[str],
    predicted_labels: Sequence[int],
    replicates: int,
    seed: int,
) -> dict[str, np.ndarray]:
    true = np.asarray(true_labels).astype(str)
    predicted = np.asarray(predicted_labels)
    generator = np.random.default_rng(seed)
    output = {"B3_F1": np.empty(replicates), "ARI": np.empty(replicates)}
    for replicate in range(replicates):
        indices = _resample_identity_indices(true, generator)
        metrics = discovery_metric_bundle(true[indices], predicted[indices])
        output["B3_F1"][replicate] = metrics["b3_f1"]
        output["ARI"][replicate] = metrics["ari"]
    return output

def bootstrap_end_to_end_discovery(
    *,
    true_labels: Sequence[str],
    is_novel: Sequence[bool],
    rejected_mask: Sequence[bool],
    rejected_cluster_labels: Sequence[int],
    replicates: int,
    seed: int,
) -> dict[str, np.ndarray]:
    labels = np.asarray(true_labels).astype(str)
    novel = np.asarray(is_novel, dtype=bool)
    rejected = np.asarray(rejected_mask, dtype=bool)
    clusters = np.asarray(rejected_cluster_labels)
    if len(clusters) != int(rejected.sum()):
        raise ValueError("Rejected cluster label count mismatch")
    full_clusters = np.full(len(labels), -1, dtype=np.int64)
    full_clusters[rejected] = clusters
    metric_names = (
        "End-to-End Discovery B3_F1",
        "End-to-End Discovery ARI",
        "rejected_pool_b3_f1",
        "rejected_pool_ari",
    )
    output = {metric: np.empty(replicates) for metric in metric_names}
    generator = np.random.default_rng(seed)
    for replicate in range(replicates):
        indices = _stratified_open_indices(labels, novel, generator)
        rejected_sub = rejected[indices]
        metrics = end_to_end_metrics_from_assignments(
            true_labels=labels[indices],
            is_novel=novel[indices],
            rejected_mask=rejected_sub,
            rejected_cluster_labels=full_clusters[indices][rejected_sub],
        )
        for metric in metric_names:
            output[metric][replicate] = metrics[metric]
    return output

def save_bootstrap(path: Path, values: Mapping[str, np.ndarray], metadata: Mapping[str, Any]) -> None:
    arrays = {f"metric__{key}": np.asarray(value, dtype=np.float64) for key, value in values.items()}
    atomic_save_npz(path, metadata_json=np.asarray(stable_json(dict(metadata))), **arrays)

def load_bootstrap(path: Path) -> tuple[dict[str, np.ndarray], dict[str, Any]]:
    with np.load(path, allow_pickle=False) as cached:
        metadata = json.loads(str(cached["metadata_json"].item()))
        values = {
            key.removeprefix("metric__"): np.asarray(cached[key], dtype=float)
            for key in cached.files
            if key.startswith("metric__")
        }
    return values, metadata

def verified_bootstrap(
    path: Path,
    *,
    expected_metadata: Mapping[str, Any],
    expected_metrics: Sequence[str],
    expected_replicates: int,
    compute: Callable[[], Mapping[str, np.ndarray]],
) -> tuple[dict[str, np.ndarray], bool]:
    path = Path(path)
    if path.is_file():
        try:
            values, metadata = load_bootstrap(path)
            if stable_json(metadata) != stable_json(dict(expected_metadata)):
                raise ValueError("bootstrap metadata changed")
            if set(values) != set(expected_metrics):
                raise ValueError("bootstrap metric set changed")
            if any(len(values[metric]) != int(expected_replicates) for metric in expected_metrics):
                raise ValueError("bootstrap replicate count changed")
            return values, True
        except Exception as exc:
            print(f"BOOTSTRAP CACHE MISS — {path}: {exc}")
    values = {key: np.asarray(value, dtype=float) for key, value in compute().items()}
    save_bootstrap(path, values, expected_metadata)
    return values, False

def hierarchical_bootstrap_summary(
    point_rows: pd.DataFrame,
    *,
    metrics: Sequence[str],
    bootstrap_path_column: str,
    group_columns: Sequence[str],
    replicates: int,
    seed: int,
) -> pd.DataFrame:
    rows = []
    grouper: Any = list(group_columns)
    for group_key, group in point_rows.groupby(grouper, sort=True):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        loaded = [load_bootstrap(Path(path))[0] for path in group[bootstrap_path_column]]
        generator = np.random.default_rng(stable_seed(seed, *group_key, "hierarchical"))
        for metric in metrics:
            distribution = np.empty(replicates)
            for replicate in range(replicates):
                selected_splits = generator.integers(0, len(loaded), size=len(loaded))
                samples = []
                for split_index in selected_splits:
                    values = loaded[int(split_index)][metric]
                    samples.append(values[generator.integers(0, len(values))])
                distribution[replicate] = float(np.nanmean(samples))
            point = float(group[metric].mean())
            rows.append(
                {
                    **dict(zip(group_columns, group_key)),
                    "metric": metric,
                    "mean": point,
                    "ci_low": float(np.nanpercentile(distribution, 2.5)),
                    "ci_high": float(np.nanpercentile(distribution, 97.5)),
                    "n_splits": int(len(group)),
                    "bootstrap_replicates": int(replicates),
                    "bootstrap_method": "split-seed bootstrap + within-split identity-block bootstrap",
                }
            )
    return pd.DataFrame(rows)

def paired_open_bootstrap_difference(
    reference: Mapping[str, Any],
    comparison: Mapping[str, Any],
    *,
    replicates: int,
    seed: int,
) -> dict[str, np.ndarray]:
    for key in ("image_ids", "true_identities", "is_novel"):
        if not np.array_equal(np.asarray(reference[key]), np.asarray(comparison[key])):
            raise ValueError(f"Paired open-set arrays do not align for {key}")
    labels = np.asarray(reference["true_identities"]).astype(str)
    novel = np.asarray(reference["is_novel"], dtype=bool)
    generator = np.random.default_rng(seed)
    output = {metric: np.empty(replicates) for metric in OPEN_BOOTSTRAP_METRICS}
    for replicate in range(replicates):
        indices = _stratified_open_indices(labels, novel, generator)
        ref_metrics = open_set_metrics_from_scores(
            knownness_scores=np.asarray(reference["knownness_scores"])[indices],
            predicted_identities=np.asarray(reference["predicted_identities"])[indices],
            true_identities=labels[indices],
            is_novel=novel[indices],
            rejection_threshold=float(reference["rejection_threshold"]),
        )
        cmp_metrics = open_set_metrics_from_scores(
            knownness_scores=np.asarray(comparison["knownness_scores"])[indices],
            predicted_identities=np.asarray(comparison["predicted_identities"])[indices],
            true_identities=labels[indices],
            is_novel=novel[indices],
            rejection_threshold=float(comparison["rejection_threshold"]),
        )
        for metric in output:
            output[metric][replicate] = ref_metrics[metric] - cmp_metrics[metric]
    return output

def paired_closed_bootstrap_difference(
    reference: pd.DataFrame,
    comparison: pd.DataFrame,
    *,
    replicates: int,
    seed: int,
) -> dict[str, np.ndarray]:
    required = {"image_id", "identity", "valid", "rank1", "average_precision"}
    if not required <= set(reference) or not required <= set(comparison):
        raise ValueError("Paired closed-set details are missing required columns")
    ref = reference.sort_values("image_id", kind="mergesort").reset_index(drop=True)
    cmp = comparison.sort_values("image_id", kind="mergesort").reset_index(drop=True)
    for column in ("image_id", "identity", "valid"):
        if not np.array_equal(ref[column].to_numpy(), cmp[column].to_numpy()):
            raise ValueError(f"Paired closed-set details do not align for {column}")
    valid = ref["valid"].astype(bool).to_numpy()
    ref = ref.loc[valid].reset_index(drop=True)
    cmp = cmp.loc[valid].reset_index(drop=True)
    generator = np.random.default_rng(seed)
    output = {"Rank-1": np.empty(replicates), "mAP": np.empty(replicates)}
    for replicate in range(replicates):
        indices = _resample_identity_indices(ref["identity"], generator)
        output["Rank-1"][replicate] = float(
            ref["rank1"].to_numpy()[indices].mean() - cmp["rank1"].to_numpy()[indices].mean()
        )
        output["mAP"][replicate] = float(
            ref["average_precision"].to_numpy()[indices].mean()
            - cmp["average_precision"].to_numpy()[indices].mean()
        )
    return output

def paired_clustering_bootstrap_difference(
    reference: Mapping[str, Any],
    comparison: Mapping[str, Any],
    *,
    replicates: int,
    seed: int,
) -> dict[str, np.ndarray]:
    for key in ("image_ids", "true_labels"):
        if not np.array_equal(np.asarray(reference[key]), np.asarray(comparison[key])):
            raise ValueError(f"Paired clustering arrays do not align for {key}")
    true = np.asarray(reference["true_labels"]).astype(str)
    ref_pred = np.asarray(reference["predicted_labels"])
    cmp_pred = np.asarray(comparison["predicted_labels"])
    generator = np.random.default_rng(seed)
    output = {"B3_F1": np.empty(replicates), "ARI": np.empty(replicates)}
    for replicate in range(replicates):
        indices = _resample_identity_indices(true, generator)
        ref_metrics = discovery_metric_bundle(true[indices], ref_pred[indices])
        cmp_metrics = discovery_metric_bundle(true[indices], cmp_pred[indices])
        output["B3_F1"][replicate] = ref_metrics["b3_f1"] - cmp_metrics["b3_f1"]
        output["ARI"][replicate] = ref_metrics["ari"] - cmp_metrics["ari"]
    return output

def paired_difference_summary(
    *,
    bootstrap_paths: Sequence[Path],
    point_differences: Mapping[str, float],
    metrics: Sequence[str],
    replicates: int,
    seed: int,
    metadata: Mapping[str, Any],
) -> pd.DataFrame:
    loaded = [load_bootstrap(path)[0] for path in bootstrap_paths]
    generator = np.random.default_rng(seed)
    rows = []
    for metric in metrics:
        distribution = np.empty(replicates)
        for replicate in range(replicates):
            selected_splits = generator.integers(0, len(loaded), size=len(loaded))
            distribution[replicate] = float(
                np.mean(
                    [
                        loaded[int(split_index)][metric][
                            generator.integers(0, len(loaded[int(split_index)][metric]))
                        ]
                        for split_index in selected_splits
                    ]
                )
            )
        low = float(np.percentile(distribution, 2.5))
        high = float(np.percentile(distribution, 97.5))
        rows.append(
            {
                **dict(metadata),
                "metric": metric,
                "mean_difference_reference_minus_comparison": float(point_differences[metric]),
                "ci_low": low,
                "ci_high": high,
                "paired_95ci_excludes_zero": bool(low > 0.0 or high < 0.0),
                "bootstrap_replicates": int(replicates),
                "bootstrap_method": "paired split-seed + identity-block bootstrap",
            }
        )
    return pd.DataFrame(rows)

def dataset_macro_hierarchical_bootstrap(
    point_rows: pd.DataFrame,
    *,
    task: str,
    metrics: Sequence[str],
    bootstrap_path_column: str,
    replicates: int,
    seed: int,
) -> pd.DataFrame:
    required = {"model", "dataset", bootstrap_path_column, *metrics}
    if not required <= set(point_rows):
        raise ValueError(f"Dataset-macro input missing {sorted(required - set(point_rows))}")
    rows = []
    for model, model_rows in point_rows.groupby("model", sort=True):
        datasets = sorted(model_rows["dataset"].astype(str).unique())
        loaded: dict[str, list[dict[str, np.ndarray]]] = {
            dataset: [
                load_bootstrap(Path(path))[0]
                for path in model_rows.loc[
                    model_rows["dataset"].astype(str) == dataset, bootstrap_path_column
                ]
            ]
            for dataset in datasets
        }
        generator = np.random.default_rng(stable_seed(seed, task, model, "dataset-macro"))
        for metric in metrics:
            distribution = np.empty(replicates)
            for replicate in range(replicates):
                chosen_datasets = generator.choice(datasets, size=len(datasets), replace=True)
                dataset_values = []
                for dataset in chosen_datasets:
                    split_values = loaded[str(dataset)]
                    split = split_values[generator.integers(0, len(split_values))]
                    values = split[metric]
                    dataset_values.append(values[generator.integers(0, len(values))])
                distribution[replicate] = float(np.nanmean(dataset_values))
            point = float(
                model_rows.groupby("dataset", sort=True)[metric].mean().mean()
            )
            rows.append(
                {
                    "task": task,
                    "model": model,
                    "metric": metric,
                    "aggregation": "Dataset-Macro Average",
                    "mean": point,
                    "ci_low": float(np.nanpercentile(distribution, 2.5)),
                    "ci_high": float(np.nanpercentile(distribution, 97.5)),
                    "number_datasets": len(datasets),
                    "bootstrap_replicates": int(replicates),
                    "bootstrap_method": "dataset + split-seed + identity-block hierarchical bootstrap",
                }
            )
    return pd.DataFrame(rows)

def save_required_plots(
    *,
    plot_dir: Path,
    closed_summary: pd.DataFrame,
    open_summary: pd.DataFrame,
    oracle_summary: pd.DataFrame,
    discovery_summary: pd.DataFrame,
    macro_summary: pd.DataFrame,
    rejection_curves: pd.DataFrame,
    clustering_curves: pd.DataFrame,
    oscr_curves: pd.DataFrame,
) -> list[str]:
    import matplotlib.pyplot as plt
    import seaborn as sns

    plot_dir = Path(plot_dir)
    plot_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid", context="paper")
    saved: list[str] = []

    def bar_plot(frame: pd.DataFrame, metric: str, filename: str, title: str) -> None:
        subset = frame.loc[frame["metric"] == metric].copy()
        if subset.empty:
            raise ValueError(f"Cannot create required plot {filename}: metric {metric} is absent")
        fig, axis = plt.subplots(figsize=(9, 4.5))
        sns.barplot(data=subset, x="dataset", y="mean", hue="model", ax=axis, errorbar=None)
        axis.set_title(title)
        axis.set_xlabel("Dataset")
        axis.set_ylabel(metric)
        axis.set_ylim(min(0.0, float(subset["mean"].min())), 1.0)
        axis.tick_params(axis="x", rotation=25)
        fig.tight_layout()
        for suffix in ("png", "pdf"):
            path = plot_dir / f"{filename}.{suffix}"
            fig.savefig(path, dpi=300, bbox_inches="tight")
            saved.append(str(path))
        plt.close(fig)

    bar_plot(closed_summary, "Rank-1", "closed_rank1", "Closed-set Rank-1")
    bar_plot(closed_summary, "mAP", "closed_map", "Closed-set mAP")
    for metric, filename in (
        ("BAKS", "open_baks"),
        ("BAUS", "open_baus"),
        ("NormAcc", "open_normacc"),
        ("Known-vs-Novel AUROC", "known_vs_novel_auroc"),
        ("OSCR-AUC", "oscr_auc"),
    ):
        bar_plot(open_summary, metric, filename, metric)
    bar_plot(oracle_summary, "B3_F1", "oracle_novel_b3", "Oracle-Novel Clustering B³ F1")
    bar_plot(oracle_summary, "ARI", "oracle_novel_ari", "Oracle-Novel Clustering ARI")
    bar_plot(
        discovery_summary,
        "End-to-End Discovery B3_F1",
        "end_to_end_discovery_b3",
        "End-to-End Discovery: coverage/contamination-adjusted B³ F1",
    )

    if rejection_curves.empty or clustering_curves.empty:
        raise ValueError("Validation threshold curves are required for plotting")
    for curves, x, y, filename, title in (
        (rejection_curves, "rejection_threshold", "NormAcc", "validation_rejection_threshold_curves", "Validation rejection calibration"),
        (clustering_curves, "clustering_threshold", "b3_f1", "validation_clustering_threshold_curves", "Validation clustering calibration"),
    ):
        fig, axis = plt.subplots(figsize=(7, 4.5))
        sns.lineplot(data=curves, x=x, y=y, hue="model", ax=axis)
        axis.set_title(title)
        axis.set_ylim(0.0, 1.0)
        fig.tight_layout()
        for suffix in ("png", "pdf"):
            path = plot_dir / f"{filename}.{suffix}"
            fig.savefig(path, dpi=300, bbox_inches="tight")
            saved.append(str(path))
        plt.close(fig)

    if oscr_curves.empty:
        raise ValueError("OSCR curve data are required for plotting")
    for dataset, subset in oscr_curves.groupby("dataset", sort=True):
        fig, axis = plt.subplots(figsize=(6.5, 4.5))
        sns.lineplot(data=subset, x="FPR", y="CCR", hue="model", ax=axis, errorbar="sd")
        axis.set_title(f"OSCR — {dataset}")
        axis.set_xlim(0.0, 1.0)
        axis.set_ylim(0.0, 1.0)
        axis.set_xlabel("False Acceptance Rate")
        axis.set_ylabel("Correct Classification Rate")
        fig.tight_layout()
        safe_dataset = str(dataset).replace("/", "_").replace("\\", "_")
        for suffix in ("png", "pdf"):
            path = plot_dir / f"oscr_curve__{safe_dataset}.{suffix}"
            fig.savefig(path, dpi=300, bbox_inches="tight")
            saved.append(str(path))
        plt.close(fig)

    if macro_summary.empty:
        raise ValueError("Dataset-macro summary is empty")
    macro_focus = macro_summary.loc[
        macro_summary["metric"].isin({"Rank-1", "NormAcc", "B3_F1", "End-to-End Discovery B3_F1"})
    ]
    fig, axis = plt.subplots(figsize=(9, 4.5))
    sns.barplot(data=macro_focus, x="metric", y="mean", hue="model", ax=axis, errorbar=None)
    axis.set_title("Dataset-Macro Task Summary")
    axis.set_ylim(0.0, 1.0)
    fig.tight_layout()
    for suffix in ("png", "pdf"):
        path = plot_dir / f"dataset_macro_summary.{suffix}"
        fig.savefig(path, dpi=300, bbox_inches="tight")
        saved.append(str(path))
    plt.close(fig)

    ci_focus = pd.concat(
        [
            closed_summary.loc[closed_summary["metric"].isin(["Rank-1", "mAP"])].assign(task="Closed-set"),
            open_summary.loc[
                open_summary["metric"].isin(["BAKS", "BAUS", "NormAcc", "Known-vs-Novel AUROC", "OSCR-AUC", "DIR@1%FAR"])
            ].assign(task="Open-set"),
            oracle_summary.loc[oracle_summary["metric"].isin(["B3_F1", "ARI"])].assign(task="Oracle-Novel"),
        ],
        ignore_index=True,
    )
    if ci_focus.empty or not {"ci_low", "ci_high"} <= set(ci_focus):
        raise ValueError("Confidence-interval summary rows are required for plotting")
    ci_focus["label"] = (
        ci_focus["task"].astype(str)
        + " / "
        + ci_focus["metric"].astype(str)
        + " / "
        + ci_focus["dataset"].astype(str)
    )
    for model, subset in ci_focus.groupby("model", sort=True):
        subset = subset.sort_values("label").reset_index(drop=True)
        fig_height = max(4.5, 0.28 * len(subset))
        fig, axis = plt.subplots(figsize=(8.5, fig_height))
        positions = np.arange(len(subset))
        # A percentile bootstrap interval need not contain the point estimate.
        # Draw the actual endpoints independently; centering xerr on the mean can
        # produce negative lengths and crash Matplotlib (or misrepresent the CI).
        axis.hlines(positions, subset["ci_low"], subset["ci_high"], color="C0")
        axis.plot(subset["ci_low"], positions, "|", color="C0", markersize=6)
        axis.plot(subset["ci_high"], positions, "|", color="C0", markersize=6)
        axis.plot(subset["mean"], positions, "o", color="C0")
        axis.set_yticks(positions, labels=subset["label"])
        axis.set_xlim(min(0.0, float(subset[["mean", "ci_low"]].min().min())), 1.0)
        axis.set_xlabel("Metric value with identity-block 95% CI")
        axis.set_title(f"Confidence intervals — {model}")
        fig.tight_layout()
        safe_model = str(model).replace("/", "_")
        for suffix in ("png", "pdf"):
            path = plot_dir / f"confidence_intervals__{safe_model}.{suffix}"
            fig.savefig(path, dpi=300, bbox_inches="tight")
            saved.append(str(path))
        plt.close(fig)
    return saved

def final_integrity_check(
    *,
    models_completed: Sequence[str],
    expected_datasets: Sequence[str],
    datasets_completed: Sequence[str],
    split_v2: Mapping[str, Any],
    split_checks: Sequence[Mapping[str, Any]],
    threshold_table: pd.DataFrame,
    result_tables: Mapping[str, pd.DataFrame],
    summary_tables: Mapping[str, pd.DataFrame],
    model_summary: pd.DataFrame,
    runtime_report: pd.DataFrame,
    split_seeds: Sequence[int],
) -> dict[str, Any]:
    failures = []
    if tuple(models_completed) != MODEL_NAMES:
        failures.append(f"models completed {list(models_completed)} != required {list(MODEL_NAMES)}")
    missing_datasets = sorted(set(expected_datasets) - set(datasets_completed))
    if missing_datasets:
        failures.append(f"datasets not completed: {missing_datasets}")
    if not split_v2.get("passed", False):
        failures.append("ApexID Split V2 did not pass")
    if not split_checks or not all(check.get("passed", False) for check in split_checks):
        failures.append("one or more external split integrity checks failed or are absent")
    if RUN_OPEN_SET:
        if set(threshold_table.get("model", [])) != set(MODEL_NAMES):
            failures.append("threshold calibration does not contain exactly the three BHCA models")
        for source_column in ("rejection_threshold_source", "clustering_threshold_source"):
            if source_column not in threshold_table or not threshold_table[source_column].eq("ApexID Split V2 validation").all():
                failures.append(f"{source_column} is absent or not validation-only")
    required_tables = {"closed_set_results"}
    if RUN_OPEN_SET:
        required_tables.update({
            "open_set_results",
            "oracle_novel_clustering_results",
            "end_to_end_discovery_results",
        })
    for name in sorted(required_tables):
        frame = result_tables.get(name)
        if frame is None or frame.empty:
            failures.append(f"required result table {name} is empty")
        elif set(frame["model"].astype(str)) != set(MODEL_NAMES):
            failures.append(f"required result table {name} does not contain all three BHCA models")
        else:
            key_columns = ["model", "dataset", "split_seed"]
            if frame.duplicated(key_columns).any():
                failures.append(f"required result table {name} contains duplicate model/dataset/seed rows")
            expected_rows = len(MODEL_NAMES) * len(expected_datasets) * len(split_seeds)
            if len(frame) != expected_rows:
                failures.append(f"required result table {name} has {len(frame)} rows, expected {expected_rows}")
    finite_requirements = {"closed_set_results": ("Rank-1", "mAP")}
    if RUN_OPEN_SET:
        finite_requirements.update({
            "open_set_results": ("BAKS", "BAUS", "NormAcc", "Known-vs-Novel AUROC", "OSCR-AUC", "DIR@1%FAR"),
            "oracle_novel_clustering_results": ("B3_F1", "ARI"),
            "end_to_end_discovery_results": ("End-to-End Discovery B3_F1", "End-to-End Discovery ARI"),
        })
    for name, columns in finite_requirements.items():
        frame = result_tables.get(name, pd.DataFrame())
        for column in columns:
            if column not in frame or not np.isfinite(pd.to_numeric(frame[column], errors="coerce")).all():
                failures.append(f"{name}.{column} is missing or contains non-finite headline values")
    for name, summary in summary_tables.items():
        if summary.empty or not {"metric", "mean", "ci_low", "ci_high", "bootstrap_replicates"} <= set(summary):
            failures.append(f"summary table {name} is empty or structurally incomplete")
            continue
        bootstrapped = summary["bootstrap_replicates"].astype(int) > 0
        if bootstrapped.any():
            numeric = summary.loc[bootstrapped, ["mean", "ci_low", "ci_high"]].apply(pd.to_numeric)
            if not np.isfinite(numeric.to_numpy(dtype=float)).all():
                failures.append(f"summary table {name} has non-finite requested confidence intervals")
            if not (numeric["ci_low"] <= numeric["ci_high"]).all():
                failures.append(f"summary table {name} has inverted confidence intervals")
    if "oracle" in " ".join(result_tables.get("open_set_results", pd.DataFrame()).columns).lower():
        failures.append("headline open_set_results contains oracle columns")
    if model_summary.empty or model_summary["checkpoint_sha256"].isna().any():
        failures.append("model checkpoint provenance is incomplete")
    if runtime_report.empty:
        failures.append("runtime report is empty")
    return {
        "passed": not failures,
        "status": "PASSED" if not failures else "FAILED",
        "failures": failures,
        "benchmark_version": BENCHMARK_VERSION,
    }

def _markdown_table(frame: pd.DataFrame, max_rows: int | None = None) -> str:
    if frame.empty:
        return "_No executed results._"
    shown = frame.head(max_rows) if max_rows else frame
    try:
        return shown.to_markdown(index=False, floatfmt=".4f")
    except ImportError:
        return "```csv\n" + shown.to_csv(index=False) + "```"

def generate_final_report(
    *,
    output_root: Path,
    integrity: Mapping[str, Any],
    benchmark_manifest: Mapping[str, Any],
    model_summary: pd.DataFrame,
    dataset_summary: pd.DataFrame,
    threshold_calibration: pd.DataFrame,
    closed_summary: pd.DataFrame,
    open_summary: pd.DataFrame,
    oracle_summary: pd.DataFrame,
    discovery_summary: pd.DataFrame,
    dataset_macro: pd.DataFrame,
    paired_comparisons: pd.DataFrame,
    runtime_report: pd.DataFrame,
    limitations: Sequence[str],
) -> tuple[str, dict[str, Any]]:
    output_root = Path(output_root)
    if not runtime_report.empty and (runtime_report["stage"] == "benchmark_wall_clock").any():
        runtime_total = float(runtime_report.loc[runtime_report["stage"] == "benchmark_wall_clock", "seconds"].max())
    else:
        runtime_total = float(runtime_report["seconds"].sum()) if not runtime_report.empty else 0.0
    stage_seconds = (
        runtime_report.loc[runtime_report["stage"] != "benchmark_wall_clock"]
        .groupby("stage", sort=True)["seconds"]
        .sum()
        .to_dict()
        if not runtime_report.empty
        else {}
    )
    cache_rows = runtime_report.loc[runtime_report.get("stage", pd.Series(dtype=str)) == "embedding_extraction"]
    cache_rate = float(cache_rows["cache_hit"].astype(bool).mean()) if not cache_rows.empty else float("nan")

    def metric_lookup(frame: pd.DataFrame, model: str, dataset: str, metric: str) -> float | None:
        if frame.empty:
            return None
        selected = frame.loc[
            (frame["model"].astype(str) == model)
            & (frame["dataset"].astype(str) == dataset)
            & (frame["metric"].astype(str) == metric),
            "mean",
        ]
        return float(selected.iloc[0]) if len(selected) else None

    findings: list[str] = []
    all_datasets = sorted(
        set(closed_summary.get("dataset", pd.Series(dtype=str)).astype(str))
        | set(open_summary.get("dataset", pd.Series(dtype=str)).astype(str))
    )
    for model in MODEL_NAMES:
        for dataset in all_datasets:
            rank1 = metric_lookup(closed_summary, model, dataset, "Rank-1")
            normacc = metric_lookup(open_summary, model, dataset, "NormAcc")
            auroc = metric_lookup(open_summary, model, dataset, "Known-vs-Novel AUROC")
            oracle_b3 = metric_lookup(oracle_summary, model, dataset, "B3_F1")
            end_b3 = metric_lookup(discovery_summary, model, dataset, "End-to-End Discovery B3_F1")
            if rank1 is not None and normacc is not None and rank1 - normacc >= 0.15:
                findings.append(
                    f"{model} / {dataset}: closed-set Rank-1 exceeds open-set NormAcc by "
                    f"{rank1 - normacc:.3f} ({rank1:.3f} vs {normacc:.3f})."
                )
            if auroc is not None and normacc is not None and auroc - normacc >= 0.15:
                findings.append(
                    f"{model} / {dataset}: threshold-free known-vs-novel AUROC exceeds frozen-threshold "
                    f"NormAcc by {auroc - normacc:.3f} ({auroc:.3f} vs {normacc:.3f})."
                )
            if oracle_b3 is not None and end_b3 is not None and oracle_b3 - end_b3 >= 0.15:
                findings.append(
                    f"{model} / {dataset}: Oracle-Novel B³ F1 exceeds coverage-adjusted end-to-end "
                    f"B³ F1 by {oracle_b3 - end_b3:.3f} ({oracle_b3:.3f} vs {end_b3:.3f})."
                )

    ranking_specs = [
        ("Closed-set Re-ID", "closed_set", "Rank-1"),
        ("Open-set identification", "open_set", "NormAcc"),
        ("Novelty detection", "open_set", "Known-vs-Novel AUROC"),
        ("Novel clustering", "oracle_novel_clustering", "B3_F1"),
        ("End-to-end discovery", "end_to_end_discovery", "End-to-End Discovery B3_F1"),
    ]
    rankings: dict[str, list[dict[str, Any]]] = {}
    for label, task, metric in ranking_specs:
        selected = dataset_macro.loc[
            (dataset_macro.get("task", pd.Series(dtype=str)).astype(str) == task)
            & (dataset_macro.get("metric", pd.Series(dtype=str)).astype(str) == metric)
        ].sort_values("mean", ascending=False)
        rankings[label] = [
            {"rank": index + 1, "model": row.model, "mean": float(row.mean), "metric": metric}
            for index, row in enumerate(selected.itertuples(index=False))
        ]
    combined_components = []
    for model in MODEL_NAMES:
        values = [
            ranking_row["mean"]
            for ranking in rankings.values()
            for ranking_row in ranking
            if ranking_row["model"] == model
        ]
        if values:
            combined_components.append(
                {"model": model, "secondary_combined_score": float(np.mean(values)), "components": len(values)}
            )
    combined_components.sort(key=lambda item: item["secondary_combined_score"], reverse=True)
    lines = [
        "# Final Three-Model BHCA Benchmark Report",
        "",
        "## A. Benchmark integrity",
        "",
        f"**BENCHMARK INTEGRITY CHECK: {integrity.get('status', 'UNKNOWN')}**",
        "",
    ]
    if integrity.get("failures"):
        lines.extend(["Failures:", ""] + [f"- {item}" for item in integrity["failures"]] + [""])
    lines.extend(
        [
            "## B. Model information",
            "",
            _markdown_table(model_summary),
            "",
            "## C. Dataset information",
            "",
            _markdown_table(dataset_summary),
            "",
            "## D. Validation threshold calibration",
            "",
            _markdown_table(threshold_calibration),
            "",
            "Rejection thresholds maximize validation NormAcc with the repository's highest-threshold tie-break. "
            "Clustering thresholds maximize validation B³ F1 with the predeclared highest-similarity tie-break. "
            "Both are frozen before external evaluation.",
            "",
            "## E. Closed-set results",
            "",
            _markdown_table(closed_summary),
            "",
            "## F. Open-set identification results",
            "",
            _markdown_table(open_summary),
            "",
            "## G. Oracle-Novel Clustering",
            "",
            _markdown_table(oracle_summary),
            "",
            "This conditional experiment uses the true novel mask only to define the evaluation subset; "
            "its clustering threshold is still validation-calibrated.",
            "",
            "## H. End-to-End Novel Identity Discovery",
            "",
            _markdown_table(discovery_summary),
            "",
            "Rejected samples are selected without ground-truth novelty labels, then clustered with the frozen "
            "validation clustering threshold. Coverage and known contamination are reported alongside clustering quality.",
            "The supplementary coverage-adjusted end-to-end score is conditional novel B³ F1 (or ARI) × "
            "identity-balanced novel discovery recall × (1 − known contamination rate); all factors are also reported separately.",
            "",
            "## I. Cross-dataset generalization",
            "",
            _markdown_table(dataset_macro),
            "",
            "Dataset-Macro Average weights datasets equally. These three BHCA RGB ConvNeXt-Small models share architecture and differ by hierarchical loss.",
            "",
            "## J. Statistical comparison",
            "",
            _markdown_table(paired_comparisons),
            "",
            "A paired difference is described as distinguishable only when its paired 95% bootstrap interval excludes zero; "
            "this is not a claim of universal superiority.",
            "",
            "## K. Failure analysis and task-specific ranking",
            "",
            "Measured patterns and task rankings are generated from executed result DataFrames. No causal explanations are inferred.",
            "",
            "## Runtime Summary",
            "",
            f"- Total benchmark wall-clock time: {runtime_total:.2f} seconds",
            f"- Embedding inference/cache validation: {stage_seconds.get('embedding_extraction', 0.0):.2f} seconds",
            f"- Similarity computation: {stage_seconds.get('similarity_computation', 0.0):.2f} seconds",
            f"- Clustering and calibration: {sum(value for key, value in stage_seconds.items() if 'cluster' in key):.2f} seconds",
            f"- Statistics/bootstrap: {sum(value for key, value in stage_seconds.items() if 'bootstrap' in key or 'statistics' in key):.2f} seconds",
            f"- Report and plot generation: {stage_seconds.get('report_generation', 0.0) + stage_seconds.get('plot_generation', 0.0):.2f} seconds",
            f"- Verified canonical embedding cache reuse rate: {cache_rate:.3f}" if math.isfinite(cache_rate) else "- Cache reuse rate: unavailable",
            "",
            "## Remaining Benchmark Limitations",
            "",
        ]
    )
    if findings:
        insertion = ["Measured findings:", ""] + [f"- {item}" for item in findings] + [""]
    else:
        insertion = ["- No predeclared ≥0.15 task-gap pattern was detected, or results were not executed.", ""]
    ranking_lines = ["Task-specific Dataset-Macro rankings:", ""]
    for label, ranking in rankings.items():
        if ranking:
            ranking_lines.append(
                f"- {label}: "
                + ", ".join(
                    f"{item['rank']}. {item['model']} ({item['mean']:.4f})" for item in ranking
                )
            )
    if combined_components:
        ranking_lines.extend(
            [
                "",
                "Secondary combined score (unweighted mean of the five task-specific Dataset-Macro metrics; not primary):",
                "",
                *[
                    f"- {index + 1}. {item['model']}: {item['secondary_combined_score']:.4f}"
                    for index, item in enumerate(combined_components)
                ],
                "",
            ]
        )
    marker = lines.index("## Runtime Summary")
    lines[marker:marker] = insertion + ranking_lines
    lines.extend([f"- {item}" for item in limitations] or ["- None recorded."])
    markdown = "\n".join(lines).rstrip() + "\n"
    structured = {
        "benchmark_version": BENCHMARK_VERSION,
        "integrity": dict(integrity),
        "manifest": dict(benchmark_manifest),
        "models": model_summary.to_dict(orient="records"),
        "datasets": dataset_summary.to_dict(orient="records"),
        "threshold_calibration": threshold_calibration.to_dict(orient="records"),
        "closed_set_summary": closed_summary.to_dict(orient="records"),
        "open_set_summary": open_summary.to_dict(orient="records"),
        "oracle_novel_clustering_summary": oracle_summary.to_dict(orient="records"),
        "end_to_end_discovery_summary": discovery_summary.to_dict(orient="records"),
        "dataset_macro_summary": dataset_macro.to_dict(orient="records"),
        "paired_model_comparisons": paired_comparisons.to_dict(orient="records"),
        "measured_failure_patterns": findings,
        "task_specific_rankings": rankings,
        "secondary_combined_ranking": combined_components,
        "runtime_summary": {
            "total_wall_clock_seconds": runtime_total,
            "stage_seconds": stage_seconds,
            "embedding_cache_reuse_rate": cache_rate,
        },
        "remaining_limitations": list(limitations),
    }
    atomic_write_text(output_root / "final_benchmark_report.md", markdown)
    write_json(output_root / "final_benchmark_report.json", structured)
    return markdown, structured


In [3]:
# ============================================================
# 3. Final benchmark protocol
# ============================================================

STAGE02_VERSION = "apexid-cvpr-stage02-bhca-closed-1.1.0-no-spac"
STAGE02_PROTOCOL_VERSION = "cvpr_stage02_closed_set_only_v1_no_spac"

# Stage 02 is deliberately closed-set only. Paper-style open-set discovery is
# evaluated by notebooks 03a -> 03b -> 03c.
RUN_OPEN_SET = False

# SPAC dropped from Stage 02 evaluation (still may exist in Stage 00/01 bundles).
EXTERNAL_DATASETS = (
    "bristol_gorillas",
    "elephants",
    "gcn_id",
    "jaguar_antonio",
)
SPLIT_SEEDS = (42, 123, 3407, 2026, 1337)
NOVEL_IDENTITY_RATIO = 0.2
GALLERY_OBSERVATION_GROUP_RATIO = 0.8

# Explicit scientific execution contract.
THRESHOLD_FITTING_IN_STAGE02 = False
EMBEDDING_EXTRACTION_IN_STAGE02 = False
TEST_SPLIT_REGENERATION = False
OFFICIAL_CALIBRATION_SOURCE = "Stage 01 validation only"

# Paper-grade defaults. Lower only for a clearly labeled debugging run.
BOOTSTRAP_REPLICATES = int(os.environ.get("APEXID_BOOTSTRAP_REPLICATES", "1000"))
BOOTSTRAP_SEED = 20260911
SIMILARITY_CHUNK_SIZE = int(os.environ.get("APEXID_SIMILARITY_CHUNK", "4096"))
CLUSTERING_THRESHOLD_CANDIDATES = np.linspace(-1.0, 1.0, 201)

RUN_PAIRED_COMPARISONS = os.environ.get("APEXID_RUN_PAIRED", "1") == "1"
RUN_ORACLE_DIAGNOSTICS = RUN_OPEN_SET and os.environ.get("APEXID_RUN_ORACLE_DIAGNOSTICS", "1") == "1"
RUN_PLOTS = os.environ.get("APEXID_RUN_PLOTS", "1") == "1"

PATHS = ensure_result_tree(OUTPUT_ROOT)
runtime_recorder = RuntimeRecorder()
seed_everything(42)

remaining_limitations = [
    "This benchmark compares three BHCA RGB ConvNeXt-Small backbones (A, SCA, SCSF2) with the same frozen Stage 00 splits and Stage 01 logistic rejector protocol.",
    "Dataset-Macro results cover the four evaluated external datasets (bristol_gorillas, elephants, gcn_id, jaguar_antonio); spac_wild is excluded from Stage 02.",
]

print(json.dumps({
    "protocol": STAGE02_PROTOCOL_VERSION,
    "models": MODEL_NAMES,
    "datasets": EXTERNAL_DATASETS,
    "split_seeds": SPLIT_SEEDS,
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "paired_comparisons": RUN_PAIRED_COMPARISONS,
    "oracle_diagnostics": RUN_ORACLE_DIAGNOSTICS,
    "plots": RUN_PLOTS,
    "run_open_set": RUN_OPEN_SET,
    "threshold_fitting_in_stage02": THRESHOLD_FITTING_IN_STAGE02,
    "embedding_extraction_in_stage02": EMBEDDING_EXTRACTION_IN_STAGE02,
    "test_split_regeneration": TEST_SPLIT_REGENERATION,
}, indent=2))


{
  "protocol": "cvpr_stage02_closed_set_only_v1_no_spac",
  "models": [
    "ApexID_BHCA_A_v2",
    "ApexID_BHCA_SCA_v2",
    "ApexID_BHCA_SCSF2_v2"
  ],
  "datasets": [
    "bristol_gorillas",
    "elephants",
    "gcn_id",
    "jaguar_antonio"
  ],
  "split_seeds": [
    42,
    123,
    3407,
    2026,
    1337
  ],
  "bootstrap_replicates": 1000,
  "paired_comparisons": true,
  "oracle_diagnostics": false,
  "plots": true,
  "run_open_set": false,
  "threshold_fitting_in_stage02": false,
  "embedding_extraction_in_stage02": false,
  "test_split_regeneration": false
}


## 4. Local input resolution and startup preflight

The explicit roots are authoritative. This cell validates Stage 00, Stage 01, all three BHCA calibration entries, all 18 expected embedding caches, every frozen dataset/seed manifest, and output writability before benchmark computation begins. Missing or incompatible artifacts cause an immediate failure; there is no extraction, recalibration, split-generation, or model-substitution fallback.


In [4]:
# ============================================================
# 4. Local Stage 00/01 resolution and strong startup preflight
# ============================================================

def require_file(path: Path, description: str) -> Path:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path


def require_directory(path: Path, description: str) -> Path:
    path = Path(path)
    if not path.is_dir():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path


if STAGE02_NOTEBOOK_PATH.is_file():
    require_file(STAGE02_NOTEBOOK_PATH, "Stage 02 notebook")
else:
    print("WARNING: Stage 02 notebook file not found at", STAGE02_NOTEBOOK_PATH)
    print("Set APEXID_STAGE02_NOTEBOOK if the closed-set source-hash check should run.")
require_directory(STAGE00_ROOT, "frozen Stage 00 root")
require_directory(STAGE01_ROOT, "frozen Stage 01 root")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(dir=OUTPUT_ROOT, prefix=".stage02_write_test_", delete=True):
    pass

STAGE00_MANIFEST_PATH = require_file(
    STAGE00_ROOT / "preparation_manifest.json", "Stage 00 preparation manifest"
)
STAGE01_MANIFEST_PATH = require_file(
    STAGE01_ROOT / "stage01_manifest.json", "Stage 01 manifest"
)
stage00_manifest = json.loads(STAGE00_MANIFEST_PATH.read_text(encoding="utf-8"))
stage01_manifest = json.loads(STAGE01_MANIFEST_PATH.read_text(encoding="utf-8"))
if stage00_manifest.get("status") != "passed":
    raise RuntimeError("Stage 00 manifest is not marked passed")
if not stage00_manifest.get("apexid_split_v2", {}).get("passed", False):
    raise RuntimeError("Stage 00 ApexID Split V2 integrity record is not marked passed")
if stage01_manifest.get("status") != "passed":
    raise RuntimeError("Stage 01 manifest is not marked passed")
# Notebook 01 extracts then calibrates, so embeddings_only is False. Stage 02
# still never extracts; caches are validated below.

stage01_models = set(stage01_manifest.get("models", {}))
completed_models = tuple(stage01_manifest.get("models_completed", ()))
if stage01_models != EXPECTED_MODELS:
    raise RuntimeError(
        f"Stage 01 model metadata set mismatch: {sorted(stage01_models)} != {sorted(EXPECTED_MODELS)}"
    )
if completed_models != MODEL_NAMES:
    raise RuntimeError(
        f"Stage 01 completed model order mismatch: {completed_models} != {MODEL_NAMES}"
    )

FROZEN_THRESHOLDS_PATH = STAGE01_ROOT / "calibration" / "frozen_thresholds.json"
if RUN_OPEN_SET:
    require_file(FROZEN_THRESHOLDS_PATH, "canonical Stage 01 frozen thresholds")
    frozen_thresholds = json.loads(FROZEN_THRESHOLDS_PATH.read_text(encoding="utf-8"))
else:
    # Closed-set Rank-1/mAP does not consume a rejection or clustering threshold.
    frozen_thresholds = {
        model: {
            "rejection_method": "raw_max_cosine",
            "rejection_score_name": "s1",
            "rejection_threshold": 0.0,
            "clustering_threshold": 0.0,
        }
        for model in MODEL_NAMES
    }
if set(frozen_thresholds) != EXPECTED_MODELS:
    raise RuntimeError(
        f"Frozen threshold model set mismatch: {sorted(frozen_thresholds)} != {sorted(EXPECTED_MODELS)}"
    )
compatibility_thresholds = STAGE01_ROOT / "frozen_thresholds.json"
if RUN_OPEN_SET and compatibility_thresholds.is_file():
    compatibility_payload = json.loads(compatibility_thresholds.read_text(encoding="utf-8"))
    if stable_json(compatibility_payload) != stable_json(frozen_thresholds):
        raise RuntimeError("Root-level and canonical Stage 01 frozen threshold files disagree")
calibrators_by_model = {}
calibrator_sha256_by_model = {}
rejection_method_by_model = {}
for model_name in MODEL_NAMES:
    values = frozen_thresholds[model_name]
    required_keys = {"rejection_method", "rejection_score_name", "rejection_threshold", "clustering_threshold"}
    if not required_keys.issubset(values):
        raise ValueError(f"{model_name}: frozen threshold entry lacks {sorted(required_keys - set(values))}")
    if not all(np.isfinite(float(values[key])) for key in ("rejection_threshold", "clustering_threshold")):
        raise ValueError(f"{model_name}: frozen thresholds are not finite")
    method = str(values["rejection_method"])
    if method not in SUPPORTED_REJECTION_METHODS:
        raise ValueError(f"{model_name}: unsupported rejection method {method}")
    rejection_method_by_model[model_name] = method
    if method == "margin_gallery_logistic":
        if values.get("rejection_score_name") != "p_known":
            raise ValueError(f"{model_name}: logistic threshold is not labeled p_known")
        required_logistic = {
            "rejection_probability_threshold", "rejection_calibrator_path",
            "rejection_calibrator_sha256", "logistic_feature_names",
        }
        if not required_logistic.issubset(values):
            raise ValueError(f"{model_name}: logistic threshold entry is incomplete")
        artifact_path = Path(values["rejection_calibrator_path"])
        if not artifact_path.is_absolute():
            artifact_path = STAGE01_ROOT / artifact_path
        artifact_path = require_file(artifact_path, f"frozen rejection calibrator for {model_name}")
        observed_hash = sha256_file(artifact_path)
        expected_hash = str(values["rejection_calibrator_sha256"])
        if observed_hash != expected_hash:
            raise RuntimeError(f"{model_name}: rejection calibrator SHA256 mismatch")
        calibrator = json.loads(artifact_path.read_text(encoding="utf-8"))
        validate_rejection_calibrator(calibrator, expected_model=model_name)
        if tuple(values["logistic_feature_names"]) != LOGISTIC_FEATURE_NAMES:
            raise ValueError(f"{model_name}: frozen-threshold feature order mismatch")
        if calibrator.get("calibration_protocol_version") != stage01_manifest.get("calibration_protocol_version"):
            raise ValueError(f"{model_name}: calibrator protocol version mismatch")
        probability_threshold = float(values["rejection_probability_threshold"])
        if not math.isclose(probability_threshold, float(values["rejection_threshold"]), rel_tol=0.0, abs_tol=0.0):
            raise ValueError(f"{model_name}: compatibility and probability thresholds differ")
        if not math.isclose(probability_threshold, float(calibrator["rejection_probability_threshold"]), rel_tol=0.0, abs_tol=0.0):
            raise ValueError(f"{model_name}: threshold entry and calibrator threshold differ")
        calibrators_by_model[model_name] = calibrator
        calibrator_sha256_by_model[model_name] = observed_hash
    else:
        if values.get("rejection_score_name") != "s1":
            raise ValueError(f"{model_name}: raw rejection threshold is not labeled s1")
        calibrators_by_model[model_name] = None
        calibrator_sha256_by_model[model_name] = None

stage01_embedding_view = STAGE01_ROOT / "embeddings"
if stage01_embedding_view.is_dir():
    EMBEDDINGS_ROOT = stage01_embedding_view.resolve()
    EMBEDDING_ROOT_SOURCE = "stage01_embedding_view"
else:
    embedding_source_root = stage01_manifest.get("embedding_source", {}).get("root")
    if not embedding_source_root:
        raise FileNotFoundError(
            "Stage 01 embeddings directory is missing and stage01_manifest.json "
            "does not define embedding_source.root."
        )
    EMBEDDINGS_ROOT = Path(embedding_source_root).resolve()
    EMBEDDING_ROOT_SOURCE = "stage01_manifest.embedding_source.root"
require_directory(EMBEDDINGS_ROOT, "resolved Stage 01 embedding root")

stage00_dataset_hashes = dict(stage00_manifest.get("dataset_manifest_hashes", {}))
STAGE01_DATASETS = (
    ("apexid_calibration", *EXTERNAL_DATASETS)
    if RUN_OPEN_SET else EXTERNAL_DATASETS
)
if not set(STAGE01_DATASETS).issubset(stage00_dataset_hashes):
    missing_hashes = sorted(set(STAGE01_DATASETS) - set(stage00_dataset_hashes))
    raise RuntimeError(f"Stage 00 dataset-manifest hashes are missing {missing_hashes}")

embedding_cache_paths = {}
for model_name in MODEL_NAMES:
    for dataset_name in STAGE01_DATASETS:
        path = require_file(
            EMBEDDINGS_ROOT / model_name / dataset_name / "embeddings.npz",
            f"embedding cache for model={model_name}, dataset={dataset_name}",
        )
        try:
            with np.load(path, allow_pickle=False) as cached:
                required_keys = {"embeddings", "image_ids", "paths", "identities", "metadata_json"}
                missing_keys = required_keys - set(cached.files)
                if missing_keys:
                    raise ValueError(f"missing keys {sorted(missing_keys)}")
                metadata = json.loads(str(cached["metadata_json"].item()))
        except Exception as exc:
            raise RuntimeError(f"Cannot open embedding cache metadata: {path}: {exc}") from exc
        if metadata.get("model") != model_name or metadata.get("dataset") != dataset_name:
            raise RuntimeError(f"Embedding cache model/dataset provenance mismatch: {path}")
        if metadata.get("dataset_manifest_sha256") != stage00_dataset_hashes[dataset_name]:
            raise RuntimeError(f"Embedding cache logical dataset hash mismatch: {path}")
        embedding_cache_paths[(model_name, dataset_name)] = path

embedding_inventory_path = require_file(
    STAGE01_ROOT / "embedding_inventory.csv", "Stage 01 embedding inventory"
)
preflight_inventory = pd.read_csv(embedding_inventory_path)
if not EXPECTED_MODELS.issubset(set(preflight_inventory.get("model", []).astype(str))):
    raise RuntimeError(
        "Stage 01 embedding inventory is missing required models: "
        f"{sorted(EXPECTED_MODELS - set(preflight_inventory.get('model', []).astype(str)))}"
    )
expected_cache_pairs = {(model, dataset) for model in MODEL_NAMES for dataset in STAGE01_DATASETS}
observed_cache_pairs = set(zip(
    preflight_inventory["model"].astype(str), preflight_inventory["dataset"].astype(str)
))
# Closed-set Stage 02 only needs EXTERNAL_DATASETS. Stage 01 still inventories
# apexid_calibration for open-set/calibration provenance; that extra coverage is OK.
missing_cache_pairs = sorted(expected_cache_pairs - observed_cache_pairs)
if missing_cache_pairs:
    raise RuntimeError(
        "Stage 01 embedding inventory model/dataset coverage is incomplete; missing "
        f"{missing_cache_pairs}"
    )

THRESHOLD_CALIBRATION_PATH = STAGE01_ROOT / "calibration" / "threshold_calibration.csv"
if RUN_OPEN_SET:
    require_file(THRESHOLD_CALIBRATION_PATH, "Stage 01 threshold calibration table")
    preflight_threshold_table = pd.read_csv(THRESHOLD_CALIBRATION_PATH)
    if set(preflight_threshold_table.get("model", [])) != EXPECTED_MODELS:
        raise RuntimeError("Stage 01 threshold calibration table has the wrong model set")
    for source_column in ("rejection_threshold_source", "clustering_threshold_source"):
        if source_column not in preflight_threshold_table:
            raise RuntimeError(f"Stage 01 threshold table lacks {source_column}")
        if not preflight_threshold_table[source_column].eq("ApexID Split V2 validation").all():
            raise RuntimeError(f"Stage 01 {source_column} is not validation-only")
else:
    preflight_threshold_table = pd.DataFrame({"model": MODEL_NAMES})

required_stage00_files = []
for dataset_name in EXTERNAL_DATASETS:
    required_stage00_files.append(
        require_file(
            STAGE00_ROOT / "manifests" / f"{dataset_name}_canonical_manifest.csv",
            f"Stage 00 canonical manifest for {dataset_name}",
        )
    )
    for split_seed in SPLIT_SEEDS:
        split_root = STAGE00_ROOT / "splits" / dataset_name / f"seed_{split_seed}"
        require_file(split_root / "split_manifest.csv", f"split manifest {dataset_name}/seed_{split_seed}")
        require_file(split_root / "split_record.json", f"split record {dataset_name}/seed_{split_seed}")
        require_file(split_root / "split_diagnostics.json", f"split diagnostics {dataset_name}/seed_{split_seed}")
        manifest_key = f"{dataset_name}::seed_{split_seed}"
        if manifest_key not in stage00_manifest.get("split_manifest_hashes", {}):
            raise RuntimeError(f"Stage 00 manifest lacks frozen split hash {manifest_key}")

print("=" * 65)
print("ApexID CVPR Stage 02 — Final Three-Model BHCA Benchmark")
print("=" * 65)
print("Stage 00:\n", STAGE00_ROOT)
print("Stage 01:\n", STAGE01_ROOT)
print("Embeddings:\n", EMBEDDINGS_ROOT)
print("Embedding root source:\n", EMBEDDING_ROOT_SOURCE)
print("Output:\n", OUTPUT_ROOT)
print("Models:")
for model_name in MODEL_NAMES:
    print(f"[OK] {model_name}")
print("Threshold fitting in official Stage 02 path:\nDISABLED")
print("Embedding extraction:\nDISABLED")
print("Test split regeneration:\nDISABLED")
print("Official calibration source:\nStage 01 validation only")
print("=" * 65)
print("STARTUP PREFLIGHT: PASSED")


Set APEXID_STAGE02_NOTEBOOK if the closed-set source-hash check should run.
ApexID CVPR Stage 02 — Final Three-Model BHCA Benchmark
Stage 00:
 /kaggle/input/datasets/seifelkerdany/apexid-cvpr-prepared-bundle/apexid_cvpr_prepared
Stage 01:
 /kaggle/input/datasets/mohamedbassat/apexid-cvpr-stage01-bhca-logistic/apexid_cvpr_stage01_bhca_logistic
Embeddings:
 /kaggle/input/datasets/mohamedbassat/apexid-cvpr-stage01-bhca-logistic/apexid_cvpr_stage01_bhca_logistic/embeddings
Embedding root source:
 stage01_embedding_view
Output:
 /kaggle/working/apexid_cvpr_final_benchmark_bhca_logistic
Models:
[OK] ApexID_BHCA_A_v2
[OK] ApexID_BHCA_SCA_v2
[OK] ApexID_BHCA_SCSF2_v2
Threshold fitting in official Stage 02 path:
DISABLED
Embedding extraction:
DISABLED
Test split regeneration:
DISABLED
Official calibration source:
Stage 01 validation only
STARTUP PREFLIGHT: PASSED


## 5. Load frozen benchmark state

Only Stage 00 canonical manifests/splits and Stage 01 validation-frozen calibration are loaded. Source images are never opened. Official Stage 02 predictions cannot invoke the oracle fitting branch.


In [5]:
# ============================================================
# 5. Frozen benchmark state
# ============================================================

dataset_manifest_hashes = dict(stage00_manifest["dataset_manifest_hashes"])
split_v2_audit = dict(stage00_manifest["apexid_split_v2"])
repo_provenance = dict(stage00_manifest.get("repository", {}))

dataset_summary = pd.read_csv(STAGE00_ROOT / "tables" / "dataset_summary.csv")
if "dataset" in dataset_summary.columns:
    dataset_summary = dataset_summary.loc[
        dataset_summary["dataset"].astype(str).isin(EXTERNAL_DATASETS)
    ].reset_index(drop=True)
split_integrity_checks = pd.read_csv(
    STAGE00_ROOT / "tables" / "split_integrity_checks.csv"
)
if "dataset" in split_integrity_checks.columns:
    # Stage 00 may still inventory spac_wild; Stage 02 only checks evaluated animals.
    split_integrity_checks = split_integrity_checks.loc[
        split_integrity_checks["dataset"].astype(str).isin(EXTERNAL_DATASETS)
    ].reset_index(drop=True)
if len(split_integrity_checks) != len(EXTERNAL_DATASETS) * len(SPLIT_SEEDS):
    raise RuntimeError(
        "Stage 00 split-integrity table has an unexpected number of rows for "
        f"EXTERNAL_DATASETS={list(EXTERNAL_DATASETS)} "
        f"(got {len(split_integrity_checks)}, expected "
        f"{len(EXTERNAL_DATASETS) * len(SPLIT_SEEDS)})"
    )
if not split_integrity_checks["passed"].astype(str).str.lower().eq("true").all():
    raise RuntimeError("One or more frozen Stage 00 split integrity checks failed")
# Keep only hashes for datasets actually evaluated in this Stage 02 run.
dataset_manifest_hashes = {
    name: dataset_manifest_hashes[name]
    for name in EXTERNAL_DATASETS
    if name in dataset_manifest_hashes
}
if set(dataset_manifest_hashes) != set(EXTERNAL_DATASETS):
    missing = sorted(set(EXTERNAL_DATASETS) - set(dataset_manifest_hashes))
    raise RuntimeError(f"Stage 00 dataset-manifest hashes missing for evaluation set: {missing}")

canonical_frames = {}
for dataset_name in EXTERNAL_DATASETS:
    frame = pd.read_csv(
        STAGE00_ROOT / "manifests" / f"{dataset_name}_canonical_manifest.csv",
        dtype=str,
    ).fillna("")
    if not {"image_id", "identity", "dataset", "path"}.issubset(frame):
        raise ValueError(f"{dataset_name}: canonical manifest is structurally incomplete")
    if frame["image_id"].duplicated().any() or set(frame["dataset"]) != {dataset_name}:
        raise ValueError(f"{dataset_name}: canonical manifest identity/order contract is invalid")
    canonical_frames[dataset_name] = frame
external_frames = {name: canonical_frames[name] for name in EXTERNAL_DATASETS}

split_manifests = {}
split_manifest_hashes = {}
for dataset_name in EXTERNAL_DATASETS:
    canonical_ids = set(canonical_frames[dataset_name]["image_id"].astype(str))
    for split_seed in SPLIT_SEEDS:
        path = STAGE00_ROOT / "splits" / dataset_name / f"seed_{split_seed}" / "split_manifest.csv"
        split_frame = pd.read_csv(path, dtype=str).fillna("")
        if split_frame["image_id"].duplicated().any():
            raise ValueError(f"{dataset_name}/seed_{split_seed}: duplicate split image_id")
        if set(split_frame["image_id"].astype(str)) != canonical_ids:
            raise ValueError(f"{dataset_name}/seed_{split_seed}: split membership differs from canonical manifest")
        split_manifests[(dataset_name, split_seed)] = split_frame
        key = f"{dataset_name}::seed_{split_seed}"
        split_manifest_hashes[(dataset_name, split_seed)] = stage00_manifest["split_manifest_hashes"][key]

# Closed-set evaluation does not load or export rejection/clustering calibration.
model_summary = pd.read_csv(STAGE01_ROOT / "tables" / "model_summary.csv")
if set(model_summary.get("model", [])) != EXPECTED_MODELS:
    raise RuntimeError("Stage 01 model summary has an incompatible model set")
if RUN_OPEN_SET:
    threshold_calibration = pd.read_csv(THRESHOLD_CALIBRATION_PATH)
    rejection_curves = pd.read_csv(STAGE01_ROOT / "calibration" / "rejection_threshold_curves.csv")
    clustering_curves = pd.read_csv(STAGE01_ROOT / "calibration" / "clustering_threshold_curves.csv")
    for name, frame in (
        ("threshold calibration", threshold_calibration),
        ("rejection curves", rejection_curves),
        ("clustering curves", clustering_curves),
    ):
        if set(frame.get("model", [])) != EXPECTED_MODELS:
            raise RuntimeError(f"Stage 01 {name} has an incompatible model set")
    atomic_write_csv(PATHS["tables"] / "threshold_calibration.csv", threshold_calibration)
    atomic_write_csv(PATHS["calibration"] / "threshold_calibration.csv", threshold_calibration)
    atomic_write_csv(PATHS["calibration"] / "rejection_threshold_curves.csv", rejection_curves)
    atomic_write_csv(PATHS["calibration"] / "clustering_threshold_curves.csv", clustering_curves)
else:
    threshold_calibration = pd.DataFrame(columns=["model"])
    rejection_curves = pd.DataFrame()
    clustering_curves = pd.DataFrame()
    print("Open-set calibration loading: SKIPPED")

model_metadata_by_name = dict(stage01_manifest["models"])
models_completed = list(MODEL_NAMES)

display(model_summary)


Open-set calibration loading: SKIPPED


,model,model_identifier,checkpoint_path,checkpoint_sha256,config_path,config_sha256,config_provenance,revision,embedding_dimension,input_resolution,parameters,l2_normalized,transform,mixed_precision_validation,selected_batch_size,checkpoint_provenance_type
0,ApexID_BHCA_A_v2,/kaggle/input/models/mohamedbassat/apexid-conv...,/kaggle/input/models/mohamedbassat/apexid-conv...,a4981988409c2a6d85a8c9224eb94f7bbd5cb1d55d8014...,rgb_bhca_eval_fallback_imagenet,6a66af69a02c2f88fcef08108a6f798bc70e02f28e84a0...,sidecar YAML next to best_backbone.pt,137,2048,"[440,440]",unknown,True,"{""config_sha256"":""6a66af69a02c2f88fcef08108a6f...","{""extraction_device"":""cuda_if_available"",""sour...",64,bhca_backbone_sidecar
1,ApexID_BHCA_SCA_v2,/kaggle/input/models/mohamedbassat/apexid-conv...,/kaggle/input/models/mohamedbassat/apexid-conv...,674cc88a2f3dd430bb1db84163c84a072a3a79941e56b2...,rgb_bhca_eval_fallback_imagenet,6a66af69a02c2f88fcef08108a6f798bc70e02f28e84a0...,sidecar YAML next to best_backbone.pt,137,2048,"[440,440]",unknown,True,"{""config_sha256"":""6a66af69a02c2f88fcef08108a6f...","{""extraction_device"":""cuda_if_available"",""sour...",64,bhca_backbone_sidecar
2,ApexID_BHCA_SCSF2_v2,/kaggle/input/models/mohamedbassat/apexid-conv...,/kaggle/input/models/mohamedbassat/apexid-conv...,a53dbd738fc9f1e69b86fa57ccfa4db9a10d41cbcc57fd...,rgb_bhca_eval_fallback_imagenet,6a66af69a02c2f88fcef08108a6f798bc70e02f28e84a0...,sidecar YAML next to best_backbone.pt,125,2048,"[440,440]",unknown,True,"{""config_sha256"":""6a66af69a02c2f88fcef08108a6f...","{""extraction_device"":""cuda_if_available"",""sour...",64,bhca_backbone_sidecar


## 6. Frozen embedding access

Each canonical cache is loaded from the resolved `EMBEDDINGS_ROOT` once per model/dataset evaluation block. Missing, mislabelled, reordered, or provenance-incompatible caches fail immediately. The five frozen split seeds only select rows from these arrays; they never trigger inference.


In [6]:
# ============================================================
# 6. Canonical frozen embedding access
# ============================================================

def canonical_embeddings_for(model_name, dataset_name):
    if model_name not in EXPECTED_MODELS:
        raise KeyError(f"Model is not part of the sealed Stage 02 protocol: {model_name}")
    if dataset_name not in EXTERNAL_DATASETS:
        raise KeyError(f"Dataset is not part of the sealed Stage 02 protocol: {dataset_name}")
    path = EMBEDDINGS_ROOT / model_name / dataset_name / "embeddings.npz"
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing embedding cache for model={model_name}, dataset={dataset_name}: {path}. "
            "Stage 02 cannot extract or regenerate embeddings."
        )
    embeddings, metadata, image_ids, stored_paths, stored_identities = load_embedding_cache(path)
    if metadata.get("model") != model_name or metadata.get("dataset") != dataset_name:
        raise RuntimeError(f"Embedding provenance model/dataset mismatch: {path}")
    if metadata.get("dataset_manifest_sha256") != dataset_manifest_hashes[dataset_name]:
        raise RuntimeError(f"Embedding logical dataset hash mismatch: {path}")
    if len(np.unique(image_ids.astype(str))) != len(image_ids):
        raise RuntimeError(f"Embedding cache contains duplicate image identifiers: {path}")

    canonical = canonical_frames[dataset_name].copy()
    expected_ids = canonical["image_id"].astype(str).to_numpy()
    if len(image_ids) != len(expected_ids) or set(image_ids.astype(str)) != set(expected_ids):
        raise RuntimeError(f"Embedding cache membership differs from frozen Stage 00: {path}")
    indexed = canonical.set_index(canonical["image_id"].astype(str), drop=False)
    frame = indexed.loc[image_ids.astype(str)].reset_index(drop=True)
    if not np.array_equal(stored_identities.astype(str), frame["identity"].astype(str).to_numpy()):
        raise RuntimeError(f"Embedding labels do not align with frozen Stage 00: {path}")
    if not np.array_equal(stored_paths.astype(str), frame["path"].astype(str).to_numpy()):
        raise RuntimeError(f"Embedding paths do not align with frozen Stage 00: {path}")
    return frame, embeddings, metadata


def rows_for_roles(frame, embeddings, split_manifest, role_column, roles):
    if role_column not in split_manifest:
        raise KeyError(f"Frozen split manifest lacks role column {role_column}")
    position = {
        image_id: index
        for index, image_id in enumerate(frame["image_id"].astype(str))
    }
    selected = (
        split_manifest.loc[split_manifest[role_column].isin(roles)]
        .copy()
        .reset_index(drop=True)
    )
    missing = sorted(set(selected["image_id"].astype(str)) - set(position))
    if missing:
        raise RuntimeError(f"Frozen split references image IDs absent from embeddings: {missing[:5]}")
    indices = np.asarray(
        [position[image_id] for image_id in selected["image_id"].astype(str)],
        dtype=int,
    )
    return selected, embeddings[indices]


print("Stage 01 caches connected through:", EMBEDDINGS_ROOT)
print("No model inference or embedding extraction can run.")


Stage 01 caches connected through: /kaggle/input/datasets/mohamedbassat/apexid-cvpr-stage01-bhca-logistic/apexid_cvpr_stage01_bhca_logistic/embeddings
No model inference or embedding extraction can run.


## 7. Closed-set identification

Primary closed-set evaluation keeps conventional image-gallery retrieval for Rank-k/mAP comparability.

In [7]:
closed_rows = []
closed_detail_paths = {}

for model_name in MODEL_NAMES:
    for dataset_name in external_frames:
        frame, embeddings, embedding_metadata = canonical_embeddings_for(model_name, dataset_name)
        for split_seed in SPLIT_SEEDS:
            split_manifest = split_manifests[(dataset_name, split_seed)]
            gallery_rows, gallery_embeddings = rows_for_roles(
                frame, embeddings, split_manifest, "closed_role", {"gallery"}
            )
            query_rows, query_embeddings = rows_for_roles(
                frame, embeddings, split_manifest, "closed_role", {"query_known"}
            )
            # Closed-set cache semantics are deliberately independent of rejection calibration.
            closed_meta = {
                "schema_version": 1,
                "benchmark_version": "apexid-cvpr-closed-set-frozen-v1",
                "model": model_name,
                "dataset": dataset_name,
                "split_seed": int(split_seed),
                "embedding_provenance_sha256": sha256_bytes(stable_json(dict(embedding_metadata)).encode("utf-8")),
                "split_manifest_sha256": split_manifest_hashes[(dataset_name, split_seed)],
                "gallery_scoring": "image-gallery retrieval",
            }
            closed_cache_path = PATHS["scores"] / model_name / dataset_name / f"seed_{split_seed}_closed_results.npz"
            closed_cache_hit = False
            start = time.perf_counter()
            if closed_cache_path.is_file():
                try:
                    metrics, query_details = load_closed_result_cache(
                        closed_cache_path, expected_metadata=closed_meta
                    )
                    assert np.array_equal(query_details["image_id"], query_rows["image_id"].astype(str))
                    closed_cache_hit = True
                except Exception as exc:
                    print("CLOSED SCORE CACHE MISS —", model_name, dataset_name, split_seed, exc)
            if not closed_cache_hit:
                metrics, query_details = closed_set_retrieval_metrics(
                    query_embeddings=query_embeddings,
                    query_identities=query_rows["identity"].astype(str),
                    gallery_embeddings=gallery_embeddings,
                    gallery_identities=gallery_rows["identity"].astype(str),
                    chunk_size=SIMILARITY_CHUNK_SIZE,
                )
                query_details.insert(0, "image_id", query_rows["image_id"].astype(str).to_numpy())
                save_closed_result_cache(
                    closed_cache_path, metadata=closed_meta, metrics=metrics, query_details=query_details
                )
            runtime_recorder.record(
                "closed_set_metric_computation", time.perf_counter() - start,
                model=model_name, dataset=dataset_name, split_seed=split_seed,
                number_queries=len(query_rows), number_gallery_identities=gallery_rows["identity"].nunique(),
                cache_hit=closed_cache_hit,
            )
            detail_path = PATHS["closed_set"] / model_name / dataset_name / f"seed_{split_seed}_query_details.csv"
            atomic_write_csv(detail_path, query_details)
            bootstrap_path = PATHS["statist" \
            "ics"] / "closed_set" / model_name / dataset_name / f"seed_{split_seed}.npz"
            bootstrap_start = time.perf_counter()
            bootstrap_metadata = {
                "unit": "query identity block", "model": model_name, "dataset": dataset_name,
                "split_seed": split_seed, "score_provenance": sha256_bytes(stable_json(closed_meta).encode("utf-8")),
                "replicates": BOOTSTRAP_REPLICATES, "bootstrap_seed": BOOTSTRAP_SEED,
            }
            bootstrap_values, bootstrap_cache_hit = verified_bootstrap(
                bootstrap_path,
                expected_metadata=bootstrap_metadata,
                expected_metrics=("Rank-1", "mAP"),
                expected_replicates=BOOTSTRAP_REPLICATES,
                compute=lambda: bootstrap_closed_details(
                    query_details,
                    replicates=BOOTSTRAP_REPLICATES,
                    seed=stable_seed(BOOTSTRAP_SEED, "closed", model_name, dataset_name, split_seed),
                ),
            )
            runtime_recorder.record(
                "bootstrap", time.perf_counter() - bootstrap_start,
                model=model_name, dataset=dataset_name, split_seed=split_seed,
                number_queries=len(query_rows), experiment="closed_set",
                cache_hit=bootstrap_cache_hit,
            )
            closed_detail_paths[(model_name, dataset_name, split_seed)] = detail_path
            closed_rows.append({
                "model": model_name, "dataset": dataset_name, "split_seed": split_seed,
                "n_gallery_samples": len(gallery_rows), "n_query_samples": len(query_rows),
                "n_gallery_identities": gallery_rows["identity"].nunique(),
                **metrics, "bootstrap_path": str(bootstrap_path),
            })

closed_set_results = pd.DataFrame(closed_rows)
atomic_write_csv(PATHS["closed_set"] / "closed_set_results.csv", closed_set_results)
atomic_write_csv(PATHS["tables"] / "closed_set_results.csv", closed_set_results)
display(closed_set_results.head())


# Closed-set regression guard: the metric implementation is byte-identical to
# the function in the pre-logistic backup. With the same frozen arrays this is
# a deterministic before/after equality guarantee, independent of stale result caches.
if STAGE02_NOTEBOOK_PATH.is_file():
    _stage02_payload = json.loads(STAGE02_NOTEBOOK_PATH.read_text(encoding="utf-8"))
    _runtime_source = "".join(_stage02_payload["cells"][4]["source"])
    _runtime_tree = ast.parse(_runtime_source)
    _closed_node = next(
        node for node in _runtime_tree.body
        if isinstance(node, ast.FunctionDef) and node.name == "closed_set_retrieval_metrics"
    )
    _closed_source = ast.get_source_segment(_runtime_source, _closed_node)
    _closed_source_sha256 = sha256_bytes(_closed_source.encode("utf-8"))
    closed_set_regression = {
        "passed": _closed_source_sha256 == CLOSED_SET_PRE_LOGISTIC_FUNCTION_SHA256,
        "comparison": "pre-logistic versus logistic notebook closed_set_retrieval_metrics source",
        "pre_logistic_function_sha256": CLOSED_SET_PRE_LOGISTIC_FUNCTION_SHA256,
        "current_function_sha256": _closed_source_sha256,
        "frozen_embedding_inputs": True,
        "deterministic_algorithm": True,
        "result_semantics": "numerically identical for identical frozen input arrays",
        "historical_result_caches_used_as_reference": False,
    }
    if not closed_set_regression["passed"]:
        raise RuntimeError("Closed-set implementation changed relative to the pre-logistic notebook")
    write_json(PATHS["tables"] / "closed_set_regression_check.json", closed_set_regression)
    print("CLOSED-SET REGRESSION: PASSED; implementation SHA256 =", _closed_source_sha256)
else:
    closed_set_regression = {
        "passed": None,
        "skipped": True,
        "reason": "Stage 02 notebook file not found on disk; source-hash check skipped",
        "comparison": "pre-logistic versus logistic notebook closed_set_retrieval_metrics source",
        "pre_logistic_function_sha256": CLOSED_SET_PRE_LOGISTIC_FUNCTION_SHA256,
        "current_function_sha256": None,
        "frozen_embedding_inputs": True,
        "deterministic_algorithm": True,
        "result_semantics": "numerically identical for identical frozen input arrays",
        "historical_result_caches_used_as_reference": False,
    }
    write_json(PATHS["tables"] / "closed_set_regression_check.json", closed_set_regression)
    print("CLOSED-SET REGRESSION: SKIPPED (Stage 02 notebook file not found)")


,model,dataset,split_seed,n_gallery_samples,n_query_samples,n_gallery_identities,Rank-1,Rank-5,Rank-10,CMC@20,CMC@50,mAP,identity-balanced Rank-1,effective_rank_5,effective_rank_10,effective_rank_20,effective_rank_50,bootstrap_path
0,ApexID_BHCA_A_v2,bristol_gorillas,42,4215,1109,7,0.918846,0.963931,0.986474,0.996393,1.000000,0.255055,0.919278,5,10,20,50,/kaggle/working/apexid_cvpr_final_benchmark_bh...
1,ApexID_BHCA_A_v2,bristol_gorillas,123,4234,1090,7,0.894495,0.960550,0.982569,0.993578,0.998165,0.263198,0.895014,5,10,20,50,/kaggle/working/apexid_cvpr_final_benchmark_bh...
2,ApexID_BHCA_A_v2,bristol_gorillas,3407,4240,1084,7,0.902214,0.972325,0.988930,0.994465,0.995387,0.263467,0.903854,5,10,20,50,/kaggle/working/apexid_cvpr_final_benchmark_bh...
3,ApexID_BHCA_A_v2,bristol_gorillas,2026,4218,1106,7,0.909584,0.964738,0.986438,0.995479,0.999096,0.259442,0.913529,5,10,20,50,/kaggle/working/apexid_cvpr_final_benchmark_bh...
4,ApexID_BHCA_A_v2,bristol_gorillas,1337,4226,1098,7,0.920765,0.976321,0.987250,0.994536,1.000000,0.263583,0.922048,5,10,20,50,/kaggle/working/apexid_cvpr_final_benchmark_bh...


CLOSED-SET REGRESSION: SKIPPED (Stage 02 notebook file not found)


## 8. Open-set identification and discovery

The official knownness score dispatches from each Stage 01 threshold entry. Logistic mode requires top-2 centroid scores, manually computes frozen `p_known`, and applies the frozen probability threshold. Raw mode remains available only when Stage 01 explicitly declares it. Missing or invalid logistic artifacts are fatal; there is no fallback.

Identity assignment remains nearest-centroid cosine. End-to-end discovery uses the same frozen known/novel decision before applying the unchanged frozen clustering method.


In [8]:
open_rows = []
oracle_rows = []
discovery_rows = []
oracle_diagnostic_rows = []
oscr_curve_frames = []
open_score_paths = {}
oracle_prediction_paths = {}

if set(frozen_thresholds) != EXPECTED_MODELS:
    raise RuntimeError("Official evaluation cannot start without all three Stage 01 threshold entries")
if THRESHOLD_FITTING_IN_STAGE02:
    raise RuntimeError("Official Stage 02 threshold fitting must remain disabled")

for model_name in (MODEL_NAMES if RUN_OPEN_SET else ()):
    threshold_entry = frozen_thresholds[model_name]
    rejection_method = threshold_entry["rejection_method"]
    rejection_score_name = threshold_entry["rejection_score_name"]
    rejection_threshold = float(threshold_entry["rejection_threshold"])
    clustering_threshold = float(threshold_entry["clustering_threshold"])
    calibrator = calibrators_by_model[model_name]
    calibrator_sha256 = calibrator_sha256_by_model[model_name]
    for dataset_name in external_frames:
        frame, embeddings, embedding_metadata = canonical_embeddings_for(model_name, dataset_name)
        for split_seed in SPLIT_SEEDS:
            split_manifest = split_manifests[(dataset_name, split_seed)]
            gallery_rows, gallery_embeddings = rows_for_roles(
                frame, embeddings, split_manifest, "open_role", {"gallery"}
            )
            query_rows, query_embeddings = rows_for_roles(
                frame, embeddings, split_manifest, "open_role", {"query_known", "query_novel"}
            )
            is_novel = query_rows["open_role"].eq("query_novel").to_numpy()
            identity_gallery = build_identity_gallery(gallery_embeddings, gallery_rows["identity"].astype(str))
            gallery_identity_count = len(identity_gallery.identities)
            if gallery_identity_count < 2:
                raise RuntimeError("Open-set logistic features require at least two gallery identities")
            feature_names = LOGISTIC_FEATURE_NAMES if rejection_method == "margin_gallery_logistic" else ("s1",)
            score_meta = score_cache_metadata(
                model=model_name, dataset=dataset_name, split_seed=split_seed,
                embedding_metadata=embedding_metadata,
                split_manifest_sha256=split_manifest_hashes[(dataset_name, split_seed)],
                rejection_method=rejection_method,
                calibration_protocol_version=stage01_manifest["calibration_protocol_version"],
                calibrator_sha256=calibrator_sha256,
                feature_names=feature_names,
                gallery_identity_count=gallery_identity_count,
            )
            score_path = PATHS["scores"] / model_name / dataset_name / f"seed_{split_seed}_open_scores.npz"
            score_cache_hit = False
            if score_path.is_file():
                try:
                    scores = load_open_score_cache(score_path, expected_metadata=score_meta)
                    assert np.array_equal(scores["image_ids"], query_rows["image_id"].astype(str))
                    score_cache_hit = True
                except Exception as exc:
                    print("SCORE CACHE MISS —", model_name, dataset_name, split_seed, exc)
            if not score_cache_hit:
                start = time.perf_counter()
                scored = score_identity_gallery(
                    query_embeddings, identity_gallery,
                    chunk_size=SIMILARITY_CHUNK_SIZE, top_k=2,
                )
                s1 = scored["top_scores"][:, 0].astype(np.float64)
                s2 = scored["top_scores"][:, 1].astype(np.float64)
                margin = s1 - s2
                gallery_size_values = np.full(len(s1), gallery_identity_count, dtype=np.int64)
                log_gallery_size_values = np.full(len(s1), math.log(float(gallery_identity_count)), dtype=np.float64)
                if rejection_method == "margin_gallery_logistic":
                    if calibrator is None:
                        raise RuntimeError(f"{model_name}: logistic method has no loaded calibrator")
                    p_known = apply_margin_gallery_logistic(s1, s2, gallery_identity_count, calibrator)
                    knownness_scores = p_known
                elif rejection_method == "raw_max_cosine":
                    p_known = np.full(len(s1), np.nan, dtype=np.float64)
                    knownness_scores = s1
                else:
                    raise AssertionError("Unreachable rejection method")
                runtime_recorder.record(
                    "similarity_computation", time.perf_counter() - start,
                    model=model_name, dataset=dataset_name, split_seed=split_seed,
                    number_queries=len(query_rows), number_gallery_identities=len(identity_gallery.identities),
                    cache_hit=False,
                )
                save_open_score_cache(
                    score_path, metadata=score_meta,
                    image_ids=query_rows["image_id"].astype(str),
                    true_identities=query_rows["identity"].astype(str),
                    is_novel=is_novel,
                    predicted_identities=scored["predicted_identities"],
                    s1=s1, s2=s2, margin=margin,
                    gallery_size=gallery_size_values,
                    log_gallery_size=log_gallery_size_values,
                    p_known=p_known,
                    knownness_scores=knownness_scores,
                )
                scores = load_open_score_cache(score_path, expected_metadata=score_meta)
            else:
                runtime_recorder.record(
                    "similarity_computation", 0.0, model=model_name, dataset=dataset_name,
                    split_seed=split_seed, number_queries=len(query_rows),
                    number_gallery_identities=len(identity_gallery.identities), cache_hit=True,
                )
            open_score_paths[(model_name, dataset_name, split_seed)] = score_path

            metrics = open_set_metrics_from_scores(
                knownness_scores=scores["knownness_scores"],
                predicted_identities=scores["predicted_identities"],
                true_identities=scores["true_identities"],
                is_novel=scores["is_novel"],
                rejection_threshold=rejection_threshold,
            )
            open_bootstrap_path = PATHS["statistics"] / "open_set" / model_name / dataset_name / f"seed_{split_seed}.npz"
            bootstrap_start = time.perf_counter()
            open_bootstrap_metadata = {
                "unit": "stratified known/novel identity block", "model": model_name,
                "dataset": dataset_name, "split_seed": split_seed,
                "score_provenance": sha256_bytes(stable_json(score_meta).encode("utf-8")),
                "rejection_threshold": rejection_threshold,
                "replicates": BOOTSTRAP_REPLICATES, "bootstrap_seed": BOOTSTRAP_SEED,
            }
            open_bootstrap, open_bootstrap_hit = verified_bootstrap(
                open_bootstrap_path,
                expected_metadata=open_bootstrap_metadata,
                expected_metrics=OPEN_BOOTSTRAP_METRICS,
                expected_replicates=BOOTSTRAP_REPLICATES,
                compute=lambda: bootstrap_open_scores(
                    knownness_scores=scores["knownness_scores"],
                    predicted_identities=scores["predicted_identities"],
                    true_identities=scores["true_identities"],
                    is_novel=scores["is_novel"],
                    rejection_threshold=rejection_threshold,
                    replicates=BOOTSTRAP_REPLICATES,
                    seed=stable_seed(BOOTSTRAP_SEED, "open", model_name, dataset_name, split_seed),
                ),
            )
            runtime_recorder.record(
                "bootstrap", time.perf_counter() - bootstrap_start,
                model=model_name, dataset=dataset_name, split_seed=split_seed,
                number_queries=len(query_rows), experiment="open_set",
                cache_hit=open_bootstrap_hit,
            )
            open_rows.append({
                "model": model_name, "dataset": dataset_name, "split_seed": split_seed,
                "gallery_scoring": "centroid", "threshold_source": "ApexID Split V2 validation",
                "rejection_method": rejection_method,
                "rejection_score_name": rejection_score_name,
                "threshold_semantics": "probability" if rejection_method == "margin_gallery_logistic" else "raw cosine",
                "calibrator_sha256": calibrator_sha256,
                **metrics, "bootstrap_path": str(open_bootstrap_path),
            })
            known = ~scores["is_novel"]
            curve = oscr_curve(
                known_scores=scores["knownness_scores"][known],
                novel_scores=scores["knownness_scores"][~known],
                known_correct=scores["predicted_identities"][known] == scores["true_identities"][known],
            )
            curve["model"], curve["dataset"], curve["split_seed"] = model_name, dataset_name, split_seed
            oscr_curve_frames.append(curve)

            # Oracle-Novel Clustering: build one hierarchy, then reuse it for the frozen cut
            # and the isolated oracle diagnostic sweep.
            cluster_start = time.perf_counter()
            oracle_hierarchy = build_cluster_hierarchy(query_embeddings[is_novel])
            oracle_predicted = cut_cluster_hierarchy(oracle_hierarchy, clustering_threshold)
            oracle_metrics = discovery_metric_bundle(
                query_rows.loc[is_novel, "identity"].astype(str), oracle_predicted
            )
            runtime_recorder.record(
                "clustering", time.perf_counter() - cluster_start,
                model=model_name, dataset=dataset_name, split_seed=split_seed,
                number_queries=int(is_novel.sum()), experiment="oracle_novel",
            )
            oracle_bootstrap_path = PATHS["statistics"] / "oracle_novel" / model_name / dataset_name / f"seed_{split_seed}.npz"
            bootstrap_start = time.perf_counter()
            oracle_bootstrap_metadata = {
                "unit": "novel identity block", "model": model_name, "dataset": dataset_name,
                "split_seed": split_seed, "clustering_threshold": clustering_threshold,
                "split_manifest_sha256": split_manifest_hashes[(dataset_name, split_seed)],
                "embedding_provenance": sha256_bytes(stable_json(embedding_metadata).encode("utf-8")),
                "predicted_assignment_sha256": sha256_bytes(np.asarray(oracle_predicted, dtype=np.int64).tobytes()),
                "replicates": BOOTSTRAP_REPLICATES, "bootstrap_seed": BOOTSTRAP_SEED,
            }
            oracle_bootstrap, oracle_bootstrap_hit = verified_bootstrap(
                oracle_bootstrap_path,
                expected_metadata=oracle_bootstrap_metadata,
                expected_metrics=("B3_F1", "ARI"),
                expected_replicates=BOOTSTRAP_REPLICATES,
                compute=lambda: bootstrap_clustering(
                    true_labels=query_rows.loc[is_novel, "identity"].astype(str),
                    predicted_labels=oracle_predicted,
                    replicates=BOOTSTRAP_REPLICATES,
                    seed=stable_seed(BOOTSTRAP_SEED, "oracle-novel", model_name, dataset_name, split_seed),
                ),
            )
            runtime_recorder.record(
                "bootstrap", time.perf_counter() - bootstrap_start,
                model=model_name, dataset=dataset_name, split_seed=split_seed,
                number_queries=int(is_novel.sum()), experiment="oracle_novel",
                cache_hit=oracle_bootstrap_hit,
            )
            oracle_prediction_path = PATHS["oracle_novel_clustering"] / model_name / dataset_name / f"seed_{split_seed}_predictions.npz"
            atomic_save_npz(
                oracle_prediction_path,
                image_ids=query_rows.loc[is_novel, "image_id"].astype(str).to_numpy(dtype="U"),
                true_labels=query_rows.loc[is_novel, "identity"].astype(str).to_numpy(dtype="U"),
                predicted_labels=np.asarray(oracle_predicted, dtype=np.int64),
            )
            oracle_prediction_paths[(model_name, dataset_name, split_seed)] = oracle_prediction_path
            oracle_rows.append({
                "model": model_name, "dataset": dataset_name, "split_seed": split_seed,
                "clustering_threshold": clustering_threshold,
                "threshold_source": "ApexID Split V2 validation",
                "B3_Precision": oracle_metrics["b3_precision"],
                "B3_Recall": oracle_metrics["b3_recall"],
                "B3_F1": oracle_metrics["b3_f1"],
                "ARI": oracle_metrics["ari"], "AMI": oracle_metrics["ami"],
                "NMI": oracle_metrics["nmi"], "V-measure": oracle_metrics["v_measure"],
                **{key: value for key, value in oracle_metrics.items() if key not in {"b3_precision", "b3_recall", "b3_f1", "ari", "ami", "nmi", "v_measure"}},
                "bootstrap_path": str(oracle_bootstrap_path),
            })

            # End-to-end: only frozen-score rejection decides which queries are clustered.
            cluster_start = time.perf_counter()
            discovery_metrics, rejected_mask, discovery_predicted = end_to_end_discovery(
                query_embeddings=query_embeddings,
                true_labels=query_rows["identity"].astype(str),
                is_novel=is_novel,
                knownness_scores=scores["knownness_scores"],
                rejection_threshold=rejection_threshold,
                clustering_threshold=clustering_threshold,
            )
            runtime_recorder.record(
                "clustering", time.perf_counter() - cluster_start,
                model=model_name, dataset=dataset_name, split_seed=split_seed,
                number_queries=int(discovery_metrics["number_rejected_samples"]), experiment="end_to_end",
            )
            discovery_bootstrap_path = PATHS["statistics"] / "end_to_end" / model_name / dataset_name / f"seed_{split_seed}.npz"
            bootstrap_start = time.perf_counter()
            discovery_bootstrap_metadata = {
                "unit": "stratified query identity block over detection and clustering", "model": model_name,
                "dataset": dataset_name, "split_seed": split_seed,
                "score_provenance": sha256_bytes(stable_json(score_meta).encode("utf-8")),
                "rejection_threshold": rejection_threshold,
                "clustering_threshold": clustering_threshold,
                "replicates": BOOTSTRAP_REPLICATES, "bootstrap_seed": BOOTSTRAP_SEED,
            }
            def compute_discovery_bootstrap():
                return bootstrap_end_to_end_discovery(
                    true_labels=query_rows["identity"].astype(str),
                    is_novel=is_novel,
                    rejected_mask=rejected_mask,
                    rejected_cluster_labels=discovery_predicted,
                    replicates=BOOTSTRAP_REPLICATES,
                    seed=stable_seed(BOOTSTRAP_SEED, "end-to-end", model_name, dataset_name, split_seed),
                )
            discovery_bootstrap, discovery_bootstrap_hit = verified_bootstrap(
                discovery_bootstrap_path,
                expected_metadata=discovery_bootstrap_metadata,
                expected_metrics=(
                    "End-to-End Discovery B3_F1", "End-to-End Discovery ARI",
                    "rejected_pool_b3_f1", "rejected_pool_ari",
                ),
                expected_replicates=BOOTSTRAP_REPLICATES,
                compute=compute_discovery_bootstrap,
            )
            runtime_recorder.record(
                "bootstrap", time.perf_counter() - bootstrap_start,
                model=model_name, dataset=dataset_name, split_seed=split_seed,
                number_queries=int(rejected_mask.sum()), experiment="end_to_end",
                cache_hit=discovery_bootstrap_hit,
            )
            discovery_rows.append({
                "model": model_name, "dataset": dataset_name, "split_seed": split_seed,
                "threshold_source": "ApexID Split V2 validation", **discovery_metrics,
                "bootstrap_path": str(discovery_bootstrap_path),
            })

            # Test-label-selected upper bounds are isolated here and never copied into headline rows.
            official_acceptance_before_oracle = (
                scores["knownness_scores"] >= rejection_threshold
            ).copy()
            if RUN_ORACLE_DIAGNOSTICS:
                oracle_diagnostic_start = time.perf_counter()
                oracle_rejection_threshold, _oracle_rejection_curve = select_rejection_threshold(
                    knownness_scores=scores["knownness_scores"],
                    predicted_identities=scores["predicted_identities"],
                    true_identities=scores["true_identities"],
                    is_novel=scores["is_novel"],
                )
                oracle_rejection_metrics = open_set_metrics_from_scores(
                    knownness_scores=scores["knownness_scores"],
                    predicted_identities=scores["predicted_identities"],
                    true_identities=scores["true_identities"],
                    is_novel=scores["is_novel"],
                    rejection_threshold=oracle_rejection_threshold,
                )
                oracle_cluster_threshold, oracle_cluster_curve, _ = calibrate_clustering_threshold(
                    query_embeddings[is_novel], query_rows.loc[is_novel, "identity"].astype(str),
                    candidates=CLUSTERING_THRESHOLD_CANDIDATES,
                    hierarchy=oracle_hierarchy,
                )
                oracle_cluster_selected = oracle_cluster_curve.loc[oracle_cluster_curve["selected"]].iloc[0]
                runtime_recorder.record(
                    "oracle_diagnostics", time.perf_counter() - oracle_diagnostic_start,
                    model=model_name, dataset=dataset_name, split_seed=split_seed,
                    number_queries=len(query_rows),
                )
                # The oracle may sweep only the already-frozen score; official arrays remain immutable.
                official_acceptance_after_oracle = scores["knownness_scores"] >= rejection_threshold
                if not np.array_equal(official_acceptance_before_oracle, official_acceptance_after_oracle):
                    raise RuntimeError("Oracle diagnostics modified official predictions")
                oracle_diagnostic_rows.append({
                    "model": model_name, "dataset": dataset_name, "split_seed": split_seed,
                    "diagnostic_only": True,
                    "frozen_rejection_method": rejection_method,
                    "frozen_rejection_score_name": rejection_score_name,
                    "logistic_refit_on_test": False,
                    "oracle_test_rejection_threshold": oracle_rejection_threshold,
                    "oracle_test_normacc": oracle_rejection_metrics["NormAcc"],
                    "oracle_test_cluster_threshold": oracle_cluster_threshold,
                    "oracle_test_b3_f1": oracle_cluster_selected["b3_f1"],
                    "oracle_test_ari": oracle_cluster_selected["ari"],
                })
open_set_results = pd.DataFrame(open_rows, columns=[
    "model", "dataset", "split_seed", "threshold_source", "rejection_method", "rejection_score_name"
])
oracle_novel_clustering_results = pd.DataFrame(oracle_rows, columns=[
    "model", "dataset", "split_seed", "threshold_source"
])
end_to_end_discovery_results = pd.DataFrame(discovery_rows, columns=[
    "model", "dataset", "split_seed", "threshold_source"
])
oracle_diagnostics = pd.DataFrame(oracle_diagnostic_rows)

# Leakage guard: official tables must contain only Stage 01 validation-frozen
# threshold sources. Oracle-fit values live exclusively in oracle_diagnostics.
for _name, _frame in (
    ("open_set_results", open_set_results),
    ("oracle_novel_clustering_results", oracle_novel_clustering_results),
    ("end_to_end_discovery_results", end_to_end_discovery_results),
):
    if RUN_OPEN_SET and set(_frame["model"].astype(str)) != EXPECTED_MODELS:
        raise RuntimeError(f"{_name} does not contain exactly the three sealed BHCA models")
    if not _frame["threshold_source"].eq("ApexID Split V2 validation").all():
        raise RuntimeError(f"{_name} contains a non-validation official threshold source")
if any("oracle_test" in column.lower() for column in open_set_results.columns):
    raise RuntimeError("Oracle values leaked into the official open-set table")
for model_name in MODEL_NAMES:
    selected = open_set_results.loc[open_set_results["model"].eq(model_name)]
    if not selected["rejection_method"].eq(rejection_method_by_model[model_name]).all():
        raise RuntimeError(f"{model_name}: official results use the wrong rejection method")
    if rejection_method_by_model[model_name] == "margin_gallery_logistic" and not selected["rejection_score_name"].eq("p_known").all():
        raise RuntimeError(f"{model_name}: logistic official results do not use p_known")
if not oracle_diagnostics.empty and not oracle_diagnostics["diagnostic_only"].astype(bool).all():
    raise RuntimeError("Oracle diagnostic rows are not explicitly marked diagnostic_only")

if oracle_diagnostics.empty:
    oracle_diagnostics = pd.DataFrame(columns=[
        "model", "dataset", "split_seed", "diagnostic_only",
        "oracle_test_rejection_threshold", "oracle_test_normacc",
        "oracle_test_cluster_threshold", "oracle_test_b3_f1",
        "oracle_test_ari",
    ])
oscr_curves = (
    pd.concat(oscr_curve_frames, ignore_index=True)
    if oscr_curve_frames else pd.DataFrame(columns=["model", "dataset", "split_seed"])
)
if not RUN_OPEN_SET:
    print("OPEN-SET / ORACLE / DISCOVERY: SKIPPED (RUN_OPEN_SET=False)")

for filename, frame in (
    ("open_set_results.csv", open_set_results),
    ("oracle_novel_clustering_results.csv", oracle_novel_clustering_results),
    ("end_to_end_discovery_results.csv", end_to_end_discovery_results),
):
    task_dir = filename.removesuffix("_results.csv")
    destination = PATHS[task_dir] if task_dir in PATHS else PATHS["tables"]
    atomic_write_csv(destination / filename, frame)
    atomic_write_csv(PATHS["tables"] / filename, frame)
atomic_write_csv(PATHS["open_set"] / "oscr_curves.csv", oscr_curves)
atomic_write_csv(PATHS["oracle_diagnostics"] / "oracle_diagnostics.csv", oracle_diagnostics)
atomic_write_csv(PATHS["tables"] / "oracle_diagnostics.csv", oracle_diagnostics)
display(open_set_results.head())


OPEN-SET / ORACLE / DISCOVERY: SKIPPED (RUN_OPEN_SET=False)


,model,dataset,split_seed,threshold_source,rejection_method,rejection_score_name


## 9. Oracle-Novel Clustering

In [9]:
display(oracle_novel_clustering_results.head())
print("Ground-truth novelty selected this conditional evaluation subset only; tau_cluster remained frozen.")


,model,dataset,split_seed,threshold_source


Ground-truth novelty selected this conditional evaluation subset only; tau_cluster remained frozen.


## 10. End-to-End Novel Identity Discovery

In [10]:
display(end_to_end_discovery_results.head())
print("Rejected samples were selected without the true novel mask. Coverage and known contamination are explicit.")


,model,dataset,split_seed,threshold_source


Rejected samples were selected without the true novel mask. Coverage and known contamination are explicit.


## 11. Statistical confidence intervals

The final paper run uses 1,000 identity-block/hierarchical bootstrap replicates. Set `APEXID_BOOTSTRAP_REPLICATES` lower only for a debugging run.

In [11]:
statistics_start = time.perf_counter()
closed_set_summary = hierarchical_bootstrap_summary(
    closed_set_results,
    metrics=("Rank-1", "mAP"), bootstrap_path_column="bootstrap_path",
    group_columns=("model", "dataset"), replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED,
)
if RUN_OPEN_SET:
    open_set_summary = hierarchical_bootstrap_summary(
        open_set_results,
        metrics=OPEN_BOOTSTRAP_METRICS, bootstrap_path_column="bootstrap_path",
        group_columns=("model", "dataset"), replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED,
    )
    oracle_novel_clustering_summary = hierarchical_bootstrap_summary(
        oracle_novel_clustering_results,
        metrics=("B3_F1", "ARI"), bootstrap_path_column="bootstrap_path",
        group_columns=("model", "dataset"), replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED,
    )
    end_to_end_discovery_summary = hierarchical_bootstrap_summary(
        end_to_end_discovery_results,
        metrics=(
            "End-to-End Discovery B3_F1", "End-to-End Discovery ARI",
            "rejected_pool_b3_f1", "rejected_pool_ari",
        ), bootstrap_path_column="bootstrap_path",
        group_columns=("model", "dataset"), replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED,
    )
else:
    empty_summary_columns = [
        "model", "dataset", "metric", "mean", "ci_low", "ci_high",
        "n_splits", "bootstrap_replicates", "bootstrap_method",
    ]
    open_set_summary = pd.DataFrame(columns=empty_summary_columns)
    oracle_novel_clustering_summary = pd.DataFrame(columns=empty_summary_columns)
    end_to_end_discovery_summary = pd.DataFrame(columns=empty_summary_columns)

# Add deterministic non-bootstrap summary metrics while retaining requested identity-block CIs.
def append_point_only_metrics(summary, details, metrics):
    rows = []
    for (model, dataset), group in details.groupby(["model", "dataset"], sort=True):
        for metric in metrics:
            rows.append({
                "model": model, "dataset": dataset, "metric": metric,
                "mean": float(group[metric].mean()), "ci_low": np.nan, "ci_high": np.nan,
                "n_splits": len(group), "bootstrap_replicates": 0,
                "bootstrap_method": "point estimate; CI not requested",
            })
    return pd.concat([summary, pd.DataFrame(rows)], ignore_index=True)

closed_set_summary = append_point_only_metrics(
    closed_set_summary, closed_set_results,
    ("Rank-5", "Rank-10", "CMC@20", "CMC@50", "identity-balanced Rank-1"),
)
open_set_summary = append_point_only_metrics(
    open_set_summary, open_set_results,
    ("TNR@95TPR", "FPR@95TPR", "FAR_at_frozen_threshold", "known_TPR_at_frozen_threshold",
     "novel_rejection_rate", "DIR@5%FAR", "known_identification_Rank-1_among_accepted"),
)
oracle_novel_clustering_summary = append_point_only_metrics(
    oracle_novel_clustering_summary, oracle_novel_clustering_results,
    ("B3_Precision", "B3_Recall", "AMI", "NMI", "V-measure", "predicted_cluster_count", "true_identity_count",
     "mean_fragments_per_identity", "fraction_true_identities_fragmented", "fraction_predicted_clusters_merged"),
)
end_to_end_discovery_summary = append_point_only_metrics(
    end_to_end_discovery_summary, end_to_end_discovery_results,
    ("novel_sample_discovery_recall", "novel_identity_balanced_discovery_recall", "known_contamination_rate",
     "rejected_novel_b3_f1", "rejected_novel_ari"),
)

for name, frame in (
    ("closed_set_summary.csv", closed_set_summary),
    ("open_set_summary.csv", open_set_summary),
    ("oracle_novel_clustering_summary.csv", oracle_novel_clustering_summary),
    ("end_to_end_discovery_summary.csv", end_to_end_discovery_summary),
):
    atomic_write_csv(PATHS["tables"] / name, frame)
display(open_set_summary.head(20))
runtime_recorder.record("statistics_aggregation", time.perf_counter() - statistics_start)


,model,dataset,metric,mean,ci_low,ci_high,n_splits,bootstrap_replicates,bootstrap_method


## 12. Paired model comparisons

`ApexID_BHCA_A_v2` is the reference. Differences use the existing paired split-seed plus identity-block bootstrap on identical frozen query membership. Every reported delta is `reference − comparison`; therefore a positive delta means A performs better for the included higher-is-better metrics. Statistical evidence is reported by the paired 95% confidence interval rather than inferred from means alone.


In [12]:
if RUN_PAIRED_COMPARISONS:
    paired_start = time.perf_counter()
    paired_frames = []
    reference_model = "ApexID_BHCA_A_v2"
    comparison_models = ("ApexID_BHCA_SCA_v2", "ApexID_BHCA_SCSF2_v2")
    if {reference_model, *comparison_models} != EXPECTED_MODELS:
        raise RuntimeError("Paired comparison configuration does not match the sealed model set")

    for comparison_model in comparison_models:
        for dataset_name in external_frames:
            # Closed-set paired differences.
            bootstrap_paths = []
            for split_seed in SPLIT_SEEDS:
                reference = pd.read_csv(closed_detail_paths[(reference_model, dataset_name, split_seed)])
                comparison = pd.read_csv(closed_detail_paths[(comparison_model, dataset_name, split_seed)])
                path = PATHS["statistics"] / "paired_closed" / comparison_model / dataset_name / f"seed_{split_seed}.npz"
                paired_metadata = {
                    "reference": reference_model, "comparison": comparison_model,
                    "dataset": dataset_name, "split_seed": split_seed,
                    "reference_details_sha256": sha256_file(closed_detail_paths[(reference_model, dataset_name, split_seed)]),
                    "comparison_details_sha256": sha256_file(closed_detail_paths[(comparison_model, dataset_name, split_seed)]),
                    "replicates": BOOTSTRAP_REPLICATES, "bootstrap_seed": BOOTSTRAP_SEED,
                }
                verified_bootstrap(
                    path, expected_metadata=paired_metadata, expected_metrics=("Rank-1", "mAP"),
                    expected_replicates=BOOTSTRAP_REPLICATES,
                    compute=lambda: paired_closed_bootstrap_difference(
                        reference, comparison, replicates=BOOTSTRAP_REPLICATES,
                        seed=stable_seed(BOOTSTRAP_SEED, "paired-closed", comparison_model, dataset_name, split_seed),
                    ),
                )
                bootstrap_paths.append(path)
            ref_points = closed_set_results.loc[(closed_set_results.model == reference_model) & (closed_set_results.dataset == dataset_name)]
            cmp_points = closed_set_results.loc[(closed_set_results.model == comparison_model) & (closed_set_results.dataset == dataset_name)]
            paired_frames.append(paired_difference_summary(
                bootstrap_paths=bootstrap_paths,
                point_differences={metric: float(ref_points[metric].mean() - cmp_points[metric].mean()) for metric in ("Rank-1", "mAP")},
                metrics=("Rank-1", "mAP"), replicates=BOOTSTRAP_REPLICATES,
                seed=stable_seed(BOOTSTRAP_SEED, "paired-closed-summary", comparison_model, dataset_name),
                metadata={"task": "closed_set", "dataset": dataset_name, "reference_model": reference_model, "comparison_model": comparison_model},
            ))

            if not RUN_OPEN_SET:
                continue

            # Open-set paired differences using identical query membership and block draws.
            bootstrap_paths = []
            for split_seed in SPLIT_SEEDS:
                reference = load_open_score_cache(open_score_paths[(reference_model, dataset_name, split_seed)])
                comparison = load_open_score_cache(open_score_paths[(comparison_model, dataset_name, split_seed)])
                reference["rejection_threshold"] = frozen_thresholds[reference_model]["rejection_threshold"]
                comparison["rejection_threshold"] = frozen_thresholds[comparison_model]["rejection_threshold"]
                path = PATHS["statistics"] / "paired_open" / comparison_model / dataset_name / f"seed_{split_seed}.npz"
                paired_metadata = {
                    "reference": reference_model, "comparison": comparison_model,
                    "dataset": dataset_name, "split_seed": split_seed,
                    "reference_threshold": reference["rejection_threshold"],
                    "comparison_threshold": comparison["rejection_threshold"],
                    "reference_score_sha256": sha256_file(open_score_paths[(reference_model, dataset_name, split_seed)]),
                    "comparison_score_sha256": sha256_file(open_score_paths[(comparison_model, dataset_name, split_seed)]),
                    "replicates": BOOTSTRAP_REPLICATES, "bootstrap_seed": BOOTSTRAP_SEED,
                }
                verified_bootstrap(
                    path, expected_metadata=paired_metadata, expected_metrics=OPEN_BOOTSTRAP_METRICS,
                    expected_replicates=BOOTSTRAP_REPLICATES,
                    compute=lambda: paired_open_bootstrap_difference(
                        reference, comparison, replicates=BOOTSTRAP_REPLICATES,
                        seed=stable_seed(BOOTSTRAP_SEED, "paired-open", comparison_model, dataset_name, split_seed),
                    ),
                )
                bootstrap_paths.append(path)
            ref_points = open_set_results.loc[(open_set_results.model == reference_model) & (open_set_results.dataset == dataset_name)]
            cmp_points = open_set_results.loc[(open_set_results.model == comparison_model) & (open_set_results.dataset == dataset_name)]
            paired_frames.append(paired_difference_summary(
                bootstrap_paths=bootstrap_paths,
                point_differences={metric: float(ref_points[metric].mean() - cmp_points[metric].mean()) for metric in OPEN_BOOTSTRAP_METRICS},
                metrics=OPEN_BOOTSTRAP_METRICS, replicates=BOOTSTRAP_REPLICATES,
                seed=stable_seed(BOOTSTRAP_SEED, "paired-open-summary", comparison_model, dataset_name),
                metadata={"task": "open_set", "dataset": dataset_name, "reference_model": reference_model, "comparison_model": comparison_model},
            ))

            # Oracle-Novel cluster comparisons; the inference subset is identical.
            bootstrap_paths = []
            for split_seed in SPLIT_SEEDS:
                def load_cluster(path):
                    with np.load(path, allow_pickle=False) as values:
                        return {key: values[key] for key in values.files}
                reference = load_cluster(oracle_prediction_paths[(reference_model, dataset_name, split_seed)])
                comparison = load_cluster(oracle_prediction_paths[(comparison_model, dataset_name, split_seed)])
                path = PATHS["statistics"] / "paired_oracle_novel" / comparison_model / dataset_name / f"seed_{split_seed}.npz"
                paired_metadata = {
                    "reference": reference_model, "comparison": comparison_model,
                    "dataset": dataset_name, "split_seed": split_seed,
                    "reference_threshold": frozen_thresholds[reference_model]["clustering_threshold"],
                    "comparison_threshold": frozen_thresholds[comparison_model]["clustering_threshold"],
                    "reference_predictions_sha256": sha256_bytes(np.asarray(reference["predicted_labels"]).tobytes()),
                    "comparison_predictions_sha256": sha256_bytes(np.asarray(comparison["predicted_labels"]).tobytes()),
                    "replicates": BOOTSTRAP_REPLICATES, "bootstrap_seed": BOOTSTRAP_SEED,
                }
                verified_bootstrap(
                    path, expected_metadata=paired_metadata, expected_metrics=("B3_F1", "ARI"),
                    expected_replicates=BOOTSTRAP_REPLICATES,
                    compute=lambda: paired_clustering_bootstrap_difference(
                        reference, comparison, replicates=BOOTSTRAP_REPLICATES,
                        seed=stable_seed(BOOTSTRAP_SEED, "paired-cluster", comparison_model, dataset_name, split_seed),
                    ),
                )
                bootstrap_paths.append(path)
            ref_points = oracle_novel_clustering_results.loc[(oracle_novel_clustering_results.model == reference_model) & (oracle_novel_clustering_results.dataset == dataset_name)]
            cmp_points = oracle_novel_clustering_results.loc[(oracle_novel_clustering_results.model == comparison_model) & (oracle_novel_clustering_results.dataset == dataset_name)]
            paired_frames.append(paired_difference_summary(
                bootstrap_paths=bootstrap_paths,
                point_differences={metric: float(ref_points[metric].mean() - cmp_points[metric].mean()) for metric in ("B3_F1", "ARI")},
                metrics=("B3_F1", "ARI"), replicates=BOOTSTRAP_REPLICATES,
                seed=stable_seed(BOOTSTRAP_SEED, "paired-cluster-summary", comparison_model, dataset_name),
                metadata={"task": "oracle_novel_clustering", "dataset": dataset_name, "reference_model": reference_model, "comparison_model": comparison_model},
            ))

    paired_model_comparisons = pd.concat(paired_frames, ignore_index=True)
    paired_model_comparisons["difference_direction"] = "reference_minus_comparison"
    paired_model_comparisons["positive_delta_interpretation"] = "reference_model_better"
    if set(paired_model_comparisons["reference_model"].astype(str)) != {reference_model}:
        raise RuntimeError("Paired comparison output contains an unexpected reference model")
    if set(paired_model_comparisons["comparison_model"].astype(str)) != set(comparison_models):
        raise RuntimeError("Paired comparison output is missing a required HA comparison")
    atomic_write_csv(PATHS["tables"] / "paired_model_comparisons.csv", paired_model_comparisons)
    display(paired_model_comparisons.head(20))
    runtime_recorder.record("paired_bootstrap", time.perf_counter() - paired_start)

else:
    paired_model_comparisons = pd.DataFrame(
        columns=[
            "task", "dataset", "reference_model",
            "comparison_model", "metric", "mean_difference_reference_minus_comparison",
            "ci_low", "ci_high", "difference_direction", "positive_delta_interpretation",
        ]
    )
    atomic_write_csv(
        PATHS["tables"] / "paired_model_comparisons.csv",
        paired_model_comparisons,
    )

,task,dataset,reference_model,comparison_model,metric,mean_difference_reference_minus_comparison,ci_low,ci_high,paired_95ci_excludes_zero,bootstrap_replicates,bootstrap_method,difference_direction,positive_delta_interpretation
0,closed_set,bristol_gorillas,ApexID_BHCA_A_v2,ApexID_BHCA_SCA_v2,Rank-1,-0.001847,-0.007283,0.002623,False,1000,paired split-seed + identity-block bootstrap,reference_minus_comparison,reference_model_better
1,closed_set,bristol_gorillas,ApexID_BHCA_A_v2,ApexID_BHCA_SCA_v2,mAP,0.000489,-0.002117,0.002697,False,1000,paired split-seed + identity-block bootstrap,reference_minus_comparison,reference_model_better
2,closed_set,elephants,ApexID_BHCA_A_v2,ApexID_BHCA_SCA_v2,Rank-1,0.000479,-0.018305,0.022576,False,1000,paired split-seed + identity-block bootstrap,reference_minus_comparison,reference_model_better
3,closed_set,elephants,ApexID_BHCA_A_v2,ApexID_BHCA_SCA_v2,mAP,0.000946,-0.001683,0.003880,False,1000,paired split-seed + identity-block bootstrap,reference_minus_comparison,reference_model_better
4,closed_set,gcn_id,ApexID_BHCA_A_v2,ApexID_BHCA_SCA_v2,Rank-1,0.001552,-0.017013,0.019504,False,1000,paired split-seed + identity-block bootstrap,reference_minus_comparison,reference_model_better
5,closed_set,gcn_id,ApexID_BHCA_A_v2,ApexID_BHCA_SCA_v2,mAP,0.004472,-0.001248,0.010825,False,1000,paired split-seed + identity-block bootstrap,reference_minus_comparison,reference_model_better
6,closed_set,jaguar_antonio,ApexID_BHCA_A_v2,ApexID_BHCA_SCA_v2,Rank-1,0.005628,-0.030904,0.037366,False,1000,paired split-seed + identity-block bootstrap,reference_minus_comparison,reference_model_better
7,closed_set,jaguar_antonio,ApexID_BHCA_A_v2,ApexID_BHCA_SCA_v2,mAP,0.004517,0.000322,0.009330,True,1000,paired split-seed + identity-block bootstrap,reference_minus_comparison,reference_model_better
8,closed_set,bristol_gorillas,ApexID_BHCA_A_v2,ApexID_BHCA_SCSF2_v2,Rank-1,-0.008800,-0.016966,-0.002233,True,1000,paired split-seed + identity-block bootstrap,reference_minus_comparison,reference_model_better
9,closed_set,bristol_gorillas,ApexID_BHCA_A_v2,ApexID_BHCA_SCSF2_v2,mAP,0.000669,-0.002160,0.003051,False,1000,paired split-seed + identity-block bootstrap,reference_minus_comparison,reference_model_better


## 13. Cross-dataset aggregation

Dataset-Macro Average weights the evaluated external datasets equally (`EXTERNAL_DATASETS`; currently four — `spac_wild` excluded). The optional pooled dataset is intentionally excluded.

In [13]:
macro_start = time.perf_counter()
macro_closed = closed_set_results.loc[closed_set_results["dataset"].isin(EXTERNAL_DATASETS)]
macro_open = open_set_results.loc[open_set_results["dataset"].isin(EXTERNAL_DATASETS)]
macro_oracle = oracle_novel_clustering_results.loc[
    oracle_novel_clustering_results["dataset"].isin(EXTERNAL_DATASETS)
]
macro_discovery = end_to_end_discovery_results.loc[
    end_to_end_discovery_results["dataset"].isin(EXTERNAL_DATASETS)
]
macro_frames = [
    dataset_macro_hierarchical_bootstrap(
        macro_closed, task="closed_set", metrics=("Rank-1", "mAP"),
        bootstrap_path_column="bootstrap_path", replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED,
    )
]
if RUN_OPEN_SET:
    macro_frames.extend([
        dataset_macro_hierarchical_bootstrap(
            macro_open, task="open_set", metrics=OPEN_BOOTSTRAP_METRICS,
            bootstrap_path_column="bootstrap_path", replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED,
        ),
        dataset_macro_hierarchical_bootstrap(
            macro_oracle, task="oracle_novel_clustering", metrics=("B3_F1", "ARI"),
            bootstrap_path_column="bootstrap_path", replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED,
        ),
        dataset_macro_hierarchical_bootstrap(
            macro_discovery, task="end_to_end_discovery",
            metrics=("End-to-End Discovery B3_F1", "End-to-End Discovery ARI"),
            bootstrap_path_column="bootstrap_path", replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED,
        ),
    ])
dataset_macro_summary = pd.concat(macro_frames, ignore_index=True)
atomic_write_csv(PATHS["tables"] / "dataset_macro_summary.csv", dataset_macro_summary)

display(dataset_macro_summary)
runtime_recorder.record("dataset_macro_bootstrap", time.perf_counter() - macro_start)


,task,model,metric,aggregation,mean,ci_low,ci_high,number_datasets,bootstrap_replicates,bootstrap_method
0,closed_set,ApexID_BHCA_A_v2,Rank-1,Dataset-Macro Average,0.627425,0.290209,0.888281,4,1000,dataset + split-seed + identity-block hierarch...
1,closed_set,ApexID_BHCA_A_v2,mAP,Dataset-Macro Average,0.297504,0.113347,0.506762,4,1000,dataset + split-seed + identity-block hierarch...
2,closed_set,ApexID_BHCA_SCA_v2,Rank-1,Dataset-Macro Average,0.625972,0.315736,0.884473,4,1000,dataset + split-seed + identity-block hierarch...
3,closed_set,ApexID_BHCA_SCA_v2,mAP,Dataset-Macro Average,0.294897,0.112652,0.500209,4,1000,dataset + split-seed + identity-block hierarch...
4,closed_set,ApexID_BHCA_SCSF2_v2,Rank-1,Dataset-Macro Average,0.641393,0.344765,0.880398,4,1000,dataset + split-seed + identity-block hierarch...
5,closed_set,ApexID_BHCA_SCSF2_v2,mAP,Dataset-Macro Average,0.302402,0.116404,0.525373,4,1000,dataset + split-seed + identity-block hierarch...


## 14. Oracle diagnostics

In [14]:
display(oracle_diagnostics.head())
print("These post-hoc test-label upper bounds are excluded from every headline summary and ranking.")


,model,dataset,split_seed,diagnostic_only,oracle_test_rejection_threshold,oracle_test_normacc,oracle_test_cluster_threshold,oracle_test_b3_f1,oracle_test_ari


These post-hoc test-label upper bounds are excluded from every headline summary and ranking.


## 15. Publication figures

Plots are derived only from the executed result DataFrames and frozen Stage 01 validation curves.

In [15]:
if RUN_PLOTS and RUN_OPEN_SET:
    plot_start = time.perf_counter()
    plot_paths = save_required_plots(
        plot_dir=PATHS["plots"],
        closed_summary=closed_set_summary,
        open_summary=open_set_summary,
        oracle_summary=oracle_novel_clustering_summary,
        discovery_summary=end_to_end_discovery_summary,
        macro_summary=dataset_macro_summary,
        rejection_curves=rejection_curves,
        clustering_curves=clustering_curves,
        oscr_curves=oscr_curves,
    )
    runtime_recorder.record("plot_generation", time.perf_counter() - plot_start)
    print(f"Saved {len(plot_paths)} publication figure files.")

else:
    plot_paths = []
    reason = "RUN_OPEN_SET=False" if not RUN_OPEN_SET else "APEXID_RUN_PLOTS=0"
    print(f"Plot generation disabled by {reason}.")

Plot generation disabled by RUN_OPEN_SET=False.


## 16. Final integrity, provenance, manifest, and report

The manifest distinguishes probability thresholds from raw-cosine thresholds and records every per-model calibrator SHA-256. It also records that Stage 02 fitted no threshold, standardizer, or logistic coefficients; extracted no embedding; trained no model; regenerated no test split; and kept oracle analysis diagnostic-only.

The closed-set metric implementation is SHA-256 checked against the pre-logistic backup. Because the implementation is byte-identical and deterministic on the same frozen arrays, closed-set outputs remain numerically identical; stale historical result caches are not treated as ground truth.


In [16]:
# ============================================================
# 16. Final integrity, provenance, report, and dual manifest names
# ============================================================

runtime_recorder.record("benchmark_wall_clock", time.perf_counter() - BENCHMARK_WALL_START)
runtime_recorder.save(OUTPUT_ROOT / "runtime_report.csv")
runtime_report = runtime_recorder.frame()

integrity = final_integrity_check(
    models_completed=models_completed,
    expected_datasets=EXTERNAL_DATASETS,
    datasets_completed=tuple(external_frames),
    split_v2=split_v2_audit,
    split_checks=split_integrity_checks.to_dict(orient="records"),
    threshold_table=threshold_calibration,
    result_tables={
        "closed_set_results": closed_set_results,
        **({
            "open_set_results": open_set_results,
            "oracle_novel_clustering_results": oracle_novel_clustering_results,
            "end_to_end_discovery_results": end_to_end_discovery_results,
        } if RUN_OPEN_SET else {}),
    },
    summary_tables={
        "closed_set_summary": closed_set_summary,
        "dataset_macro_summary": dataset_macro_summary,
        **({
            "open_set_summary": open_set_summary,
            "oracle_novel_clustering_summary": oracle_novel_clustering_summary,
            "end_to_end_discovery_summary": end_to_end_discovery_summary,
        } if RUN_OPEN_SET else {}),
    },
    model_summary=model_summary,
    runtime_report=runtime_report,
    split_seeds=SPLIT_SEEDS,
)
integrity["status"] = "FROZEN INPUTS / EXECUTION COMPLETE" if integrity["passed"] else "FAILED"
integrity["note"] = (
    "Stage 02 consumed frozen Stage 00 splits, Stage 01 embeddings, and Stage 01 "
    "validation-only thresholds. Oracle test-label fits remained diagnostic-only."
)
if not integrity["passed"]:
    raise RuntimeError("Stage 02 final integrity check failed:\n- " + "\n- ".join(integrity["failures"]))

environment = {
    "stage02": environment_provenance(),
    "stage00": stage00_manifest.get("environment"),
    "stage01": stage01_manifest.get("environment"),
}
execution_guarantees = {
    "threshold_fitting_in_stage02": False,
    "embedding_extraction_in_stage02": False,
    "model_training_in_stage02": False,
    "test_split_regeneration": False,
    "official_calibration_source": OFFICIAL_CALIBRATION_SOURCE,
    "oracle_analysis_diagnostic_only": True,
    "logistic_calibrator_fitting_in_stage02": False,
    "feature_standardizer_fitting_in_stage02": False,
}
optimized_validation = {
    "stage02_mode": "local_frozen_inputs",
    "startup_preflight": "passed",
    "final_integrity_check": "passed",
    **execution_guarantees,
}

benchmark_manifest = {
    "benchmark_version": BENCHMARK_VERSION,
    "stage02_version": STAGE02_VERSION,
    "stage02_protocol_version": STAGE02_PROTOCOL_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "execution_status": "completed",
    "execution_mode": "local_frozen_inputs",
    "execution_guarantees": execution_guarantees,
    "repository": repo_provenance,
    "environment": environment,
    "optimized_algorithm_validation": optimized_validation,
    "models_evaluated": list(MODEL_NAMES),
    "models": model_metadata_by_name,
    "datasets": list(EXTERNAL_DATASETS),
    "dataset_manifest_hashes": dataset_manifest_hashes,
    "split_manifest_hashes": {
        f"{dataset}/seed_{seed}": value
        for (dataset, seed), value in split_manifest_hashes.items()
    },
    "split_v2": split_v2_audit,
    "fixed_split_seeds": list(SPLIT_SEEDS),
    "bootstrap": {
        "replicates": BOOTSTRAP_REPLICATES,
        "seed": BOOTSTRAP_SEED,
        "unit": "identity block / hierarchical bootstrap",
        "n_jobs": N_JOBS,
    },
    "paired_comparisons": {
        "reference_model": MODEL_NAMES[0],
        "comparison_models": list(MODEL_NAMES[1:]),
        "difference_direction": "reference_minus_comparison",
        "positive_delta_interpretation": "reference_model_better",
    },
    "novel_identity_ratio": NOVEL_IDENTITY_RATIO,
    "gallery_observation_group_ratio": GALLERY_OBSERVATION_GROUP_RATIO,
    "closed_set_gallery_scoring": "image-gallery cosine retrieval",
    "open_set_gallery_scoring": "disabled" if not RUN_OPEN_SET else "L2-normalized identity centroid cosine top-2",
    "rejection_method": None if not RUN_OPEN_SET else (
        "margin_gallery_logistic"
        if set(rejection_method_by_model.values()) == {"margin_gallery_logistic"}
        else dict(rejection_method_by_model)
    ),
    "rejection_score": None if not RUN_OPEN_SET else (
        "p_known"
        if set(rejection_method_by_model.values()) == {"margin_gallery_logistic"}
        else {name: frozen_thresholds[name]["rejection_score_name"] for name in MODEL_NAMES}
    ),
    "calibration_source": None if not RUN_OPEN_SET else "Stage 01 validation only",
    "threshold_semantics": "disabled" if not RUN_OPEN_SET else (
        "probability"
        if set(rejection_method_by_model.values()) == {"margin_gallery_logistic"}
        else "declared per model"
    ),
    "calibrator_artifact_sha256": calibrator_sha256_by_model if RUN_OPEN_SET else {},
    "logistic_feature_names": list(LOGISTIC_FEATURE_NAMES) if RUN_OPEN_SET else [],
    "closed_set_regression_check": closed_set_regression,
    "frozen_thresholds": frozen_thresholds if RUN_OPEN_SET else {},
    "threshold_sources": "disabled for closed-set-only Stage 02" if not RUN_OPEN_SET else "ApexID Split V2 validation only; loaded from Stage 01",
    "stage00": {
        "root": str(STAGE00_ROOT),
        "manifest": str(STAGE00_MANIFEST_PATH),
        "manifest_sha256": sha256_file(STAGE00_MANIFEST_PATH),
    },
    "stage01": {
        "root": str(STAGE01_ROOT),
        "manifest": str(STAGE01_MANIFEST_PATH),
        "manifest_sha256": sha256_file(STAGE01_MANIFEST_PATH),
        "calibration_protocol_version": stage01_manifest.get("calibration_protocol_version", "unknown") if RUN_OPEN_SET else None,
        "frozen_thresholds_file": str(FROZEN_THRESHOLDS_PATH) if RUN_OPEN_SET else None,
        "frozen_thresholds_sha256": sha256_file(FROZEN_THRESHOLDS_PATH) if RUN_OPEN_SET else None,
        "rejection_calibrator_sha256": calibrator_sha256_by_model if RUN_OPEN_SET else {},
    },
    "embedding_root": str(EMBEDDINGS_ROOT),
    "embedding_root_source": EMBEDDING_ROOT_SOURCE,
    "stage02_notebook": {
        "path": str(STAGE02_NOTEBOOK_PATH),
        "sha256": sha256_file(STAGE02_NOTEBOOK_PATH) if STAGE02_NOTEBOOK_PATH.is_file() else None,
        "closed_set_regression": closed_set_regression,
    },
    # Compatibility keys retained for existing report consumers.
    "stage00_root": str(STAGE00_ROOT),
    "stage01_root": str(STAGE01_ROOT),
}
write_json(OUTPUT_ROOT / "benchmark_manifest.json", benchmark_manifest)
write_json(OUTPUT_ROOT / "stage02_manifest.json", benchmark_manifest)

report_start = time.perf_counter()
report_markdown, report_json = generate_final_report(
    output_root=OUTPUT_ROOT,
    integrity=integrity,
    benchmark_manifest=benchmark_manifest,
    model_summary=model_summary,
    dataset_summary=dataset_summary,
    threshold_calibration=threshold_calibration,
    closed_summary=closed_set_summary,
    open_summary=open_set_summary,
    oracle_summary=oracle_novel_clustering_summary,
    discovery_summary=end_to_end_discovery_summary,
    dataset_macro=dataset_macro_summary,
    paired_comparisons=paired_model_comparisons,
    runtime_report=runtime_report,
    limitations=remaining_limitations,
)
runtime_recorder.record("report_generation", time.perf_counter() - report_start)
runtime_recorder.save(OUTPUT_ROOT / "runtime_report.csv")
runtime_report = runtime_recorder.frame()

# Regenerate once so the final report includes report-generation timing.
report_markdown, report_json = generate_final_report(
    output_root=OUTPUT_ROOT,
    integrity=integrity,
    benchmark_manifest=benchmark_manifest,
    model_summary=model_summary,
    dataset_summary=dataset_summary,
    threshold_calibration=threshold_calibration,
    closed_summary=closed_set_summary,
    open_summary=open_set_summary,
    oracle_summary=oracle_novel_clustering_summary,
    discovery_summary=end_to_end_discovery_summary,
    dataset_macro=dataset_macro_summary,
    paired_comparisons=paired_model_comparisons,
    runtime_report=runtime_report,
    limitations=remaining_limitations,
)

display(Markdown(report_markdown))


# Final Three-Model BHCA Benchmark Report

## A. Benchmark integrity

**BENCHMARK INTEGRITY CHECK: FROZEN INPUTS / EXECUTION COMPLETE**

## B. Model information

| model                | model_identifier                                                                                                                                      | checkpoint_path                                                                                                                                       | checkpoint_sha256                                                | config_path                     | config_sha256                                                    | config_provenance                     |   revision |   embedding_dimension | input_resolution   | parameters   | l2_normalized   | transform                                                                                                                                                                                                                                                                                                                    | mixed_precision_validation                                                                                            |   selected_batch_size | checkpoint_provenance_type   |
|:---------------------|:------------------------------------------------------------------------------------------------------------------------------------------------------|:------------------------------------------------------------------------------------------------------------------------------------------------------|:-----------------------------------------------------------------|:--------------------------------|:-----------------------------------------------------------------|:--------------------------------------|-----------:|----------------------:|:-------------------|:-------------|:----------------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------|----------------------:|:-----------------------------|
| ApexID_BHCA_A_v2     | /kaggle/input/models/mohamedbassat/apexid-convnext-small-bhca-a-v2-rgb/pytorch/default/1/apexid_convnext_small_bhca_a_v2_rgb/best_backbone.pt         | /kaggle/input/models/mohamedbassat/apexid-convnext-small-bhca-a-v2-rgb/pytorch/default/1/apexid_convnext_small_bhca_a_v2_rgb/best_backbone.pt         | a4981988409c2a6d85a8c9224eb94f7bbd5cb1d55d80144f0e7870581ae20d13 | rgb_bhca_eval_fallback_imagenet | 6a66af69a02c2f88fcef08108a6f798bc70e02f28e84a0077b716f0826a54fd7 | sidecar YAML next to best_backbone.pt |        137 |                  2048 | [440,440]          | unknown      | True            | {"config_sha256":"6a66af69a02c2f88fcef08108a6f798bc70e02f28e84a0077b716f0826a54fd7","config_source":"rgb_bhca_eval_fallback_imagenet","grayscale":false,"image_size":[440,440],"implementation":"apexid.data.build_eval_transforms","input_channels":3,"mean":[0.485,0.456,0.406],"std":[0.229,0.224,0.225],"wavelet":false} | {"extraction_device":"cuda_if_available","source_cache_amp_enabled":true,"stage01_local_calibration_uses_cuda":false} |                    64 | bhca_backbone_sidecar        |
| ApexID_BHCA_SCA_v2   | /kaggle/input/models/mohamedbassat/apexid-convnext-small-bhca-sca-v2-rgb/pytorch/default/1/apexid_convnext_small_bhca_sca_v2_rgb/best_backbone.pt     | /kaggle/input/models/mohamedbassat/apexid-convnext-small-bhca-sca-v2-rgb/pytorch/default/1/apexid_convnext_small_bhca_sca_v2_rgb/best_backbone.pt     | 674cc88a2f3dd430bb1db84163c84a072a3a79941e56b266f4bbc2ed1a857ede | rgb_bhca_eval_fallback_imagenet | 6a66af69a02c2f88fcef08108a6f798bc70e02f28e84a0077b716f0826a54fd7 | sidecar YAML next to best_backbone.pt |        137 |                  2048 | [440,440]          | unknown      | True            | {"config_sha256":"6a66af69a02c2f88fcef08108a6f798bc70e02f28e84a0077b716f0826a54fd7","config_source":"rgb_bhca_eval_fallback_imagenet","grayscale":false,"image_size":[440,440],"implementation":"apexid.data.build_eval_transforms","input_channels":3,"mean":[0.485,0.456,0.406],"std":[0.229,0.224,0.225],"wavelet":false} | {"extraction_device":"cuda_if_available","source_cache_amp_enabled":true,"stage01_local_calibration_uses_cuda":false} |                    64 | bhca_backbone_sidecar        |
| ApexID_BHCA_SCSF2_v2 | /kaggle/input/models/mohamedbassat/apexid-convnext-small-bhca-scsf2-v2-rgb/pytorch/default/1/apexid_convnext_small_bhca_scsf2_v2_rgb/best_backbone.pt | /kaggle/input/models/mohamedbassat/apexid-convnext-small-bhca-scsf2-v2-rgb/pytorch/default/1/apexid_convnext_small_bhca_scsf2_v2_rgb/best_backbone.pt | a53dbd738fc9f1e69b86fa57ccfa4db9a10d41cbcc57fd35b0b52ee2f515e7a5 | rgb_bhca_eval_fallback_imagenet | 6a66af69a02c2f88fcef08108a6f798bc70e02f28e84a0077b716f0826a54fd7 | sidecar YAML next to best_backbone.pt |        125 |                  2048 | [440,440]          | unknown      | True            | {"config_sha256":"6a66af69a02c2f88fcef08108a6f798bc70e02f28e84a0077b716f0826a54fd7","config_source":"rgb_bhca_eval_fallback_imagenet","grayscale":false,"image_size":[440,440],"implementation":"apexid.data.build_eval_transforms","input_channels":3,"mean":[0.485,0.456,0.406],"std":[0.229,0.224,0.225],"wavelet":false} | {"extraction_device":"cuda_if_available","source_cache_amp_enabled":true,"stage01_local_calibration_uses_cuda":false} |                    64 | bhca_backbone_sidecar        |

## C. Dataset information

| dataset          |   raw_records |   records_removed | removal_reasons                                                            |   final_records_before_group_eligibility |   final_identities_before_group_eligibility | dataset_source                                                             | preprocessing                                     | segmentation_background_removal   |   overlap_removal_count | strategy                                                              |   explicit_column |   temporal_column |   camera_column |   images |   identities |   observation_groups |   exact_duplicate_groups |   near_duplicate_pairs |   dhash_threshold |   phash_threshold |   cross_identity_perceptual_candidates |   final_images |   final_identities | species                 | dataset_manifest_sha256                                          |   protocol_excluded_identities |   protocol_excluded_images | protocol_exclusion_reason                                         |
|:-----------------|--------------:|------------------:|:---------------------------------------------------------------------------|-----------------------------------------:|--------------------------------------------:|:---------------------------------------------------------------------------|:--------------------------------------------------|:----------------------------------|------------------------:|:----------------------------------------------------------------------|------------------:|------------------:|----------------:|---------:|-------------:|---------------------:|-------------------------:|-----------------------:|------------------:|------------------:|---------------------------------------:|---------------:|-------------------:|:------------------------|:-----------------------------------------------------------------|-------------------------------:|---------------------------:|:------------------------------------------------------------------|
| bristol_gorillas |          5324 |                 0 | {'invalid_or_unknown_identity': 0}                                         |                                     5324 |                                           7 | /kaggle/input/datasets/jamalhussien/bristol-gorillas-sam3-black-background | SAM3 foreground with black background             | required: SAM3 black background   |                       0 | visual similarity fallback: same-identity exact+dHash+pHash must-link |               nan |               nan |             nan |     5324 |            7 |                 4215 |                      134 |                   2338 |                 4 |                 6 |                                      0 |           5324 |                  7 | western_lowland_gorilla | febf0780c076df6ae15de39e3c5305fa188746a6f693b4323b00d8c876fc21e7 |                              0 |                          0 | fewer than two observation groups; retained in canonical manifest |
| elephants        |          2051 |                 0 | {'invalid_or_unknown_identity': 0}                                         |                                     2051 |                                         274 | /kaggle/input/datasets/jamalhussien/apexid-wildlife-reid-test-data         | manifest-defined source image                     | not declared                      |                       0 | visual similarity fallback: same-identity exact+dHash+pHash must-link |               nan |               nan |             nan |     2051 |          274 |                 2030 |                        6 |                     21 |                 4 |                 6 |                                      1 |           2051 |                274 | unknown                 | 8c5f7f1e7960708eae55c2c2f69576b3dd1118c18e225a4d740d60363618adc0 |                              2 |                          2 | fewer than two observation groups; retained in canonical manifest |
| gcn_id           |          1232 |                 0 | {'invalid_or_unknown_identity': 0}                                         |                                     1232 |                                         207 | /kaggle/input/datasets/jamalhussien/apexid-wildlife-reid-test-data         | manifest-defined source image                     | not declared                      |                       0 | visual similarity fallback: same-identity exact+dHash+pHash must-link |               nan |               nan |             nan |     1232 |          207 |                 1152 |                        1 |                    112 |                 4 |                 6 |                                     25 |           1232 |                207 | unknown                 | e06cb8418e646f307e8c9c527710f5a8bd069f8ee6591ec92f5fc1238c8bd45d |                              0 |                          0 | fewer than two observation groups; retained in canonical manifest |
| jaguar_antonio   |          1104 |                64 | {'invalid_or_unknown_identity': 0, 'apexid_training_identity_overlap': 64} |                                     1040 |                                          54 | /kaggle/working/apexid_jaguar_materialized                                 | FiftyOne body-mask foreground on black background | required and validated            |                      64 | visual similarity fallback: same-identity exact+dHash+pHash must-link |               nan |               nan |             nan |     1040 |           54 |                  675 |                        0 |                   1024 |                 4 |                 6 |                                   1598 |           1040 |                 54 | jaguar                  | 76816f0fe1a55029eaf4d9f32edcb5b58618eafaf5d5c68ecd1553d526fde9e4 |                              3 |                         11 | fewer than two observation groups; retained in canonical manifest |

## D. Validation threshold calibration

_No executed results._

Rejection thresholds maximize validation NormAcc with the repository's highest-threshold tie-break. Clustering thresholds maximize validation B³ F1 with the predeclared highest-similarity tie-break. Both are frozen before external evaluation.

## E. Closed-set results

| model                | dataset          | metric                   |   mean |   ci_low |   ci_high |   n_splits |   bootstrap_replicates | bootstrap_method                                             |
|:---------------------|:-----------------|:-------------------------|-------:|---------:|----------:|-----------:|-----------------------:|:-------------------------------------------------------------|
| ApexID_BHCA_A_v2     | bristol_gorillas | Rank-1                   | 0.9092 |   0.8921 |    0.9236 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_A_v2     | bristol_gorillas | mAP                      | 0.2609 |   0.2369 |    0.2803 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_A_v2     | elephants        | Rank-1                   | 0.1553 |   0.1341 |    0.1771 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_A_v2     | elephants        | mAP                      | 0.0600 |   0.0545 |    0.0660 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_A_v2     | gcn_id           | Rank-1                   | 0.8416 |   0.8151 |    0.8680 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_A_v2     | gcn_id           | mAP                      | 0.6109 |   0.5878 |    0.6342 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_A_v2     | jaguar_antonio   | Rank-1                   | 0.6036 |   0.5384 |    0.6561 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_A_v2     | jaguar_antonio   | mAP                      | 0.2582 |   0.2251 |    0.2860 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCA_v2   | bristol_gorillas | Rank-1                   | 0.9110 |   0.8953 |    0.9261 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCA_v2   | bristol_gorillas | mAP                      | 0.2605 |   0.2376 |    0.2812 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCA_v2   | elephants        | Rank-1                   | 0.1548 |   0.1239 |    0.1838 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCA_v2   | elephants        | mAP                      | 0.0590 |   0.0531 |    0.0659 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCA_v2   | gcn_id           | Rank-1                   | 0.8400 |   0.8083 |    0.8691 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCA_v2   | gcn_id           | mAP                      | 0.6065 |   0.5819 |    0.6312 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCA_v2   | jaguar_antonio   | Rank-1                   | 0.5980 |   0.5278 |    0.6663 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCA_v2   | jaguar_antonio   | mAP                      | 0.2537 |   0.2199 |    0.2824 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCSF2_v2 | bristol_gorillas | Rank-1                   | 0.9180 |   0.9057 |    0.9311 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCSF2_v2 | bristol_gorillas | mAP                      | 0.2603 |   0.2353 |    0.2802 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCSF2_v2 | elephants        | Rank-1                   | 0.1717 |   0.1495 |    0.1962 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCSF2_v2 | elephants        | mAP                      | 0.0632 |   0.0575 |    0.0692 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCSF2_v2 | gcn_id           | Rank-1                   | 0.8592 |   0.8314 |    0.8848 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCSF2_v2 | gcn_id           | mAP                      | 0.6235 |   0.5976 |    0.6474 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCSF2_v2 | jaguar_antonio   | Rank-1                   | 0.6167 |   0.5484 |    0.6767 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_SCSF2_v2 | jaguar_antonio   | mAP                      | 0.2626 |   0.2278 |    0.2947 |          5 |                   1000 | split-seed bootstrap + within-split identity-block bootstrap |
| ApexID_BHCA_A_v2     | bristol_gorillas | Rank-5                   | 0.9676 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | bristol_gorillas | Rank-10                  | 0.9863 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | bristol_gorillas | CMC@20                   | 0.9949 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | bristol_gorillas | CMC@50                   | 0.9985 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | bristol_gorillas | identity-balanced Rank-1 | 0.9107 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | elephants        | Rank-5                   | 0.3082 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | elephants        | Rank-10                  | 0.3987 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | elephants        | CMC@20                   | 0.4935 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | elephants        | CMC@50                   | 0.6539 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | elephants        | identity-balanced Rank-1 | 0.1404 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | gcn_id           | Rank-5                   | 0.9148 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | gcn_id           | Rank-10                  | 0.9547 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | gcn_id           | CMC@20                   | 0.9774 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | gcn_id           | CMC@50                   | 0.9883 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | gcn_id           | identity-balanced Rank-1 | 0.8559 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | jaguar_antonio   | Rank-5                   | 0.7616 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | jaguar_antonio   | Rank-10                  | 0.7986 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | jaguar_antonio   | CMC@20                   | 0.8409 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | jaguar_antonio   | CMC@50                   | 0.8878 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_A_v2     | jaguar_antonio   | identity-balanced Rank-1 | 0.5214 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | bristol_gorillas | Rank-5                   | 0.9665 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | bristol_gorillas | Rank-10                  | 0.9865 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | bristol_gorillas | CMC@20                   | 0.9947 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | bristol_gorillas | CMC@50                   | 0.9989 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | bristol_gorillas | identity-balanced Rank-1 | 0.9133 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | elephants        | Rank-5                   | 0.2965 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | elephants        | Rank-10                  | 0.3809 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | elephants        | CMC@20                   | 0.4930 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | elephants        | CMC@50                   | 0.6435 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | elephants        | identity-balanced Rank-1 | 0.1439 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | gcn_id           | Rank-5                   | 0.9314 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | gcn_id           | Rank-10                  | 0.9523 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | gcn_id           | CMC@20                   | 0.9773 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | gcn_id           | CMC@50                   | 0.9916 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | gcn_id           | identity-balanced Rank-1 | 0.8515 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | jaguar_antonio   | Rank-5                   | 0.7605 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | jaguar_antonio   | Rank-10                  | 0.8070 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | jaguar_antonio   | CMC@20                   | 0.8481 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | jaguar_antonio   | CMC@50                   | 0.8947 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCA_v2   | jaguar_antonio   | identity-balanced Rank-1 | 0.4907 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | bristol_gorillas | Rank-5                   | 0.9696 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | bristol_gorillas | Rank-10                  | 0.9840 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | bristol_gorillas | CMC@20                   | 0.9942 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | bristol_gorillas | CMC@50                   | 0.9984 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | bristol_gorillas | identity-balanced Rank-1 | 0.9197 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | elephants        | Rank-5                   | 0.3148 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | elephants        | Rank-10                  | 0.3908 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | elephants        | CMC@20                   | 0.4963 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | elephants        | CMC@50                   | 0.6586 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | elephants        | identity-balanced Rank-1 | 0.1545 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | gcn_id           | Rank-5                   | 0.9375 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | gcn_id           | Rank-10                  | 0.9633 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | gcn_id           | CMC@20                   | 0.9790 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | gcn_id           | CMC@50                   | 0.9874 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | gcn_id           | identity-balanced Rank-1 | 0.8684 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | jaguar_antonio   | Rank-5                   | 0.7685 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | jaguar_antonio   | Rank-10                  | 0.8018 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | jaguar_antonio   | CMC@20                   | 0.8351 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | jaguar_antonio   | CMC@50                   | 0.8935 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |
| ApexID_BHCA_SCSF2_v2 | jaguar_antonio   | identity-balanced Rank-1 | 0.5135 | nan      |  nan      |          5 |                      0 | point estimate; CI not requested                             |

## F. Open-set identification results

_No executed results._

## G. Oracle-Novel Clustering

_No executed results._

This conditional experiment uses the true novel mask only to define the evaluation subset; its clustering threshold is still validation-calibrated.

## H. End-to-End Novel Identity Discovery

_No executed results._

Rejected samples are selected without ground-truth novelty labels, then clustered with the frozen validation clustering threshold. Coverage and known contamination are reported alongside clustering quality.
The supplementary coverage-adjusted end-to-end score is conditional novel B³ F1 (or ARI) × identity-balanced novel discovery recall × (1 − known contamination rate); all factors are also reported separately.

## I. Cross-dataset generalization

| task       | model                | metric   | aggregation           |   mean |   ci_low |   ci_high |   number_datasets |   bootstrap_replicates | bootstrap_method                                             |
|:-----------|:---------------------|:---------|:----------------------|-------:|---------:|----------:|------------------:|-----------------------:|:-------------------------------------------------------------|
| closed_set | ApexID_BHCA_A_v2     | Rank-1   | Dataset-Macro Average | 0.6274 |   0.2902 |    0.8883 |                 4 |                   1000 | dataset + split-seed + identity-block hierarchical bootstrap |
| closed_set | ApexID_BHCA_A_v2     | mAP      | Dataset-Macro Average | 0.2975 |   0.1133 |    0.5068 |                 4 |                   1000 | dataset + split-seed + identity-block hierarchical bootstrap |
| closed_set | ApexID_BHCA_SCA_v2   | Rank-1   | Dataset-Macro Average | 0.6260 |   0.3157 |    0.8845 |                 4 |                   1000 | dataset + split-seed + identity-block hierarchical bootstrap |
| closed_set | ApexID_BHCA_SCA_v2   | mAP      | Dataset-Macro Average | 0.2949 |   0.1127 |    0.5002 |                 4 |                   1000 | dataset + split-seed + identity-block hierarchical bootstrap |
| closed_set | ApexID_BHCA_SCSF2_v2 | Rank-1   | Dataset-Macro Average | 0.6414 |   0.3448 |    0.8804 |                 4 |                   1000 | dataset + split-seed + identity-block hierarchical bootstrap |
| closed_set | ApexID_BHCA_SCSF2_v2 | mAP      | Dataset-Macro Average | 0.3024 |   0.1164 |    0.5254 |                 4 |                   1000 | dataset + split-seed + identity-block hierarchical bootstrap |

Dataset-Macro Average weights datasets equally. These three BHCA RGB ConvNeXt-Small models share architecture and differ by hierarchical loss.

## J. Statistical comparison

| task       | dataset          | reference_model   | comparison_model     | metric   |   mean_difference_reference_minus_comparison |   ci_low |   ci_high | paired_95ci_excludes_zero   |   bootstrap_replicates | bootstrap_method                             | difference_direction       | positive_delta_interpretation   |
|:-----------|:-----------------|:------------------|:---------------------|:---------|---------------------------------------------:|---------:|----------:|:----------------------------|-----------------------:|:---------------------------------------------|:---------------------------|:--------------------------------|
| closed_set | bristol_gorillas | ApexID_BHCA_A_v2  | ApexID_BHCA_SCA_v2   | Rank-1   |                                      -0.0018 |  -0.0073 |    0.0026 | False                       |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | bristol_gorillas | ApexID_BHCA_A_v2  | ApexID_BHCA_SCA_v2   | mAP      |                                       0.0005 |  -0.0021 |    0.0027 | False                       |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | elephants        | ApexID_BHCA_A_v2  | ApexID_BHCA_SCA_v2   | Rank-1   |                                       0.0005 |  -0.0183 |    0.0226 | False                       |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | elephants        | ApexID_BHCA_A_v2  | ApexID_BHCA_SCA_v2   | mAP      |                                       0.0009 |  -0.0017 |    0.0039 | False                       |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | gcn_id           | ApexID_BHCA_A_v2  | ApexID_BHCA_SCA_v2   | Rank-1   |                                       0.0016 |  -0.0170 |    0.0195 | False                       |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | gcn_id           | ApexID_BHCA_A_v2  | ApexID_BHCA_SCA_v2   | mAP      |                                       0.0045 |  -0.0012 |    0.0108 | False                       |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | jaguar_antonio   | ApexID_BHCA_A_v2  | ApexID_BHCA_SCA_v2   | Rank-1   |                                       0.0056 |  -0.0309 |    0.0374 | False                       |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | jaguar_antonio   | ApexID_BHCA_A_v2  | ApexID_BHCA_SCA_v2   | mAP      |                                       0.0045 |   0.0003 |    0.0093 | True                        |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | bristol_gorillas | ApexID_BHCA_A_v2  | ApexID_BHCA_SCSF2_v2 | Rank-1   |                                      -0.0088 |  -0.0170 |   -0.0022 | True                        |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | bristol_gorillas | ApexID_BHCA_A_v2  | ApexID_BHCA_SCSF2_v2 | mAP      |                                       0.0007 |  -0.0022 |    0.0031 | False                       |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | elephants        | ApexID_BHCA_A_v2  | ApexID_BHCA_SCSF2_v2 | Rank-1   |                                      -0.0164 |  -0.0296 |    0.0000 | False                       |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | elephants        | ApexID_BHCA_A_v2  | ApexID_BHCA_SCSF2_v2 | mAP      |                                      -0.0032 |  -0.0052 |   -0.0013 | True                        |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | gcn_id           | ApexID_BHCA_A_v2  | ApexID_BHCA_SCSF2_v2 | Rank-1   |                                      -0.0177 |  -0.0318 |   -0.0026 | True                        |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | gcn_id           | ApexID_BHCA_A_v2  | ApexID_BHCA_SCSF2_v2 | mAP      |                                      -0.0126 |  -0.0199 |   -0.0056 | True                        |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | jaguar_antonio   | ApexID_BHCA_A_v2  | ApexID_BHCA_SCSF2_v2 | Rank-1   |                                      -0.0130 |  -0.0391 |    0.0116 | False                       |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |
| closed_set | jaguar_antonio   | ApexID_BHCA_A_v2  | ApexID_BHCA_SCSF2_v2 | mAP      |                                      -0.0044 |  -0.0111 |    0.0010 | False                       |                   1000 | paired split-seed + identity-block bootstrap | reference_minus_comparison | reference_model_better          |

A paired difference is described as distinguishable only when its paired 95% bootstrap interval excludes zero; this is not a claim of universal superiority.

## K. Failure analysis and task-specific ranking

Measured patterns and task rankings are generated from executed result DataFrames. No causal explanations are inferred.

- No predeclared ≥0.15 task-gap pattern was detected, or results were not executed.

Task-specific Dataset-Macro rankings:

- Closed-set Re-ID: 1. ApexID_BHCA_SCSF2_v2 (0.6414), 2. ApexID_BHCA_A_v2 (0.6274), 3. ApexID_BHCA_SCA_v2 (0.6260)

Secondary combined score (unweighted mean of the five task-specific Dataset-Macro metrics; not primary):

- 1. ApexID_BHCA_SCSF2_v2: 0.6414
- 2. ApexID_BHCA_A_v2: 0.6274
- 3. ApexID_BHCA_SCA_v2: 0.6260

## Runtime Summary

- Total benchmark wall-clock time: 220.62 seconds
- Embedding inference/cache validation: 0.00 seconds
- Similarity computation: 0.00 seconds
- Clustering and calibration: 0.00 seconds
- Statistics/bootstrap: 184.96 seconds
- Report and plot generation: 0.07 seconds
- Cache reuse rate: unavailable

## Remaining Benchmark Limitations

- This benchmark compares three BHCA RGB ConvNeXt-Small backbones (A, SCA, SCSF2) with the same frozen Stage 00 splits and Stage 01 logistic rejector protocol.
- Dataset-Macro results cover the four evaluated external datasets (bristol_gorillas, elephants, gcn_id, jaguar_antonio); spac_wild is excluded from Stage 02.


## 17. Paper tables and lightweight export bundle

The complete Stage 02 output keeps resumable score/bootstrap caches. A second lightweight ZIP contains only paper-facing tables, plots, manifests, diagnostics, and reports.

In [17]:
# ============================================================
# 17. Paper-facing exports
# ============================================================

paper_tables = "\n\n".join([
    "# Paper Tables",
    "## Threshold Calibration\n"
    + threshold_calibration.to_markdown(index=False, floatfmt=".4f"),
    "## Dataset-Macro Comparison\n"
    + dataset_macro_summary.to_markdown(index=False, floatfmt=".4f"),
    "## Paired Comparisons\n"
    + paired_model_comparisons.to_markdown(index=False, floatfmt=".4f"),
]) + "\n"

atomic_write_text(
    OUTPUT_ROOT / "paper_tables.md",
    paper_tables,
)

# Useful compact model × task leaderboard for paper drafting.
macro_focus_metrics = {
    "closed_set": "Rank-1",
    "open_set": "NormAcc",
    "oracle_novel_clustering": "B3_F1",
    "end_to_end_discovery": "End-to-End Discovery B3_F1",
}
leaderboard_rows = []
for task, metric in macro_focus_metrics.items():
    selected = dataset_macro_summary.loc[
        (dataset_macro_summary["task"].astype(str) == task)
        & (dataset_macro_summary["metric"].astype(str) == metric)
    ].copy()
    if selected.empty:
        continue
    selected = selected.sort_values("mean", ascending=False)
    for rank, row in enumerate(selected.itertuples(index=False), start=1):
        leaderboard_rows.append({
            "task": task,
            "metric": metric,
            "rank": rank,
            "model": row.model,
            "dataset_macro_mean": row.mean,
            "ci_low": row.ci_low,
            "ci_high": row.ci_high,
        })

leaderboard_summary = pd.DataFrame(leaderboard_rows)
atomic_write_csv(
    PATHS["tables"] / "leaderboard_summary.csv",
    leaderboard_summary,
)

runtime_recorder.save(
    OUTPUT_ROOT / "runtime_report.csv"
)

# Lightweight paper bundle: omit score caches and bootstrap NPZs.
paper_bundle_root = OUTPUT_ROOT / "_paper_bundle"
if paper_bundle_root.exists():
    shutil.rmtree(paper_bundle_root)
paper_bundle_root.mkdir(parents=True, exist_ok=True)

for filename in (
    "final_benchmark_report.md",
    "final_benchmark_report.json",
    "benchmark_manifest.json",
    "stage02_manifest.json",
    "paper_tables.md",
    "runtime_report.csv",
):
    source = OUTPUT_ROOT / filename
    if source.is_file():
        shutil.copy2(source, paper_bundle_root / filename)

for directory in ("tables", "plots", "reports", "oracle_diagnostics", "calibration"):
    source = OUTPUT_ROOT / directory
    if source.is_dir():
        shutil.copytree(
            source,
            paper_bundle_root / directory,
            dirs_exist_ok=True,
        )

bundle_path = Path(
    shutil.make_archive(
        str(OUTPUT_ROOT / "apexid_cvpr_final_paper_outputs"),
        "zip",
        root_dir=paper_bundle_root,
    )
)
shutil.rmtree(paper_bundle_root)

print("STAGE 02 FINAL BENCHMARK COMPLETE")
print("Full resumable output:", OUTPUT_ROOT)
print("Paper-only ZIP:", bundle_path)
display(leaderboard_summary)

STAGE 02 FINAL BENCHMARK COMPLETE
Full resumable output: /kaggle/working/apexid_cvpr_final_benchmark_bhca_logistic
Paper-only ZIP: /kaggle/working/apexid_cvpr_final_benchmark_bhca_logistic/apexid_cvpr_final_paper_outputs.zip


,task,metric,rank,model,dataset_macro_mean,ci_low,ci_high
0,closed_set,Rank-1,1,ApexID_BHCA_SCSF2_v2,0.641393,0.344765,0.880398
1,closed_set,Rank-1,2,ApexID_BHCA_A_v2,0.627425,0.290209,0.888281
2,closed_set,Rank-1,3,ApexID_BHCA_SCA_v2,0.625972,0.315736,0.884473


## Final outputs

The complete resumable logistic benchmark is written to `/kaggle/working/apexid_cvpr_final_benchmark_bhca_logistic` on Kaggle, or `D:\CVPR_BM\apexid_cvpr_final_benchmark_bhca_logistic` locally.

Open-score caches contain `s1`, `s2`, margin, gallery size, log gallery size, `p_known`, and the generic official `knownness_scores`. In logistic mode, `knownness_scores` is exactly `p_known`. Their provenance binds the cache to the frozen Stage 01 calibrator hash and protocol.

Official tables, identity-block bootstraps, paired comparisons, OSCR curves, and end-to-end discovery all consume this same frozen official knownness score and threshold. Closed-set retrieval remains the original cosine-ranking pipeline and is protected by the full regression check. Oracle diagnostics are exported separately and cannot feed official predictions.
